# Combined Pipeline Review Notebook

This notebook is a **read-only review artifact** that combines the
ten modular notebooks in `notebooks/` into a single sequential
narrative for supervisor review.

**This notebook is NOT a replacement for the original modular
notebooks.** The original notebooks at `notebooks/01_*` through
`notebooks/07_*` remain the canonical working notebooks; if you
want to re-run any phase, run the corresponding original notebook,
not this combined one.

This combined notebook:

- Preserves the original cells and outputs of each source notebook
  (no re-execution was performed during merge).
- Adds a section header and a short metadata note before each
  notebook section: purpose, main artifacts read, main artifacts
  written, and whether expensive rerun is expected.
- Does not modify any original notebook on disk.

**Source notebooks merged (in execution order):**

1. `01_phase2_feature_selection.ipynb`
2. `01a_phase2_diagnostics.ipynb`
3. `01b_phase2_lightgbm_retune.ipynb`
4. `01c_scorecard.ipynb`
5. `02_economic_framework.ipynb`
6. `03_economic_stress_test.ipynb`
7. `04_bootstrap_cutoff_ci.ipynb`
8. `05_pd_quality_opcost_stress.ipynb`
9. `06_calibration_verification.ipynb`
10. `07_visualization.ipynb`


# Section 1 - 01_phase2_feature_selection.ipynb

**Section metadata**

- **Purpose**: Phase 2 - Feature selection on the Lasso track (Stage 1/2/3 funnel reducing 2,236 governed columns to the 7-feature LR no-F6E set).
- **Main artifacts read**: `artifacts/wide_abt/wide_abt.parquet` (Phase 2 wide ABT after cohort filter).
- **Main artifacts written**: `artifacts/feature_selection/` selected feature lists; `phase2_modeling_report.md`.
- **Expensive rerun expected**: Expensive (Lasso CV across the 9-cell C x seed stability sweep).

---


# Phase 2: PD Model Feature Selection

**Author:** Hoang
**Date:** 2026-05-07
**Inputs:** `artifacts/phase2_rerun_v2/modeling_abt.parquet`
**Outputs:** `artifacts/phase2_rerun_v2/{stage1, stage2, stage3, final_model.pkl}`

Three-stage feature selection on the 800/day production wide ABT (post Phase 1.5
feature factory, post `synth_credit_score_*` -> `synth_bureau_score_*` rename):

1. **Stage 1** — univariate prescreening (sparsity, variance, signal, stability)
2. **Stage 2** — Lasso (L1-penalised logistic regression, 5-fold CV)
3. **Stage 3** — statsmodels logit refinement + VIF + p-value filter

In [1]:
# Set up sys.path so 'from src import ...' works from a notebook in notebooks/
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import os
import time

import joblib
import numpy as np
import pandas as pd

from src.governance import (
    filter_pd_eligible, validate_no_score_columns,
    validate_no_loan_term_in_pd, get_meta_columns, summarise_eligible_pool,
)
from src.modeling import (
    run_stage1_prescreening, run_lasso_selection,
    fit_logit_statsmodels, compute_vif,
)
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)

np.random.seed(42)

T0 = time.time()

In [2]:
ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUTPUT_DIR   = REPO_ROOT / "artifacts/phase2_rerun_v2"
CACHE_DIR    = REPO_ROOT / "artifacts/notebook_cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"ABT_PATH    : {ABT_PATH}")
print(f"CATALOG_PATH: {CATALOG_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")

ABT_PATH    : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\modeling_abt.parquet
CATALOG_PATH: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\modeling_feature_catalog.csv
OUTPUT_DIR  : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2
CACHE_DIR   : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache


## Dry-Run Validation

Load catalog, filter PD-eligible features, then load only those columns + meta
from the ABT. Verify shapes, DRs, score-column absence, and loan-term exclusion.

In [3]:
catalog = pd.read_csv(CATALOG_PATH)
print(f"Catalog: {len(catalog):,} rows, {len(catalog.columns)} cols")

pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible features: {len(pd_eligible)}")

# Per-family breakdown of the eligible pool
elig_summary = summarise_eligible_pool(catalog)
print("\nPD-eligible by family:")
print(elig_summary.to_string(index=False))

columns_to_load = pd_eligible + get_meta_columns()
df = pd.read_parquet(ABT_PATH, columns=columns_to_load)
print(f"\nLoaded shape : {df.shape}")
print(f"Memory       : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Train rows   : {(df['split']=='train').sum():,}")
print(f"OOT rows     : {(df['split']=='oot').sum():,}")
print(f"Train DR     : {df.loc[df['split']=='train', 'default_flag_12m'].mean():.4%}")
print(f"OOT DR       : {df.loc[df['split']=='oot', 'default_flag_12m'].mean():.4%}")

f6d_count = sum(f.startswith('random_') for f in pd_eligible)
print(f"F6D pure-random in PD-eligible pool: {f6d_count} "
      f"({100*f6d_count/len(pd_eligible):.1f}%)")

# Defensive checks
assert validate_no_score_columns(df), "score/scorem leak detected!"
assert validate_no_loan_term_in_pd(df, catalog), "loan-term leak in PD pool!"
print("\nDefensive checks: PASS (no score/scorem, no loan-term in PD)")

Catalog: 2,236 rows, 14 cols
PD-eligible features: 435

PD-eligible by family:
feature_family  pd_eligible_count
           F6E                200
           F6D                100
           F6B                 47
           F5A                 37
           F6A                 25
  ORIGINAL_APP                 11
            F4                  6
           F6C                  5
            F3                  4



Loaded shape : (534314, 439)
Memory       : 1.02 GB
Train rows   : 389,525
OOT rows     : 144,789
Train DR     : 1.6330%
OOT DR       : 0.8474%
F6D pure-random in PD-eligible pool: 100 (23.0%)

Defensive checks: PASS (no score/scorem, no loan-term in PD)


## Stage 1 — Univariate Prescreening

Filter rules (per `src.modeling.run_stage1_prescreening`):
- **Sparsity:** `pct_nan <= 0.5`
- **Variance:** train stddev > 0
- **Signal:** train_gini >= 0.02
- **Stability:** `|train_gini - oot_gini| / max(train_gini, eps) <= 0.20`

Cached at `artifacts/notebook_cache/stage1_results.pkl` for re-runs.

In [4]:
cache_path = CACHE_DIR / "stage1_results.pkl"
if cache_path.exists():
    stage1 = joblib.load(cache_path)
    print(f"Loaded cached Stage 1 from {cache_path}")
else:
    t0 = time.time()
    stage1 = run_stage1_prescreening(
        df, target_col="default_flag_12m", features=pd_eligible
    )
    print(f"Stage 1 wall: {time.time() - t0:.1f}s")
    joblib.dump(stage1, cache_path)

print(f"Stage 1 survivors: {len(stage1)}")

Loaded cached Stage 1 from E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache\stage1_results.pkl
Stage 1 survivors: 120


In [5]:
print("Top 20 Stage 1 survivors by train Gini:")
print(stage1.head(20).to_string(index=False))

Top 20 Stage 1 survivors by train Gini:
                    feature  pct_nan           std  train_gini  oot_gini  gini_drop_pct  pass_sparse  pass_variance  pass_signal  pass_stability  survives
           app_nom_job_code      0.0      0.951194      0.3719    0.4328         0.1638         True           True         True            True      True
       mean_age_by_job_code      0.0      3.390114      0.3719    0.4328         0.1638         True           True         True            True      True
mean_n_children_by_job_code      0.0      0.113199      0.3719    0.4328         0.1638         True           True         True            True      True
                    act_age      0.0      6.243303      0.3406    0.3827         0.1237         True           True         True            True      True
                    log_age      0.0      0.106114      0.3406    0.3827         0.1236         True           True         True            True      True
                  act_age_z   

## Stage 2 — Lasso Selection

`LogisticRegressionCV` with L1 penalty, SAGA solver, 5-fold CV, 10 candidate
inverse regularisation strengths. Inputs are standardised; NaNs filled with
train medians. Returns features with non-zero coefficient at the CV-best `C`.

In [6]:
cache_path = CACHE_DIR / "stage2_results.pkl"
if cache_path.exists():
    stage2 = joblib.load(cache_path)
    print(f"Loaded cached Stage 2 from {cache_path}")
else:
    t0 = time.time()
    stage2 = run_lasso_selection(
        df, features=stage1["feature"].tolist(),
        target_col="default_flag_12m",
        cv=5, n_cs=10, random_state=42,
    )
    print(f"Stage 2 wall: {time.time() - t0:.1f}s")
    joblib.dump(stage2, cache_path)

print(f"Stage 2 Lasso survivors: {len(stage2)}")

Loaded cached Stage 2 from E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache\stage2_results.pkl
Stage 2 Lasso survivors: 64


In [7]:
print("Stage 2 survivors:")
for f in stage2:
    g = stage1.loc[stage1['feature']==f, 'train_gini']
    g_val = float(g.iloc[0]) if len(g) else float('nan')
    print(f"  {f:<45}  train_gini={g_val:.4f}")

Stage 2 survivors:
  app_nom_job_code                               train_gini=0.3719
  act_age_log1p                                  train_gini=0.3406
  synth_thin_file_flag                           train_gini=0.3373
  synth_age_of_newest                            train_gini=0.2856
  synth_newest_account_months                    train_gini=0.2535
  app_nom_marital_status                         train_gini=0.2448
  synth_avg_account_age                          train_gini=0.2423
  synth_oldest_mortgage_months                   train_gini=0.2384
  synth_seg_app_nom_job_code_top1                train_gini=0.2367
  synth_credit_age_years                         train_gini=0.2314
  age_x_n_children                               train_gini=0.2280
  synth_int_age_x_nchildren                      train_gini=0.2280
  marital_x_n_children                           train_gini=0.2220
  synth_int_inc_x_nchildren                      train_gini=0.2193
  synth_oldest_tradeline_v1                

## Stage 3 — Refinement (statsmodels Logit + VIF + p-value)

Fit unpenalised logit on Stage 2 survivors, inspect VIF, drop high-VIF features
(threshold 10), then drop p-values >= 0.05 and refit.

In [8]:
final_features = list(stage2)

vif_df = compute_vif(df, final_features)
print(f"VIF (top 15 by VIF):")
print(vif_df.head(15).to_string(index=False))

# Drop features with VIF > 10 (multicollinear)
high_vif = vif_df.loc[vif_df['vif'] > 10, 'feature'].tolist()
if high_vif:
    print(f"\nDropping {len(high_vif)} high-VIF features:")
    for f in high_vif:
        print(f"  - {f}")
    final_features = [f for f in final_features if f not in high_vif]
else:
    print("\nNo high-VIF features to drop.")

print(f"\nFeatures after VIF filter: {len(final_features)}")

model = fit_logit_statsmodels(df, final_features, "default_flag_12m")
print("\n" + str(model.summary()))

E:\Study\rl-debt-collection\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


VIF (top 15 by VIF):
                          feature  vif
                 age_x_n_children  inf
        synth_int_age_x_nchildren  inf
    synth_seg_app_nom_gender_top2  inf
               mean_age_by_gender  inf
                   app_nom_gender  inf
                  count_by_gender  inf
            mean_income_by_gender  inf
    synth_seg_app_nom_gender_top1  inf
                 act_age_decile_1  inf
        synth_seg_prime_age_30_50  inf
              app_nom_home_status  inf
          mean_age_by_home_status  inf
       mean_income_by_home_status  inf
app_number_of_children_quartile_2  inf
  app_number_of_children_decile_1  inf

Dropping 31 high-VIF features:
  - age_x_n_children
  - synth_int_age_x_nchildren
  - synth_seg_app_nom_gender_top2
  - mean_age_by_gender
  - app_nom_gender
  - count_by_gender
  - mean_income_by_gender
  - synth_seg_app_nom_gender_top1
  - act_age_decile_1
  - synth_seg_prime_age_30_50
  - app_nom_home_status
  - mean_age_by_home_status
  - mean_inco


                           Logit Regression Results                           
Dep. Variable:       default_flag_12m   No. Observations:               389525
Model:                          Logit   Df Residuals:                   389491
Method:                           MLE   Df Model:                           33
Date:                Thu, 07 May 2026   Pseudo R-squ.:                  0.1618
Time:                        14:23:37   Log-Likelihood:                -27225.
converged:                       True   LL-Null:                       -32483.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
const                                     5.4647      0.312     17.501      0.000       4.853       6.077
app_nom_job_code                         -1.1058 

In [9]:
significant = model.pvalues[model.pvalues < 0.05].index.tolist()
final_features = [f for f in significant if f != "const"]
print(f"Final feature set after p<0.05 filter: {len(final_features)} features")
for f in final_features:
    coef = model.params[f]
    pval = model.pvalues[f]
    print(f"  {f:<45}  coef={coef:+.5f}  p={pval:.4g}")

Final feature set after p<0.05 filter: 22 features
  app_nom_job_code                               coef=-1.10578  p=0
  synth_age_of_newest                            coef=-0.02282  p=7.402e-15
  synth_newest_account_months                    coef=-0.00415  p=0.0004052
  synth_avg_account_age                          coef=-0.00825  p=7.579e-05
  synth_oldest_mortgage_months                   coef=-0.00925  p=2.293e-11
  synth_seg_app_nom_job_code_top1                coef=+1.08448  p=8.534e-52
  synth_credit_age_years                         coef=-0.00949  p=1.477e-07
  marital_x_n_children                           coef=-0.42147  p=1.16e-82
  synth_int_inc_x_nchildren                      coef=-0.00022  p=3.628e-31
  synth_oldest_tradeline_v1                      coef=-0.00753  p=2.374e-06
  synth_oldest_card_months                       coef=-0.00455  p=0.0002744
  synth_n_late_90                                coef=-0.00753  p=3.051e-05
  synth_avg_account_age_months                

In [10]:
final_model = fit_logit_statsmodels(df, final_features, "default_flag_12m")
joblib.dump(final_model, OUTPUT_DIR / "final_model.pkl")
print(f"Saved: {OUTPUT_DIR / 'final_model.pkl'}")
print("\nFinal model summary:")
print(final_model.summary())

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\final_model.pkl

Final model summary:


                           Logit Regression Results                           
Dep. Variable:       default_flag_12m   No. Observations:               389525
Model:                          Logit   Df Residuals:                   389502
Method:                           MLE   Df Model:                           22
Date:                Thu, 07 May 2026   Pseudo R-squ.:                  0.1617
Time:                        14:23:39   Log-Likelihood:                -27230.
converged:                       True   LL-Null:                       -32483.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
const                                     5.1543      0.169     30.575      0.000       4.824       5.485
app_nom_job_code                         -1.1063  

## Performance Evaluation

Compute Gini, KS, Brier on train and OOT splits.

In [11]:
import statsmodels.api as sm
X_full = sm.add_constant(df[final_features].fillna(df[final_features].median(numeric_only=True)),
                         has_constant="add")
df["pd_score"] = final_model.predict(X_full)

train_mask = df["split"] == "train"
oot_mask   = df["split"] == "oot"

metrics = {
    "train": {
        "gini" : compute_gini(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "ks"   : compute_ks  (df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "brier": compute_brier(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "auc"  : (compute_gini(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]) + 1) / 2,
    },
    "oot": {
        "gini" : compute_gini(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "ks"   : compute_ks  (df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "brier": compute_brier(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "auc"  : (compute_gini(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]) + 1) / 2,
    },
}
print(json.dumps(metrics, indent=2))

{
  "train": {
    "gini": 0.6410536039484025,
    "ks": 0.48110621106021684,
    "brier": 0.015219356754642983,
    "auc": 0.8205268019742012
  },
  "oot": {
    "gini": 0.7181197604272356,
    "ks": 0.5475497513848578,
    "brier": 0.00815718374553534,
    "auc": 0.8590598802136178
  }
}


In [12]:
cal_train = compute_calibration_metrics(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"])
cal_oot   = compute_calibration_metrics(df.loc[oot_mask,   "default_flag_12m"], df.loc[oot_mask,   "pd_score"])
cal_df = pd.DataFrame({"train": cal_train, "oot": cal_oot})
print("Calibration metrics:")
print(cal_df.to_string())

Calibration metrics:
                      train            oot
o_to_e_bin1        1.007992       0.000000
o_to_e_bin2        0.995227       0.136039
o_to_e_bin3        0.946193       0.318722
o_to_e_bin4        0.953032       0.387455
o_to_e_bin5        0.990767       0.420720
o_to_e_bin6        0.858687       0.297431
o_to_e_bin7        0.923754       0.441476
o_to_e_bin8        0.983163       0.427902
o_to_e_bin9        1.014547       0.464239
o_to_e_bin10       1.034050       0.637332
ece                0.000622       0.007792
n             389525.000000  144789.000000
base_rate          0.016330       0.008474
mean_pred          0.016330       0.016267


## Safety Checks

Final-feature acceptance gate:
- No score/scorem under any name
- No `default_flag_12m` self-reference
- No loan-term transforms
- F6D pure-random count + tiered alert

In [13]:
# Check 1: no simulator score (synth_bureau_* allowed)
for f in final_features:
    fl = f.lower()
    if "score" in fl and "bureau" not in fl and "synth_int" not in fl:
        raise AssertionError(f"Score variable found: {f}")
print("Check 1 (no simulator score): PASS")

# Check 2: no target self-reference
assert "default_flag_12m" not in final_features
print("Check 2 (no target self-reference): PASS")

# Check 3: no loan-term transforms (look at raw string + catalog source_columns)
loan_terms = ("loan_amount", "n_installments", "installment", "principal", "tenor")
cat_idx = catalog.set_index("feature_name")
for f in final_features:
    fl = f.lower()
    # forbid raw-name match unless feature is a synthetic bureau-score derivative
    if f.startswith("synth_"):
        continue
    for lt in loan_terms:
        if lt in fl:
            raise AssertionError(f"Loan-term transform found: {f}")
    # cross-check against catalog source_columns
    if f in cat_idx.index:
        sources = str(cat_idx.at[f, "source_columns"] or "").lower()
        for lt in loan_terms:
            if lt in sources and not f.startswith("synth_"):
                raise AssertionError(f"Loan-term sourcing found in {f}: source_columns={sources}")
print("Check 3 (no loan-term transforms): PASS")

# Check 4: F6D negative-control assessment (tiered)
f6d_in_final = [f for f in final_features if f.startswith("random_")]
print(f"Check 4: F6D pure-random features surviving = {len(f6d_in_final)}")
if len(f6d_in_final) == 0:
    print("  IDEAL: zero F6D survived -> Lasso correctly rejects pure noise")
elif len(f6d_in_final) == 1:
    print(f"  WARNING: 1 F6D survived -> investigate but not auto-failure  -> {f6d_in_final[0]}")
else:
    print(f"  SERIOUS: {len(f6d_in_final)} F6D survived  -> investigate pipeline  -> {f6d_in_final}")

Check 1 (no simulator score): PASS
Check 2 (no target self-reference): PASS
Check 3 (no loan-term transforms): PASS
Check 4: F6D pure-random features surviving = 0
  IDEAL: zero F6D survived -> Lasso correctly rejects pure noise


## Save Artifacts

In [14]:
stage1.to_csv(OUTPUT_DIR / "stage1_selected_features.csv", index=False)
pd.Series(stage2, name="feature").to_csv(OUTPUT_DIR / "stage2_selected_features.csv", index=False, header=True)
pd.Series(final_features, name="feature").to_csv(OUTPUT_DIR / "final_feature_set.csv", index=False, header=True)

df_scores = df[["aid", "split", "default_flag_12m", "pd_score"]].copy()
df_scores.to_parquet(OUTPUT_DIR / "pd_scores.parquet", index=False)

with open(OUTPUT_DIR / "model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Persist final coefficient table
coef_table = pd.DataFrame({
    "feature": ["const"] + final_features,
    "coef": [final_model.params["const"]] + [final_model.params[f] for f in final_features],
    "std_err": [final_model.bse["const"]] + [final_model.bse[f] for f in final_features],
    "p_value": [final_model.pvalues["const"]] + [final_model.pvalues[f] for f in final_features],
})
coef_table.to_csv(OUTPUT_DIR / "final_coefficients.csv", index=False)

# Calibration outputs
cal_df.to_csv(OUTPUT_DIR / "calibration_metrics.csv")

print("Saved artifacts:")
for p in sorted(OUTPUT_DIR.glob("*")):
    sz = p.stat().st_size
    print(f"  {p.name:<40}  {sz:>10,} bytes")
print(f"\nWall (Phase 2 total): {time.time() - T0:.1f}s")

Saved artifacts:
  calibration_metrics.csv                          683 bytes
  final_coefficients.csv                         2,052 bytes
  final_feature_set.csv                            549 bytes
  final_model.pkl                           207,240,649 bytes
  model_metrics.json                               303 bytes
  modeling_abt.parquet                      2,381,776,565 bytes
  modeling_feature_catalog.csv                 480,190 bytes
  pd_scores.parquet                          6,925,995 bytes
  stage1_selected_features.csv                  10,076 bytes
  stage2_selected_features.csv                   1,580 bytes

Wall (Phase 2 total): 98.6s


## Final Report

| Metric | Train | OOT |
|--------|------:|----:|
| Gini   | see metrics dict above | |
| KS     | | |
| AUC    | | |
| Brier  | | |
| ECE    | see calibration metrics | |

See `model_metrics.json`, `calibration_metrics.csv`, and
`final_coefficients.csv` for the canonical numbers used by the thesis.

**F6D negative-control assessment:** see safety checks above.

**Comparison to old SET A':** SET A' (4 numeric + 5 categorical: `app_income`,
`app_number_of_children`, `app_spendings`, `act_age`, plus the five
categoricals) is the prior art baseline. The expected overlap with the new
final feature set is computed in the next cell.

In [15]:
SET_A_PRIME = {
    "app_income", "app_number_of_children", "app_spendings", "act_age",
    "app_nom_gender", "app_nom_marital_status", "app_nom_home_status",
    "app_nom_cars", "app_nom_job_code",
}
overlap = SET_A_PRIME & set(final_features)
new_only = set(final_features) - SET_A_PRIME
print(f"SET A' size: {len(SET_A_PRIME)}")
print(f"Final feature set size: {len(final_features)}")
print(f"Overlap with SET A': {len(overlap)}")
print(f"  overlap features: {sorted(overlap)}")
print(f"New-feature-factory features in final: {len(new_only)}")
print(f"  (showing first 30): {sorted(new_only)[:30]}")

SET A' size: 9
Final feature set size: 22
Overlap with SET A': 1
  overlap features: ['app_nom_job_code']
New-feature-factory features in final: 21
  (showing first 30): ['app_nom_branch', 'app_nom_city', 'marital_x_n_children', 'median_income_by_job_code', 'synth_age_of_newest', 'synth_avg_account_age', 'synth_avg_account_age_months', 'synth_credit_age_years', 'synth_int_inc_x_nchildren', 'synth_n_auto_lines', 'synth_n_card_lines', 'synth_n_installment_lines', 'synth_n_late_30', 'synth_n_late_90', 'synth_newest_account_months', 'synth_oldest_card_months', 'synth_oldest_installment_months', 'synth_oldest_mortgage_months', 'synth_oldest_tradeline_v1', 'synth_seg_app_nom_job_code_top1', 'synth_seg_app_nom_marital_status_top2']


# Section 2 - 01a_phase2_diagnostics.ipynb

**Section metadata**

- **Purpose**: Phase 2A diagnostic experiments - F6D negative-control gate, scorem absolute-Gini measurement, leakage triangulation.
- **Main artifacts read**: Phase 2 selected features and wide ABT.
- **Main artifacts written**: `artifacts/feature_selection/test1_f6e_ablation.json` and supporting diagnostics.
- **Expensive rerun expected**: Moderate.

---


# Phase 2A Diagnostics — 5 Tests

**Purpose:** address 4 corrections from Round 14 review:
1. F6E ablation (legitimacy of F6E dominance)
2. Lasso stability (sensitivity to C, seed)
3. Leak-free calibration (temporal split inside train, OOT untouched)
4. LightGBM baseline (dual-track LR + LightGBM that was missing)
5. Score-discrimination stress-test design (no execution; Phase 3/4 work)

**Anchor numbers from Phase 2A initial run:**
- Final-model OOT AUC = 0.859, Gini = 0.718, Brier = 0.0082
- Final feature set: 22 features; 17/22 are F6E synthetic-bureau
- F6D negative controls: 0 survived (PASS)
- OOT calibration drift: mean predicted 1.63% vs observed 0.85% (1.92×)

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.governance import filter_pd_eligible, get_meta_columns
from src.modeling import (
    run_stage1_prescreening, run_lasso_selection,
    fit_logit_statsmodels, compute_vif,
)
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
    StressTestPlan, perturb_to_target_gini,
)
np.random.seed(42)

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUT_DIR      = REPO_ROOT / "artifacts/phase2_rerun_v2"
DIAG_DIR     = OUT_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
catalog = pd.read_csv(CATALOG_PATH)
all_pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible pool: {len(all_pd_eligible)}")

PD-eligible pool: 435


## Test 1 — F6E ablation

Run the same Stage 1 → Stage 2 → Stage 3 pipeline excluding all F6E (and F6E-
derived) features. Goal: quantify whether discrimination is driven by F6E
synthetic-bureau features or holds without them.

In [2]:
# Identify F6E features in the catalog
f6e_cat = catalog[catalog["feature_family"] == "F6E"]["feature_name"].tolist()
print(f"F6E features in catalog: {len(f6e_cat)}")

# PD-eligible WITHOUT any F6E
non_f6e_eligible = [f for f in all_pd_eligible if f not in set(f6e_cat)]
print(f"Non-F6E PD-eligible features: {len(non_f6e_eligible)}")

# Load only those columns + meta
df_nof6e = pd.read_parquet(ABT_PATH, columns=non_f6e_eligible + get_meta_columns())
print(f"Loaded shape: {df_nof6e.shape}")
print(f"Train DR: {df_nof6e.loc[df_nof6e['split']=='train', 'default_flag_12m'].mean():.4%}")
print(f"OOT DR  : {df_nof6e.loc[df_nof6e['split']=='oot',   'default_flag_12m'].mean():.4%}")

F6E features in catalog: 200
Non-F6E PD-eligible features: 235


Loaded shape: (534314, 239)
Train DR: 1.6330%
OOT DR  : 0.8474%


In [3]:
# Stage 1 (no caching for ablation since features differ)
t = time.time()
stage1_nof6e = run_stage1_prescreening(df_nof6e, "default_flag_12m", non_f6e_eligible)
print(f"Stage 1 (no F6E) wall: {time.time()-t:.1f}s, survivors: {len(stage1_nof6e)}")

Stage 1 (no F6E) wall: 26.2s, survivors: 66


In [4]:
# Stage 2 — Lasso single-fit at C=0.05
t = time.time()
stage2_nof6e = run_lasso_selection(
    df_nof6e, stage1_nof6e["feature"].tolist(), "default_flag_12m",
    use_cv=False, fixed_c=0.05, train_subsample=100_000, random_state=42,
)
print(f"Stage 2 (no F6E) wall: {time.time()-t:.1f}s, survivors: {len(stage2_nof6e)}")

Stage 2 (no F6E) wall: 11.7s, survivors: 34


In [5]:
# Stage 3 — VIF + p-value filter + final fit
final_features_nof6e = list(stage2_nof6e)
vif_df = compute_vif(df_nof6e, final_features_nof6e)
high_vif = vif_df.loc[vif_df["vif"] > 10, "feature"].tolist()
final_features_nof6e = [f for f in final_features_nof6e if f not in high_vif]
print(f"After VIF filter: {len(final_features_nof6e)} features (dropped {len(high_vif)} high-VIF)")

model_nof6e = fit_logit_statsmodels(df_nof6e, final_features_nof6e, "default_flag_12m")
significant = model_nof6e.pvalues[model_nof6e.pvalues < 0.05].index.tolist()
final_features_nof6e = [f for f in significant if f != "const"]
print(f"After p<0.05 filter: {len(final_features_nof6e)} features")
final_model_nof6e = fit_logit_statsmodels(df_nof6e, final_features_nof6e, "default_flag_12m")
print("\nFinal-model coefficients:")
for f in final_features_nof6e:
    print(f"  {f:<45}  coef={final_model_nof6e.params[f]:+.5f}  p={final_model_nof6e.pvalues[f]:.4g}")

After VIF filter: 11 features (dropped 23 high-VIF)


After p<0.05 filter: 7 features



Final-model coefficients:
  app_nom_job_code                               coef=-1.13537  p=0
  act_age_log1p                                  coef=-5.08540  p=8.633e-256
  mean_cars_by_job_code                          coef=+35.62502  p=1.677e-59
  app_nom_marital_status                         coef=-0.61768  p=0
  median_income_by_job_code                      coef=-0.00023  p=3.374e-157
  app_nom_branch                                 coef=-0.17234  p=1.876e-36
  app_nom_city                                   coef=-0.14634  p=7.256e-27


In [6]:
# Score and metrics
import statsmodels.api as sm
X_nof6e = sm.add_constant(
    df_nof6e[final_features_nof6e].fillna(df_nof6e[final_features_nof6e].median(numeric_only=True)),
    has_constant="add",
)
df_nof6e["pd_score_nof6e"] = final_model_nof6e.predict(X_nof6e)
tr = df_nof6e["split"]=="train"; oot = df_nof6e["split"]=="oot"
metrics_nof6e = {
    "train": {
        "auc"  : (compute_gini(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]) + 1)/2,
        "gini" : compute_gini(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
        "ks"   : compute_ks  (df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
        "brier": compute_brier(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
    },
    "oot": {
        "auc"  : (compute_gini(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]) + 1)/2,
        "gini" : compute_gini(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
        "ks"   : compute_ks  (df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
        "brier": compute_brier(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
    },
}
print("\n=== Test 1 result ===")
print(json.dumps(metrics_nof6e, indent=2))

# Compare to current full-F6E model
with open(OUT_DIR / "model_metrics.json") as f:
    metrics_full = json.load(f)
print("\n=== Comparison: full-F6E vs no-F6E ===")
comp = pd.DataFrame({
    "full_F6E_train": metrics_full["train"], "no_F6E_train": metrics_nof6e["train"],
    "full_F6E_oot":   metrics_full["oot"],   "no_F6E_oot":   metrics_nof6e["oot"],
})
print(comp.round(4).to_string())

# Calibration on OOT (raw, uncalibrated)
cal_nof6e_oot = compute_calibration_metrics(
    df_nof6e.loc[oot, "default_flag_12m"], df_nof6e.loc[oot, "pd_score_nof6e"]
)
print(f"\nNo-F6E OOT calibration: ECE={cal_nof6e_oot.get('ece', float('nan')):.4f}, "
      f"mean_pred={cal_nof6e_oot.get('mean_pred', float('nan')):.4%}, "
      f"base_rate={cal_nof6e_oot.get('base_rate', float('nan')):.4%}")

# Save
test1_out = {
    "n_features_pool": len(non_f6e_eligible),
    "n_stage1": len(stage1_nof6e),
    "n_stage2": len(stage2_nof6e),
    "n_final":  len(final_features_nof6e),
    "final_features": final_features_nof6e,
    "metrics": metrics_nof6e,
    "calib_oot_uncal": {k: float(v) if isinstance(v, (int,float)) else v
                        for k, v in cal_nof6e_oot.items()},
}
with open(DIAG_DIR / "test1_f6e_ablation.json", "w") as f:
    json.dump(test1_out, f, indent=2, default=str)
print("\nSaved: diagnostics/test1_f6e_ablation.json")


=== Test 1 result ===
{
  "train": {
    "auc": 0.7938758898346447,
    "gini": 0.5877517796692895,
    "ks": 0.4518301201517804,
    "brier": 0.015378107686033909
  },
  "oot": {
    "auc": 0.8361300713104687,
    "gini": 0.6722601426209374,
    "ks": 0.5394825111384536,
    "brier": 0.008207038913770828
  }
}

=== Comparison: full-F6E vs no-F6E ===
       full_F6E_train  no_F6E_train  full_F6E_oot  no_F6E_oot
gini           0.6411        0.5878        0.7181      0.6723
ks             0.4811        0.4518        0.5475      0.5395
brier          0.0152        0.0154        0.0082      0.0082
auc            0.8205        0.7939        0.8591      0.8361

No-F6E OOT calibration: ECE=0.0078, mean_pred=1.6283%, base_rate=0.8474%

Saved: diagnostics/test1_f6e_ablation.json


## Test 2 — Lasso stability (3 C × 3 seeds)

Sensitivity of single-fit Lasso selection across (C, seed) configurations.
We use the *full-pool* Stage 1 survivors (with F6E) since the goal here is to
test the Stage 2 step's reliability, not the F6E hypothesis.

In [7]:
# Reload Stage 1 cache from full-pool run
stage1_full = joblib.load(REPO_ROOT / "artifacts/notebook_cache/stage1_results.pkl")
features_s1 = stage1_full["feature"].tolist()
print(f"Reusing Stage 1 survivors (full pool): {len(features_s1)}")

# Need full-pool df with all those columns
df_full = pd.read_parquet(ABT_PATH, columns=features_s1 + get_meta_columns())
print(f"Loaded ABT for stability test: {df_full.shape}")

Reusing Stage 1 survivors (full pool): 120


Loaded ABT for stability test: (534314, 124)


In [8]:
Cs = [0.02, 0.05, 0.10]
SEEDS = [42, 101, 202]

stability_rows = []
selected_sets = {}

for c in Cs:
    for sd in SEEDS:
        t = time.time()
        sel = run_lasso_selection(
            df_full, features_s1, "default_flag_12m",
            use_cv=False, fixed_c=c, train_subsample=100_000, random_state=sd,
        )
        wall = time.time() - t
        f6e_count = sum(1 for f in sel if f in set(catalog[catalog["feature_family"]=="F6E"]["feature_name"]))
        f6d_count = sum(1 for f in sel if f.startswith("random_"))
        stability_rows.append({
            "C": c, "seed": sd,
            "n_survivors": len(sel),
            "f6e_count": f6e_count,
            "f6e_pct": round(100*f6e_count/max(len(sel),1), 1),
            "f6d_count": f6d_count,
            "wall_sec": round(wall, 1),
        })
        selected_sets[(c, sd)] = set(sel)

stab_df = pd.DataFrame(stability_rows)
print("\nStability table (per C × seed):")
print(stab_df.to_string(index=False))


Stability table (per C × seed):
   C  seed  n_survivors  f6e_count  f6e_pct  f6d_count  wall_sec
0.02    42           45         18     40.0          0      14.1
0.02   101           46         18     39.1          0      15.0
0.02   202           39         11     28.2          0      23.9
0.05    42           64         33     51.6          0      17.6
0.05   101           59         27     45.8          0      11.7
0.05   202           54         22     40.7          0      28.7
0.10    42           72         40     55.6          0      33.4
0.10   101           66         33     50.0          0      29.3
0.10   202           64         30     46.9          0      77.3


In [9]:
# Pairwise Jaccard overlap
keys = list(selected_sets.keys())
jaccard = pd.DataFrame(0.0, index=[str(k) for k in keys], columns=[str(k) for k in keys])
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        a, b = selected_sets[ki], selected_sets[kj]
        u = a | b
        jaccard.iloc[i, j] = len(a & b) / max(len(u), 1)
print("\nPairwise Jaccard overlap of selected feature sets:")
print(jaccard.round(3).to_string())

# Frequency of each feature across all 9 runs
all_selected = pd.Series([f for s in selected_sets.values() for f in s])
freq = all_selected.value_counts()
core = freq[freq >= 8].index.tolist()  # selected in 8/9+ runs
sometimes = freq[(freq >= 5) & (freq < 8)].index.tolist()
print(f"\nFeatures selected in >=8/9 runs (core, robust): {len(core)}")
for f in core[:30]:
    print(f"  {f:<50}  count={freq[f]}")
print(f"\nFeatures selected in 5-7/9 runs (boundary): {len(sometimes)}")


Pairwise Jaccard overlap of selected feature sets:
             (0.02, 42)  (0.02, 101)  (0.02, 202)  (0.05, 42)  (0.05, 101)  (0.05, 202)  (0.1, 42)  (0.1, 101)  (0.1, 202)
(0.02, 42)        1.000        0.655        0.556       0.627        0.625        0.523      0.481       0.542       0.493
(0.02, 101)       0.655        1.000        0.667       0.486        0.780        0.538      0.439       0.623       0.528
(0.02, 202)       0.556        0.667        1.000       0.431        0.531        0.661      0.388       0.419       0.585
(0.05, 42)        0.627        0.486        0.431       1.000        0.519        0.513      0.789       0.566       0.524
(0.05, 101)       0.625        0.780        0.531       0.519        1.000        0.507      0.489       0.812       0.519
(0.05, 202)       0.523        0.538        0.661       0.513        0.507        1.000      0.518       0.500       0.844
(0.1, 42)         0.481        0.439        0.388       0.789        0.489        0.518

In [10]:
# Save (cast pandas/numpy ints to Python ints for JSON safety)
test2_out = {
    "configs": stability_rows,
    "jaccard_min": float(jaccard.replace(1.0, np.nan).min().min()),
    "jaccard_mean": float(jaccard.replace(1.0, np.nan).mean().mean()),
    "core_features_8plus_of_9": core,
    "boundary_features_5_to_7": sometimes,
    "f6e_pct_range": [float(stab_df["f6e_pct"].min()), float(stab_df["f6e_pct"].max())],
    "f6d_count_range": [int(stab_df["f6d_count"].min()), int(stab_df["f6d_count"].max())],
}
with open(DIAG_DIR / "test2_lasso_stability.json", "w") as f:
    json.dump(test2_out, f, indent=2, default=str)
print(f"\nJaccard min={test2_out['jaccard_min']:.3f}, mean={test2_out['jaccard_mean']:.3f}")
print(f"F6E share across configs: {stab_df['f6e_pct'].min()}-{stab_df['f6e_pct'].max()}%")
print(f"F6D survivors across configs: {stab_df['f6d_count'].min()}-{stab_df['f6d_count'].max()} (target=0)")
print("Saved: diagnostics/test2_lasso_stability.json")


Jaccard min=0.388, mean=0.567
F6E share across configs: 28.2-55.6%
F6D survivors across configs: 0-0 (target=0)
Saved: diagnostics/test2_lasso_stability.json


## Test 3 — Leak-free calibration

**Splits:**
- model training: cohorts 202509-202610 (14 cohorts)
- calibration: cohorts 202611, 202612 (last 2 train cohorts)
- OOT: cohorts 202701-202706 (UNTOUCHED for calibration fitting)

Refit final-feature LR on the reduced training set, score everything, then fit
Platt and isotonic calibrators on the calibration cohorts. Evaluate calibrated
PD on OOT.

In [11]:
# Load full-pool df + final-model features from Phase 2A
final_model = joblib.load(OUT_DIR / "final_model.pkl")
final_features = pd.read_csv(OUT_DIR / "final_feature_set.csv")["feature"].tolist()
print(f"Reusing final feature set ({len(final_features)} features)")

df_cal = pd.read_parquet(ABT_PATH,
                          columns=final_features + get_meta_columns())
print(f"Loaded ABT for calibration test: {df_cal.shape}")

# Define splits
TRAIN_FOR_MODEL = list(range(202509, 202613))  # 202509..202612 inclusive
TRAIN_FOR_MODEL = [p for p in TRAIN_FOR_MODEL if p % 100 in range(1, 13)]
TRAIN_FOR_MODEL = [p for p in TRAIN_FOR_MODEL if p <= 202610]  # exclude calib cohorts
CALIB_PERIODS = [202611, 202612]
print(f"Train-for-model cohorts: {TRAIN_FOR_MODEL}")
print(f"Calibration cohorts:    {CALIB_PERIODS}")

split_new = make_calibration_split(df_cal, TRAIN_FOR_MODEL, CALIB_PERIODS)
df_cal["split_new"] = split_new
print("\nNew split counts:")
print(df_cal["split_new"].value_counts().to_string())
print("\nDR by new split:")
for s in ["train_for_model", "calib", "oot"]:
    sub = df_cal[df_cal["split_new"]==s]
    if len(sub) > 0:
        print(f"  {s:<18} n={len(sub):>7,}  DR={(sub['default_flag_12m']==1).mean():.4%}  "
              f"events={int((sub['default_flag_12m']==1).sum())}")

Reusing final feature set (22 features)


Loaded ABT for calibration test: (534314, 26)
Train-for-model cohorts: [202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605, 202606, 202607, 202608, 202609, 202610]
Calibration cohorts:    [202611, 202612]

New split counts:
split_new
train_for_model    340727
oot                144789
calib               48798

DR by new split:
  train_for_model    n=340,727  DR=1.7345%  events=5910
  calib              n= 48,798  DR=0.9242%  events=451


  oot                n=144,789  DR=0.8474%  events=1227


In [12]:
# Refit LR on TRAIN_FOR_MODEL only
import statsmodels.api as sm
mask_tfm = df_cal["split_new"] == "train_for_model"
X_tfm = df_cal.loc[mask_tfm, final_features].fillna(df_cal[final_features].median(numeric_only=True))
y_tfm = df_cal.loc[mask_tfm, "default_flag_12m"].astype(int)
X_tfm = sm.add_constant(X_tfm, has_constant="add")
model_calib = sm.Logit(y_tfm, X_tfm).fit(disp=False, maxiter=200)
print(f"Refit LR on {mask_tfm.sum():,} train_for_model rows, {len(final_features)} features")

# Score everything
X_full = sm.add_constant(
    df_cal[final_features].fillna(df_cal[final_features].median(numeric_only=True)),
    has_constant="add",
)
df_cal["pd_raw"] = model_calib.predict(X_full)

# Fit calibrators on calib split only
mask_calib = df_cal["split_new"] == "calib"
y_calib = df_cal.loc[mask_calib, "default_flag_12m"].astype(int).to_numpy()
s_calib = df_cal.loc[mask_calib, "pd_raw"].to_numpy()
print(f"Fitting calibrators on {len(s_calib):,} calib rows, {y_calib.sum()} events")

platt = fit_platt(s_calib, y_calib)
iso   = fit_isotonic(s_calib, y_calib)

df_cal["pd_platt"] = apply_calibrator(df_cal["pd_raw"].to_numpy(), platt, "platt")
df_cal["pd_iso"]   = apply_calibrator(df_cal["pd_raw"].to_numpy(), iso, "isotonic")

# Save calibrators for later
joblib.dump({"platt": platt, "isotonic": iso,
             "calib_periods": CALIB_PERIODS,
             "train_for_model_periods": TRAIN_FOR_MODEL,
             "final_features": final_features},
            DIAG_DIR / "calibrators.pkl")
print("Saved: diagnostics/calibrators.pkl")

Refit LR on 340,727 train_for_model rows, 22 features


Fitting calibrators on 48,798 calib rows, 451 events
Saved: diagnostics/calibrators.pkl


In [13]:
# Evaluate on OOT (untouched by calibration fitting)
mask_oot = df_cal["split_new"] == "oot"
y_oot = df_cal.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()
results = {}
for kind in ["pd_raw", "pd_platt", "pd_iso"]:
    s = df_cal.loc[mask_oot, kind].to_numpy()
    cal = compute_calibration_metrics(y_oot, s)
    results[kind] = {
        "auc": (compute_gini(y_oot, s) + 1) / 2,
        "gini": compute_gini(y_oot, s),
        "ks": compute_ks(y_oot, s),
        "brier": compute_brier(y_oot, s),
        "ece": cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }

print("=== Test 3 OOT metrics ===")
print(pd.DataFrame(results).round(5).to_string())

# Save
with open(DIAG_DIR / "test3_calibration.json", "w") as f:
    json.dump({
        "splits": {
            "train_for_model_periods": TRAIN_FOR_MODEL,
            "calib_periods": CALIB_PERIODS,
            "n_train_for_model": int(mask_tfm.sum()),
            "n_calib": int(mask_calib.sum()),
            "n_oot": int(mask_oot.sum()),
        },
        "oot_metrics_by_calibration": results,
    }, f, indent=2, default=str)
print("\nSaved: diagnostics/test3_calibration.json")

=== Test 3 OOT metrics ===
            pd_raw  pd_platt   pd_iso
auc        0.85909   0.85909  0.85732
gini       0.71817   0.71817  0.71464
ks         0.54837   0.54837  0.54352
brier      0.00821   0.00799  0.00800
ece        0.00879   0.00107  0.00079
mean_pred  0.01727   0.00921  0.00926
base_rate  0.00847   0.00847  0.00847

Saved: diagnostics/test3_calibration.json


## Test 4 — LightGBM baseline (dual-track)

Train LightGBM on **catalog-filtered PD-eligible raw features**, with native
categorical handling and `scale_pos_weight = n_neg / n_pos`. Same temporal
calibration split as Test 3 for leak-free calibration.

In [14]:
import lightgbm as lgb

# Use the SAME PD-eligible pool (with F6E) for fair LightGBM benchmark
df_lgb = pd.read_parquet(ABT_PATH, columns=all_pd_eligible + get_meta_columns())
df_lgb["split_new"] = make_calibration_split(df_lgb, TRAIN_FOR_MODEL, CALIB_PERIODS)
print(f"LightGBM df: {df_lgb.shape}")

# Identify categorical features by catalog (CATEGORICAL_SAFE in the original taxonomy)
CAT_FEATURES = ["app_nom_branch", "app_nom_gender", "app_nom_job_code",
                "app_nom_marital_status", "app_nom_city", "app_nom_home_status"]
cat_in_pool = [c for c in CAT_FEATURES if c in all_pd_eligible]
print(f"Categorical features in pool: {cat_in_pool}")

# Cast cats to pandas Categorical for native handling
for c in cat_in_pool:
    df_lgb[c] = df_lgb[c].astype("category")

mask_tfm  = df_lgb["split_new"] == "train_for_model"
mask_calib= df_lgb["split_new"] == "calib"
mask_oot  = df_lgb["split_new"] == "oot"

X_tr = df_lgb.loc[mask_tfm, all_pd_eligible]
y_tr = df_lgb.loc[mask_tfm, "default_flag_12m"].astype(int).to_numpy()
n_pos = int(y_tr.sum()); n_neg = int(len(y_tr) - n_pos)
spw = n_neg / max(n_pos, 1)
print(f"Train: {len(y_tr):,}  pos={n_pos}  neg={n_neg}  scale_pos_weight={spw:.2f}")

LightGBM df: (534314, 440)
Categorical features in pool: ['app_nom_branch', 'app_nom_gender', 'app_nom_job_code', 'app_nom_marital_status', 'app_nom_city', 'app_nom_home_status']


Train: 340,727  pos=5910  neg=334817  scale_pos_weight=56.65


In [15]:
# Use a small validation slice from train for early stopping
from sklearn.model_selection import train_test_split
X_tr_in, X_val, y_tr_in, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42,
)
print(f"Inner train: {len(y_tr_in):,}  Val: {len(y_val):,}")

t = time.time()
lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=200,
    reg_alpha=0.0, reg_lambda=1.0,
    colsample_bytree=0.7,
    subsample=0.7, subsample_freq=1,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm.fit(
    X_tr_in, y_tr_in,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    categorical_feature=cat_in_pool,
)
print(f"LightGBM wall: {time.time()-t:.1f}s, best_iter={lgbm.best_iteration_}")

Inner train: 289,617  Val: 51,110


LightGBM wall: 6.8s, best_iter=1


In [16]:
# Score everything
df_lgb["pd_lgb_raw"] = lgbm.predict_proba(df_lgb[all_pd_eligible])[:, 1]

# Calibrate on the calib slice (NEVER on OOT)
y_calib = df_lgb.loc[mask_calib, "default_flag_12m"].astype(int).to_numpy()
s_calib = df_lgb.loc[mask_calib, "pd_lgb_raw"].to_numpy()
platt_lgb = fit_platt(s_calib, y_calib)
iso_lgb   = fit_isotonic(s_calib, y_calib)
df_lgb["pd_lgb_platt"] = apply_calibrator(df_lgb["pd_lgb_raw"].to_numpy(), platt_lgb, "platt")
df_lgb["pd_lgb_iso"]   = apply_calibrator(df_lgb["pd_lgb_raw"].to_numpy(), iso_lgb, "isotonic")

# Metrics on OOT
y_oot = df_lgb.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()
lgb_results = {}
for kind in ["pd_lgb_raw", "pd_lgb_platt", "pd_lgb_iso"]:
    s = df_lgb.loc[mask_oot, kind].to_numpy()
    cal = compute_calibration_metrics(y_oot, s)
    lgb_results[kind] = {
        "auc": (compute_gini(y_oot, s) + 1) / 2,
        "gini": compute_gini(y_oot, s),
        "ks": compute_ks(y_oot, s),
        "brier": compute_brier(y_oot, s),
        "ece": cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }
print("=== LightGBM OOT metrics ===")
print(pd.DataFrame(lgb_results).round(5).to_string())

=== LightGBM OOT metrics ===
           pd_lgb_raw  pd_lgb_platt  pd_lgb_iso
auc           0.87262       0.87262     0.87040
gini          0.74525       0.74525     0.74080
ks            0.59611       0.59611     0.59611
brier         0.01099       0.00810     0.00809
ece           0.04492       0.00095     0.00084
mean_pred     0.05339       0.00938     0.00929
base_rate     0.00847       0.00847     0.00847


In [17]:
# Top 20 feature importances
imp_df = pd.DataFrame({
    "feature": all_pd_eligible,
    "importance_split": lgbm.booster_.feature_importance(importance_type="split"),
    "importance_gain":  lgbm.booster_.feature_importance(importance_type="gain"),
}).sort_values("importance_gain", ascending=False).head(20)
print("\nTop 20 LightGBM features by gain:")
print(imp_df.to_string(index=False))

# F6D check: any random_* in top 20?
f6d_in_top = [f for f in imp_df["feature"] if f.startswith("random_")]
print(f"\nF6D negative controls in top 20: {len(f6d_in_top)}")
if f6d_in_top:
    print(f"  Features: {f6d_in_top}")
else:
    print("  IDEAL: no random_* in top 20")


Top 20 LightGBM features by gain:
                   feature  importance_split  importance_gain
         job_code_x_income                 1    750759.000000
          age_x_n_children                 5    462439.830078
                   act_age                 5    340386.220215
   count_by_marital_status                 2    237152.101562
            app_nom_gender                 3    160303.500000
mean_age_by_marital_status                 2     52034.400391
synth_self_reported_income                 1     27663.699219
       synth_secured_limit                 1     19424.000000
       synth_highest_limit                 1     14785.200195
 synth_int_balance_x_score                 1      8894.320312
     synth_revolving_limit                 1      7941.419922
 synth_int_inc_x_nchildren                 1      7693.029785
    synth_available_credit                 1      7621.359863
   synth_auto_loan_balance                 1      7273.120117
 synth_household_income_v1         

In [18]:
# Save model + results
joblib.dump({"model": lgbm, "best_iteration": lgbm.best_iteration_,
             "scale_pos_weight": spw, "categorical_features": cat_in_pool,
             "platt": platt_lgb, "isotonic": iso_lgb,
             "feature_list": all_pd_eligible},
            REPO_ROOT / "artifacts/pd_model/lightgbm_model.pkl",
            )

test4_out = {
    "n_features": len(all_pd_eligible),
    "best_iter": int(lgbm.best_iteration_),
    "scale_pos_weight": float(spw),
    "oot_metrics": lgb_results,
    "top20_features": imp_df.to_dict(orient="records"),
    "f6d_in_top20": f6d_in_top,
}
with open(DIAG_DIR / "test4_lightgbm.json", "w") as f:
    json.dump(test4_out, f, indent=2, default=str)
print(f"\nSaved: diagnostics/test4_lightgbm.json")
print(f"Saved: artifacts/pd_model/lightgbm_model.pkl")


Saved: diagnostics/test4_lightgbm.json
Saved: artifacts/pd_model/lightgbm_model.pkl


## Test 5 — Score-discrimination stress-test design (no execution)

Design only — Phase 3/4 will execute. Generates calibrated PD score variants
at controlled Gini levels (0.30, 0.45, 0.60, raw) so the profit cut-off
analysis can compare regimes that isolate discrimination from calibration.

In [19]:
plan = StressTestPlan()
print("StressTestPlan:")
print(f"  target_ginis     : {plan.target_ginis}")
print(f"  tolerance        : {plan.tolerance}")
print(f"  base_rate_match  : {plan.base_rate_match}")
print(f"  seed             : {plan.seed}")
print(f"  notes            : {plan.notes}")

# Sanity-check: run perturbation on a small sample (10K rows) at one target
# to verify the API works, but do NOT generate the full stress-test set yet.
sample_idx = np.random.RandomState(42).choice(len(df_cal), size=10000, replace=False)
y_smp = df_cal.iloc[sample_idx]["default_flag_12m"].astype(int).to_numpy()
s_smp = df_cal.iloc[sample_idx]["pd_iso"].to_numpy()
print(f"\nSanity-check (10K sample, target_gini=0.30):")
sp, meta = perturb_to_target_gini(s_smp, y_smp, target_gini=0.30, seed=42, n_iter=20)
print(json.dumps(meta, indent=2))
print("\nFunction works. Phase 3 will run on full data.")

# Save plan + sanity output
test5_out = {
    "stress_test_plan": {
        "target_ginis": list(plan.target_ginis),
        "tolerance": plan.tolerance,
        "sigma_search_bounds": list(plan.sigma_search_bounds),
        "n_search_iter": plan.n_search_iter,
        "base_rate_match": plan.base_rate_match,
        "seed": plan.seed,
        "notes": plan.notes,
    },
    "sanity_check": meta,
    "executed": False,
    "execution_phase": "Phase 3/4",
}
with open(DIAG_DIR / "test5_stress_test_design.json", "w") as f:
    json.dump(test5_out, f, indent=2, default=str)
print("\nSaved: diagnostics/test5_stress_test_design.json")
print(f"\nTotal Phase 2A diagnostics wall: {time.time()-T0:.1f}s")

StressTestPlan:
  target_ginis     : (0.6, 0.45, 0.3)
  tolerance        : 0.005
  base_rate_match  : True
  seed             : 42
  notes            : Logit-noise perturbation; re-calibrate mean to base rate after perturbation so that profit-cutoff comparison isolates discrimination shift from calibration shift.

Sanity-check (10K sample, target_gini=0.30):
{
  "sigma": 3.0810546875,
  "achieved_gini": 0.30330410126582263,
  "target_gini": 0.3,
  "raw_gini": 0.6653974683544304,
  "tolerance_met": true,
  "re_calibrated_to_base_rate": true,
  "base_rate": 0.0125,
  "seed": 42
}

Function works. Phase 3 will run on full data.

Saved: diagnostics/test5_stress_test_design.json

Total Phase 2A diagnostics wall: 324.9s


## Summary

All 5 tests complete. Per-test deliverables saved under
`artifacts/phase2_rerun_v2/diagnostics/`:

- `test1_f6e_ablation.json`
- `test2_lasso_stability.json`
- `test3_calibration.json`
- `test4_lightgbm.json`
- `test5_stress_test_design.json`

Decision recommendation written to `phase2_modeling_report.md` (Phase 3 input).

# Section 3 - 01b_phase2_lightgbm_retune.ipynb

**Section metadata**

- **Purpose**: Phase 2B - LightGBM hyperparameter tuning via Optuna (30 trials) on the same wide ABT.
- **Main artifacts read**: `artifacts/wide_abt/wide_abt.parquet` and F6D-pruned feature list.
- **Main artifacts written**: `artifacts/pd_model/lightgbm_tuned_model.pkl` plus tuning report.
- **Expensive rerun expected**: Expensive (30 Optuna trials x cross-validation).

---


# Phase 2A — LightGBM retune

**Why**: the initial LightGBM run in Phase 2A diagnostics had `best_iteration=1`
(pathological early stopping) and `random_normal_10` (F6D pure noise) in the
top 20 by gain. Both symptoms point at config issues, not a defect in the
underlying tree-boosting approach.

**Fix plan**:
1. Replace the 15% random validation split with the **temporal validation
   split from Test 3** (cohorts 202611-202612). Same distribution as OOT;
   stops early stopping from over-/under-firing on a non-representative slice.
2. **Optuna search** (30 trials, TPE sampler) over:
   `learning_rate`, `num_leaves`, `min_child_samples`, `reg_alpha`,
   `reg_lambda`, `feature_fraction`, `bagging_fraction`. Cap at
   `n_estimators=2000` with `early_stopping_rounds=100`.
3. **Verification gates** after refit:
   - `best_iteration` > 50
   - F6D pure-random features (`random_*`) absent from top-20 gain
   - OOT Gini in [0.50, 0.75] sanity range
4. **Calibration** via Platt + isotonic on the same val set (same recipe as
   Test 3, leak-free w.r.t. OOT).
5. **Save** tuned model + tuning trace + feature importance + comparison.

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler

from src.governance import filter_pd_eligible, get_meta_columns
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
)

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUT_DIR      = REPO_ROOT / "artifacts/pd_model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
optuna.logging.set_verbosity(optuna.logging.WARNING)
catalog = pd.read_csv(CATALOG_PATH)
all_pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible pool: {len(all_pd_eligible)}")

PD-eligible pool: 435


In [2]:
# Load only the columns we need
df = pd.read_parquet(ABT_PATH, columns=all_pd_eligible + get_meta_columns())
print(f"ABT loaded: {df.shape}")

# Temporal calibration split (same as Test 3)
TRAIN_FOR_MODEL = list(range(202509, 202611))
CALIB_PERIODS   = [202611, 202612]
df["split_new"] = make_calibration_split(df, TRAIN_FOR_MODEL, CALIB_PERIODS)
print("\nSplit counts:")
print(df["split_new"].value_counts().to_string())

# Cast categoricals for native handling
CAT_FEATURES = ["app_nom_branch", "app_nom_gender", "app_nom_job_code",
                "app_nom_marital_status", "app_nom_city", "app_nom_home_status"]
cat_in_pool = [c for c in CAT_FEATURES if c in all_pd_eligible]
for c in cat_in_pool:
    df[c] = df[c].astype("category")
print(f"\nCategorical features in pool: {cat_in_pool}")

mask_tr  = df["split_new"] == "train_for_model"
mask_val = df["split_new"] == "calib"
mask_oot = df["split_new"] == "oot"

X_tr  = df.loc[mask_tr,  all_pd_eligible]
y_tr  = df.loc[mask_tr,  "default_flag_12m"].astype(int).to_numpy()
X_val = df.loc[mask_val, all_pd_eligible]
y_val = df.loc[mask_val, "default_flag_12m"].astype(int).to_numpy()
X_oot = df.loc[mask_oot, all_pd_eligible]
y_oot = df.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()

n_pos, n_neg = int(y_tr.sum()), int((1 - y_tr).sum())
spw = n_neg / max(n_pos, 1)
print(f"\nTrain: {len(y_tr):,} (pos={n_pos}, neg={n_neg})")
print(f"Val  : {len(y_val):,} (pos={int(y_val.sum())}, neg={int((1-y_val).sum())})")
print(f"OOT  : {len(y_oot):,} (pos={int(y_oot.sum())}, neg={int((1-y_oot).sum())})")
print(f"scale_pos_weight = {spw:.3f}")

ABT loaded: (534314, 439)



Split counts:
split_new
train_for_model    340727
oot                144789
calib               48798

Categorical features in pool: ['app_nom_branch', 'app_nom_gender', 'app_nom_job_code', 'app_nom_marital_status', 'app_nom_city', 'app_nom_home_status']



Train: 340,727 (pos=5910, neg=334817)
Val  : 48,798 (pos=451, neg=48347)
OOT  : 144,789 (pos=1227, neg=143562)
scale_pos_weight = 56.653


## Optuna search (30 trials)

Objective: maximise validation AUC at the early-stopping-best iteration.
Search space chosen to span aggressive (deep, low reg) to conservative
(shallow, high reg) configurations.

In [3]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "scale_pos_weight": spw,
        "n_jobs": -1,
        "seed": 42,
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "num_leaves":         trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples":  trial.suggest_int("min_child_samples", 20, 1000, log=True),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "feature_fraction":   trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction":   trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq":       1,
        "max_depth":          -1,
    }
    dtr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_in_pool, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_in_pool,
                       reference=dtr, free_raw_data=False)
    model = lgb.train(
        params, dtr,
        num_boost_round=2000,
        valid_sets=[dval],
        valid_names=["val"],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    return model.best_score["val"]["auc"]

study = optuna.create_study(direction="maximize",
                             sampler=TPESampler(seed=42, n_startup_trials=8))
t0 = time.time()
study.optimize(objective, n_trials=30, show_progress_bar=False)
print(f"\nOptuna search wall: {time.time()-t0:.1f}s")
print(f"Best val AUC: {study.best_value:.5f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k:<20} {v}")


Optuna search wall: 755.5s
Best val AUC: 0.86817
Best params:
  learning_rate        0.03912141628549695
  num_leaves           20
  min_child_samples    213
  reg_alpha            0.004809461967501573
  reg_lambda           0.0018205657658407262
  feature_fraction     0.9744427686266666
  bagging_fraction     0.9828160165372797


## Refit best parameters

Train with the best hyperparameters and capture the converged `best_iteration`.

In [4]:
best_params = dict(study.best_params)
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "scale_pos_weight": spw,
    "n_jobs": -1,
    "seed": 42,
    "max_depth": -1,
    "bagging_freq": 1,
})

dtr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_in_pool, free_raw_data=False)
dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_in_pool,
                   reference=dtr, free_raw_data=False)
t0 = time.time()
booster = lgb.train(
    best_params, dtr,
    num_boost_round=2000,
    valid_sets=[dval],
    valid_names=["val"],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)
print(f"Refit wall: {time.time()-t0:.1f}s")
print(f"best_iteration: {booster.best_iteration}")
print(f"best_score val AUC: {booster.best_score['val']['auc']:.5f}")

Refit wall: 18.6s
best_iteration: 141
best_score val AUC: 0.86817


In [5]:
# Score everything (train, val, oot)
df["pd_lgb_raw"] = booster.predict(df[all_pd_eligible], num_iteration=booster.best_iteration)

# Calibrate on val (NOT OOT)
s_val = df.loc[mask_val, "pd_lgb_raw"].to_numpy()
y_val_arr = y_val
platt = fit_platt(s_val, y_val_arr)
iso   = fit_isotonic(s_val, y_val_arr)
df["pd_lgb_platt"] = apply_calibrator(df["pd_lgb_raw"].to_numpy(), platt, "platt")
df["pd_lgb_iso"]   = apply_calibrator(df["pd_lgb_raw"].to_numpy(), iso,   "isotonic")

# OOT metrics
def metrics(y, s):
    cal = compute_calibration_metrics(y, s)
    return {
        "auc"      : (compute_gini(y, s) + 1) / 2,
        "gini"     : compute_gini(y, s),
        "ks"       : compute_ks(y, s),
        "brier"    : compute_brier(y, s),
        "ece"      : cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }

oot_results = {
    "pd_lgb_raw"  : metrics(y_oot, df.loc[mask_oot, "pd_lgb_raw"].to_numpy()),
    "pd_lgb_platt": metrics(y_oot, df.loc[mask_oot, "pd_lgb_platt"].to_numpy()),
    "pd_lgb_iso"  : metrics(y_oot, df.loc[mask_oot, "pd_lgb_iso"].to_numpy()),
}
print("=== Tuned LightGBM OOT metrics ===")
print(pd.DataFrame(oot_results).round(5).to_string())

=== Tuned LightGBM OOT metrics ===
           pd_lgb_raw  pd_lgb_platt  pd_lgb_iso
auc           0.89754       0.89754     0.89625
gini          0.79508       0.79508     0.79250
ks            0.63799       0.63799     0.63436
brier         0.14355       0.00788     0.00783
ece           0.28861       0.00124     0.00091
mean_pred     0.29709       0.00939     0.00931
base_rate     0.00847       0.00847     0.00847


## Verification gates

- `best_iteration` > 50
- No F6D (`random_*`) feature in top-20 gain
- OOT Gini in [0.50, 0.75]

In [6]:
# Feature importance
imp_split = booster.feature_importance(importance_type="split")
imp_gain  = booster.feature_importance(importance_type="gain")
imp_df = pd.DataFrame({
    "feature": all_pd_eligible,
    "importance_split": imp_split,
    "importance_gain":  imp_gain,
}).sort_values("importance_gain", ascending=False).reset_index(drop=True)

top20 = imp_df.head(20)
print("Top 20 LightGBM (tuned) features by gain:")
print(top20.to_string(index=False))

f6d_in_top20 = [f for f in top20["feature"] if f.startswith("random_")]
print(f"\nF6D in top-20 gain: {len(f6d_in_top20)}  {f6d_in_top20}")

# Also show "F6D anywhere" to confirm they are properly suppressed
f6d_all = imp_df[imp_df["feature"].str.startswith("random_")]
f6d_any = f6d_all[f6d_all["importance_gain"] > 0]
print(f"F6D with non-zero gain anywhere: {len(f6d_any)} of {len(f6d_all)}")
if len(f6d_any) > 0:
    print(f6d_any.head(10).to_string(index=False))

Top 20 LightGBM (tuned) features by gain:
                      feature  importance_split  importance_gain
            job_code_x_income                93     1.392351e+06
       app_nom_marital_status                37     7.020252e+05
             age_x_n_children                63     4.260934e+05
               app_nom_gender                91     3.364310e+05
    synth_int_inc_x_nchildren                30     2.938565e+05
         synth_thin_file_flag                49     2.830825e+05
                act_age_noisy                24     2.633465e+05
    synth_household_income_v1                 5     6.962889e+04
         mean_age_by_job_code                31     6.858058e+04
                      act_age                29     6.067845e+04
mean_income_by_marital_status                42     4.972123e+04
  synth_int_spend_x_nchildren                29     4.760228e+04
         marital_x_n_children                32     3.849148e+04
      synth_employment_months                 7 

In [7]:
gates = {
    "best_iteration_gt_50":  booster.best_iteration > 50,
    "f6d_in_top20_zero":     len(f6d_in_top20) == 0,
    "oot_gini_in_range":     0.50 <= oot_results["pd_lgb_raw"]["gini"] <= 0.75,
}
print("Verification gates:")
for k, v in gates.items():
    print(f"  [{('PASS' if v else 'FAIL')}] {k}")

all_pass = all(gates.values())
print(f"\nOverall: {'ALL PASS' if all_pass else 'SOME FAILED — investigate'}")

Verification gates:
  [PASS] best_iteration_gt_50
  [PASS] f6d_in_top20_zero
  [FAIL] oot_gini_in_range

Overall: SOME FAILED — investigate


## Save artifacts

In [8]:
# Save model
joblib.dump({
    "booster": booster,
    "best_params": best_params,
    "best_iteration": int(booster.best_iteration),
    "best_val_auc": float(booster.best_score["val"]["auc"]),
    "scale_pos_weight": float(spw),
    "categorical_features": cat_in_pool,
    "platt": platt,
    "isotonic": iso,
    "feature_list": all_pd_eligible,
    "train_for_model_periods": TRAIN_FOR_MODEL,
    "calib_periods": CALIB_PERIODS,
}, OUT_DIR / "lightgbm_tuned_model.pkl")
print(f"Saved: {OUT_DIR / 'lightgbm_tuned_model.pkl'}")

# Save feature importance
imp_df.to_csv(OUT_DIR / "lightgbm_feature_importance.csv", index=False)
print(f"Saved: {OUT_DIR / 'lightgbm_feature_importance.csv'}")

# Save tuning results
trial_records = []
for t in study.trials:
    trial_records.append({
        "trial": t.number,
        "value": t.value,
        "params": t.params,
        "state": str(t.state),
    })
tuning_out = {
    "n_trials": len(study.trials),
    "best_trial": study.best_trial.number,
    "best_value": float(study.best_value),
    "best_params": study.best_params,
    "post_refit_best_iteration": int(booster.best_iteration),
    "post_refit_best_val_auc":   float(booster.best_score["val"]["auc"]),
    "scale_pos_weight": float(spw),
    "splits": {
        "train_for_model_periods": TRAIN_FOR_MODEL,
        "calib_periods": CALIB_PERIODS,
        "n_train": int(mask_tr.sum()),
        "n_val":   int(mask_val.sum()),
        "n_oot":   int(mask_oot.sum()),
    },
    "oot_metrics": oot_results,
    "verification_gates": gates,
    "f6d_in_top20": list(f6d_in_top20),
    "trials": trial_records,
}
with open(OUT_DIR / "lightgbm_tuning_results.json", "w") as f:
    json.dump(tuning_out, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'lightgbm_tuning_results.json'}")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_tuned_model.pkl
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_feature_importance.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_tuning_results.json


## Comparison with LR variants and pre-retune LightGBM

In [9]:
# Pre-retune LightGBM metrics (from diagnostics test 4)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test4_lightgbm.json") as f:
    pre_retune = json.load(f)

# LR variants (full-F6E from initial Phase 2A; no-F6E from Test 1)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/model_metrics.json") as f:
    lr_full = json.load(f)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test1_f6e_ablation.json") as f:
    lr_nof6e = json.load(f)

cmp = pd.DataFrame({
    "LR_full_F6E_oot":  lr_full["oot"],
    "LR_no_F6E_oot":    lr_nof6e["metrics"]["oot"],
    "LGB_pre_retune_raw_oot":   pre_retune["oot_metrics"]["pd_lgb_raw"],
    "LGB_pre_retune_platt_oot": pre_retune["oot_metrics"]["pd_lgb_platt"],
    "LGB_tuned_raw_oot":        oot_results["pd_lgb_raw"],
    "LGB_tuned_platt_oot":      oot_results["pd_lgb_platt"],
})
print("=== Side-by-side OOT comparison ===")
print(cmp.round(5).to_string())

# Best iteration comparison
print(f"\nbest_iteration: pre-retune=1, post-retune={booster.best_iteration}")
print(f"F6D in top-20 gain: pre-retune=1, post-retune={len(f6d_in_top20)}")

# Top-7 feature comparison: LR no-F6E uses 7 features
LR_NO_F6E = lr_nof6e["final_features"]
print(f"\nLR (no-F6E) final 7 features:")
for f in LR_NO_F6E:
    rank = imp_df[imp_df['feature'] == f].index
    rank_str = f"LGB rank {int(rank[0])+1}" if len(rank) > 0 else "(not in pool)"
    print(f"  {f:<35}  {rank_str}")

print(f"\nLightGBM (tuned) top-7 by gain:")
for i, row in imp_df.head(7).iterrows():
    print(f"  {row['feature']:<35}  gain={row['importance_gain']:>10.0f}  splits={row['importance_split']}")

print(f"\nTotal Phase 2A LGB retune wall: {time.time() - T0:.1f}s")

=== Side-by-side OOT comparison ===
           LR_full_F6E_oot  LR_no_F6E_oot  LGB_pre_retune_raw_oot  LGB_pre_retune_platt_oot  LGB_tuned_raw_oot  LGB_tuned_platt_oot
gini               0.71812        0.67226                 0.74525                   0.74525            0.79508              0.79508
ks                 0.54755        0.53948                 0.59611                   0.59611            0.63799              0.63799
brier              0.00816        0.00821                 0.01099                   0.00810            0.14355              0.00788
auc                0.85906        0.83613                 0.87262                   0.87262            0.89754              0.89754
ece                    NaN            NaN                 0.04492                   0.00095            0.28861              0.00124
mean_pred              NaN            NaN                 0.05339                   0.00938            0.29709              0.00939
base_rate              NaN            Na

# Section 4 - 01c_scorecard.ipynb

**Section metadata**

- **Purpose**: Phase 2 alternative - Weight-of-Evidence Scorecard build using OptBinning on the no-F6E feature set.
- **Main artifacts read**: Selected features and wide ABT.
- **Main artifacts written**: `artifacts/pd_model/scorecard_model.pkl` and scorecard tables.
- **Expensive rerun expected**: Expensive (OptBinning sweep).

---


# Phase 2B — Scorecard

**Primary scorecard**: 7-feature LR no-F6E (interpretable, less dependent on synthetic bureau).
**Robustness scorecard** (optional, end of notebook): 22-feature LR full-F6E.

**Workflow:**
1. Fit WoE binning on `train_for_model` cohorts only (202509-202610)
2. Numeric: monotonic constraint via merge logic
3. Categorical: event-rate-sorted with min-bin-size merging
4. Freeze bins → apply to calib (202611-202612) and OOT (202701-202706)
5. Refit logistic regression on WoE-transformed features
6. Scorecard with `factor = 20/log(2)`, `base_score = 300`, `PDO = 20`, `base_odds = 1:50`
7. Platt calibration on calib split (NEVER OOT)
8. Compare: LR no-F6E raw / LR no-F6E scorecard / LightGBM primary

**Acceptance** (Gini drop scorecard vs raw LR no-F6E on OOT):
- < 0.05 → good
- 0.05-0.10 → acceptable with documentation
- > 0.10 → illustrative only, not production-defensible

**Note**: optbinning 0.20.0 has a sklearn 1.8 API mismatch in this environment.
We use a custom WoE binner in `src/scorecard.py` with the same monotonic-merge
logic (quantile init + iterative merge until WoE is monotonic).

In [1]:
import sys, time, json, joblib, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.governance import filter_pd_eligible, get_meta_columns
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
)
from src.scorecard import (
    NumericBinner, CategoricalBinner, apply_woe,
    build_scorecard, score_from_woe,
)

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUT_DIR      = REPO_ROOT / "artifacts/scorecard"
OUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
catalog = pd.read_csv(CATALOG_PATH)
print(f"Loaded catalog: {len(catalog):,} rows")

Loaded catalog: 2,236 rows


## 1. Load 7-feature LR no-F6E features + temporal split

In [2]:
# 7-feature set from Test 1 (LR no-F6E)
FEATURES_NO_F6E = [
    "app_nom_job_code",      # categorical
    "act_age_log1p",         # numeric
    "mean_cars_by_job_code", # numeric (F5A group stat)
    "app_nom_marital_status",# categorical
    "median_income_by_job_code", # numeric (F5A)
    "app_nom_branch",        # categorical
    "app_nom_city",          # categorical
]
NUMERIC_FEATS     = ["act_age_log1p", "mean_cars_by_job_code", "median_income_by_job_code"]
CATEGORICAL_FEATS = ["app_nom_job_code", "app_nom_marital_status", "app_nom_branch", "app_nom_city"]

df = pd.read_parquet(ABT_PATH, columns=FEATURES_NO_F6E + get_meta_columns())
print(f"Loaded ABT: {df.shape}")

# Temporal calibration split (same as Test 3)
TRAIN_FOR_MODEL = list(range(202509, 202611))
CALIB_PERIODS   = [202611, 202612]
df["split_new"] = make_calibration_split(df, TRAIN_FOR_MODEL, CALIB_PERIODS)
print("\nSplit counts (new temporal split):")
print(df["split_new"].value_counts().to_string())

mask_tr  = df["split_new"] == "train_for_model"
mask_cal = df["split_new"] == "calib"
mask_oot = df["split_new"] == "oot"

print("\nDR by new split:")
for s in ["train_for_model", "calib", "oot"]:
    sub = df[df["split_new"]==s]
    print(f"  {s:<18} n={len(sub):>7,}  DR={(sub['default_flag_12m']==1).mean():.4%}")

Loaded ABT: (534314, 11)

Split counts (new temporal split):
split_new
train_for_model    340727
oot                144789
calib               48798

DR by new split:
  train_for_model    n=340,727  DR=1.7345%
  calib              n= 48,798  DR=0.9242%


  oot                n=144,789  DR=0.8474%


## 2. Fit WoE binning on `train_for_model`

- Numeric: 20 initial quantile bins → merge to enforce monotonicity + min_bin_size 5%
- Categorical: event-rate sorted → merge to ensure each bin ≥ 5% of train rows

In [3]:
binners = {}
y_tr = df.loc[mask_tr, "default_flag_12m"].astype(int)

for f in NUMERIC_FEATS:
    b = NumericBinner(feature=f, n_initial_bins=20, min_bin_size_frac=0.05, monotonic=True)
    b.fit(df.loc[mask_tr, f], y_tr)
    binners[f] = b
    print(f"NUM  {f:<32}  n_bins={len(b.bin_woe):>2}  IV={b.iv:.4f}  trend={b.monotonic_trend}")

for f in CATEGORICAL_FEATS:
    b = CategoricalBinner(feature=f, min_bin_size_frac=0.05)
    b.fit(df.loc[mask_tr, f], y_tr)
    binners[f] = b
    print(f"CAT  {f:<32}  n_bins={len(b.bin_woe):>2}  IV={b.iv:.4f}")

NUM  act_age_log1p                     n_bins=11  IV=0.3630  trend=descending
NUM  mean_cars_by_job_code             n_bins= 3  IV=0.2650  trend=descending
NUM  median_income_by_job_code         n_bins= 2  IV=0.2310  trend=ascending


CAT  app_nom_job_code                  n_bins= 3  IV=0.3923


CAT  app_nom_marital_status            n_bins= 3  IV=0.2820


CAT  app_nom_branch                    n_bins= 4  IV=0.0230


CAT  app_nom_city                      n_bins= 4  IV=0.0169


In [4]:
# Print the binning tables
for f in FEATURES_NO_F6E:
    b = binners[f]
    print(f"\n=== {f} (IV={b.iv:.4f}) ===")
    print(f"{'bin':>3}  {'label':<35}  {'n':>7}  {'events':>6}  {'ER':>7}  {'WoE':>7}")
    for i in range(len(b.bin_woe)):
        print(f"{i:>3}  {b.get_bin_label(i):<35}  {b.bin_count[i]:>7,}  "
              f"{b.bin_events[i]:>6}  {b.bin_event_rate[i]:>6.4%}  {b.bin_woe[i]:>+7.4f}")


=== app_nom_job_code (IV=0.3923) ===
bin  label                                      n  events       ER      WoE
  0  4                                    184,598    1835  0.9941%  -0.5641
  1  3                                     65,342     748  1.1447%  -0.4210
  2  2, 1                                  90,787    3327  3.6646%  +0.7678

=== act_age_log1p (IV=0.3630) ===
bin  label                                      n  events       ER      WoE
  0  (-inf, 3.932)                         44,608    1750  3.9231%  +0.8388
  1  [3.932, 3.97)                         31,167     872  2.7978%  +0.4894
  2  [3.97, 3.989)                         17,144     409  2.3857%  +0.3264
  3  [3.989, 4.025)                        17,087     362  2.1186%  +0.2051
  4  [4.025, 4.043)                        33,970     605  1.7810%  +0.0275
  5  [4.043, 4.078)                        33,847     484  1.4300%  -0.1953
  6  [4.078, 4.127)                        34,064     396  1.1625%  -0.4049
  7  [4.127, 4.

## 3. Freeze bins, apply to all splits, refit LR

In [5]:
df_woe = apply_woe(df, binners)
woe_cols = [f"{f}_woe" for f in FEATURES_NO_F6E]
print(f"WoE columns: {woe_cols}")

# Refit LR on train_for_model
X_tr = df_woe.loc[mask_tr, woe_cols]
y_tr_arr = df_woe.loc[mask_tr, "default_flag_12m"].astype(int)
X_tr_const = sm.add_constant(X_tr, has_constant="add")
model_woe = sm.Logit(y_tr_arr, X_tr_const).fit(disp=False, maxiter=200)
print("\nWoE LR coefficients:")
for c in model_woe.params.index:
    print(f"  {c:<40}  beta={model_woe.params[c]:+.5f}  p={model_woe.pvalues[c]:.4g}")

WoE columns: ['app_nom_job_code_woe', 'act_age_log1p_woe', 'mean_cars_by_job_code_woe', 'app_nom_marital_status_woe', 'median_income_by_job_code_woe', 'app_nom_branch_woe', 'app_nom_city_woe']



WoE LR coefficients:
  const                                     beta=-3.19662  p=nan
  app_nom_job_code_woe                      beta=+0.74297  p=4.835e-236
  act_age_log1p_woe                         beta=+0.49952  p=9.104e-91
  mean_cars_by_job_code_woe                 beta=+1.16314  p=0
  app_nom_marital_status_woe                beta=+1.10834  p=0
  median_income_by_job_code_woe             beta=-1.33466  p=nan
  app_nom_branch_woe                        beta=+1.06457  p=3.733e-33
  app_nom_city_woe                          beta=+1.04892  p=2.501e-24


## 4. Build scorecard

`factor = PDO / log(2)`, `base_score = 300`, `PDO = 20`, `base_odds = 1:50`.
Higher score = lower default risk.

In [6]:
PDO = 20.0
BASE_SCORE = 300.0
BASE_ODDS = 1.0 / 50.0

feature_betas = {f: model_woe.params[f"{f}_woe"] for f in FEATURES_NO_F6E}
intercept = float(model_woe.params["const"])

scorecard = build_scorecard(
    feature_betas=feature_betas,
    intercept=intercept,
    binners=binners,
    base_score=BASE_SCORE,
    pdo=PDO,
    base_odds=BASE_ODDS,
)
print(f"Scorecard rows: {len(scorecard)}")
print(scorecard.to_string(index=False))

Scorecard rows: 30
                  feature  bin_index      bin_label      n  n_events  event_rate       woe      beta  points_from_woe  points_intercept_share  points_total
         app_nom_job_code          0              4 184598      1835    0.009941 -0.564104  0.742965            12.09                   72.16         84.25
         app_nom_job_code          1              3  65342       748    0.011447 -0.421041  0.742965             9.03                   72.16         81.18
         app_nom_job_code          2           2, 1  90787      3327    0.036646  0.767806  0.742965           -16.46                   72.16         55.70
            act_age_log1p          0  (-inf, 3.932)  44608      1750    0.039231  0.838770  0.499518           -12.09                   72.16         60.07
            act_age_log1p          1  [3.932, 3.97)  31167       872    0.027978  0.489380  0.499518            -7.05                   72.16         65.11
            act_age_log1p          2  [3.97, 

## 5. Score all splits; PD via WoE-LR + Platt calibration on calib only

In [7]:
# WoE-LR predicted probabilities on full data (all splits)
df_woe_const = sm.add_constant(df_woe[woe_cols], has_constant="add")
df["pd_woe_raw"] = model_woe.predict(df_woe_const)

# Total scorecard points per row (additive, higher = lower risk)
df["scorecard_points"] = score_from_woe(df_woe, scorecard)

# Calibrate via Platt on calib only (NEVER OOT)
y_cal = df.loc[mask_cal, "default_flag_12m"].astype(int).to_numpy()
s_cal = df.loc[mask_cal, "pd_woe_raw"].to_numpy()
platt = fit_platt(s_cal, y_cal)
df["pd_woe_platt"] = apply_calibrator(df["pd_woe_raw"].to_numpy(), platt, "platt")

print(f"Train-for-model PD raw mean    : {df.loc[mask_tr, 'pd_woe_raw'].mean():.4%}")
print(f"Train-for-model DR             : {(df.loc[mask_tr, 'default_flag_12m']==1).mean():.4%}")
print(f"Calib PD raw mean              : {df.loc[mask_cal, 'pd_woe_raw'].mean():.4%}")
print(f"OOT  PD raw mean               : {df.loc[mask_oot, 'pd_woe_raw'].mean():.4%}")
print(f"OOT  PD Platt mean             : {df.loc[mask_oot, 'pd_woe_platt'].mean():.4%}")
print(f"OOT  base rate                 : {(df.loc[mask_oot, 'default_flag_12m']==1).mean():.4%}")

Train-for-model PD raw mean    : 1.7345%
Train-for-model DR             : 1.7345%
Calib PD raw mean              : 1.7501%
OOT  PD raw mean               : 1.7300%
OOT  PD Platt mean             : 0.9156%
OOT  base rate                 : 0.8474%


## 6. Performance comparison on OOT

In [8]:
def metrics_at(y, s):
    cal = compute_calibration_metrics(y, s)
    return {
        "auc"   : (compute_gini(y, s) + 1) / 2,
        "gini"  : compute_gini(y, s),
        "ks"    : compute_ks(y, s),
        "brier" : compute_brier(y, s),
        "ece"   : cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }

y_oot_arr = df.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()

scorecard_oot_raw   = metrics_at(y_oot_arr, df.loc[mask_oot, "pd_woe_raw"].to_numpy())
scorecard_oot_platt = metrics_at(y_oot_arr, df.loc[mask_oot, "pd_woe_platt"].to_numpy())

# Raw LR no-F6E baseline (Phase 2A Test 1)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test1_f6e_ablation.json") as f:
    lr_nof6e = json.load(f)
lr_nof6e_oot_raw = lr_nof6e["metrics"]["oot"]

# LightGBM primary (Phase 2A retune)
with open(REPO_ROOT / "artifacts/pd_model/lightgbm_tuning_results.json") as f:
    lgb_results = json.load(f)
lgb_oot_platt = lgb_results["oot_metrics"]["pd_lgb_platt"]

# LR full-F6E + Platt (Phase 2A Test 3)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test3_calibration.json") as f:
    test3 = json.load(f)
lr_full_oot_platt = test3["oot_metrics_by_calibration"]["pd_platt"]

cmp = pd.DataFrame({
    "LR_no_F6E_raw_oot":        lr_nof6e_oot_raw,
    "Scorecard_no_F6E_raw_oot": scorecard_oot_raw,
    "Scorecard_no_F6E_platt_oot": scorecard_oot_platt,
    "LR_full_F6E_platt_oot":    lr_full_oot_platt,
    "LightGBM_tuned_platt_oot": lgb_oot_platt,
})
print("=== Side-by-side OOT comparison ===")
print(cmp.round(5).to_string())

# Acceptance tier vs raw LR no-F6E
gini_raw_lr = lr_nof6e_oot_raw["gini"]
gini_drop_raw = gini_raw_lr - scorecard_oot_raw["gini"]
gini_drop_platt = gini_raw_lr - scorecard_oot_platt["gini"]
print(f"\nAcceptance test (raw LR no-F6E vs scorecard):")
print(f"  raw scorecard:   gini drop = {gini_drop_raw:+.4f}")
print(f"  Platt scorecard: gini drop = {gini_drop_platt:+.4f}")

def tier(d):
    if d < 0.05: return "good (<0.05)"
    if d < 0.10: return "acceptable (0.05-0.10)"
    return "illustrative only (>0.10)"

tier_raw = tier(gini_drop_raw)
tier_platt = tier(gini_drop_platt)
print(f"  Tier (raw):   {tier_raw}")
print(f"  Tier (Platt): {tier_platt}")

=== Side-by-side OOT comparison ===
           LR_no_F6E_raw_oot  Scorecard_no_F6E_raw_oot  Scorecard_no_F6E_platt_oot  LR_full_F6E_platt_oot  LightGBM_tuned_platt_oot
auc                  0.83613                   0.79939                     0.79939                0.85909                   0.89754
gini                 0.67226                   0.59877                     0.59877                0.71817                   0.79508
ks                   0.53948                   0.47683                     0.47683                0.54837                   0.63799
brier                0.00821                   0.00824                     0.00811                0.00799                   0.00788
ece                      NaN                   0.00883                     0.00200                0.00107                   0.00124
mean_pred                NaN                   0.01730                     0.00916                0.00921                   0.00939
base_rate                NaN            

## 7. Save artifacts

In [9]:
# 7a. optbin_definitions.json (binner state, JSON-friendly)
def serialize_num(b):
    return {
        "type": "numeric",
        "feature": b.feature,
        "n_initial_bins": b.n_initial_bins,
        "min_bin_size_frac": b.min_bin_size_frac,
        "monotonic": b.monotonic,
        "cuts": list(b.cuts),
        "bin_woe": list(b.bin_woe),
        "bin_event_rate": list(b.bin_event_rate),
        "bin_count": list(b.bin_count),
        "bin_events": list(b.bin_events),
        "iv": b.iv,
        "monotonic_trend": b.monotonic_trend,
    }
def serialize_cat(b):
    return {
        "type": "categorical",
        "feature": b.feature,
        "min_bin_size_frac": b.min_bin_size_frac,
        "category_to_bin": {str(k): int(v) for k, v in b.category_to_bin.items()},
        "bin_categories": [[str(c) for c in cats] for cats in b.bin_categories],
        "bin_woe": list(b.bin_woe),
        "bin_event_rate": list(b.bin_event_rate),
        "bin_count": list(b.bin_count),
        "bin_events": list(b.bin_events),
        "iv": b.iv,
    }

binner_defs = {}
for f, b in binners.items():
    binner_defs[f] = serialize_num(b) if isinstance(b, NumericBinner) else serialize_cat(b)
with open(OUT_DIR / "optbin_definitions.json", "w") as f:
    json.dump(binner_defs, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'optbin_definitions.json'}")

# 7b. scorecard_table.csv
scorecard.to_csv(OUT_DIR / "scorecard_table.csv", index=False)
print(f"Saved: {OUT_DIR / 'scorecard_table.csv'}")

# 7c. WoE-transformed ABT (slim version, only essentials)
slim = df_woe[woe_cols + ["aid", "split", "split_new", "default_flag_12m", "fin_period"]].copy()
slim["pd_woe_raw"] = df["pd_woe_raw"]
slim["pd_woe_platt"] = df["pd_woe_platt"]
slim["scorecard_points"] = df["scorecard_points"]
slim.to_parquet(OUT_DIR / "woe_transformed_abt.parquet", index=False)
print(f"Saved: {OUT_DIR / 'woe_transformed_abt.parquet'}  ({len(slim):,} rows)")

# 7d. metrics JSON
metrics_out = {
    "splits": {
        "train_for_model_periods": TRAIN_FOR_MODEL,
        "calib_periods": CALIB_PERIODS,
        "n_train_for_model": int(mask_tr.sum()),
        "n_calib": int(mask_cal.sum()),
        "n_oot": int(mask_oot.sum()),
    },
    "features": FEATURES_NO_F6E,
    "scorecard_constants": {
        "factor": PDO / math.log(2),
        "base_score": BASE_SCORE,
        "pdo": PDO,
        "base_odds": BASE_ODDS,
    },
    "iv_per_feature": {f: float(binners[f].iv) for f in FEATURES_NO_F6E},
    "model_betas": {c: float(model_woe.params[c]) for c in model_woe.params.index},
    "model_pvalues": {c: float(model_woe.pvalues[c]) for c in model_woe.pvalues.index},
    "oot_metrics": {
        "scorecard_raw":   scorecard_oot_raw,
        "scorecard_platt": scorecard_oot_platt,
        "ref_lr_no_f6e_raw":      lr_nof6e_oot_raw,
        "ref_lr_full_f6e_platt":  lr_full_oot_platt,
        "ref_lightgbm_platt":     lgb_oot_platt,
    },
    "acceptance": {
        "gini_drop_vs_lr_no_f6e_raw": float(gini_drop_raw),
        "gini_drop_vs_lr_no_f6e_platt": float(gini_drop_platt),
        "tier_raw": tier_raw,
        "tier_platt": tier_platt,
    },
}
with open(OUT_DIR / "scorecard_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'scorecard_metrics.json'}")

# 7e. 1-page summary xlsx
import openpyxl
from openpyxl import Workbook
wb = Workbook()
ws = wb.active
ws.title = "Summary"
rows = [
    ["Phase 2B Scorecard Summary", ""],
    ["", ""],
    ["Features", ", ".join(FEATURES_NO_F6E)],
    ["n train_for_model", int(mask_tr.sum())],
    ["n calib", int(mask_cal.sum())],
    ["n oot", int(mask_oot.sum())],
    ["", ""],
    ["IV (sum)", round(sum(binner_defs[f]['iv'] for f in FEATURES_NO_F6E), 4)],
    ["Acceptance tier (Platt)", tier_platt],
    ["Gini drop vs raw LR (Platt)", round(gini_drop_platt, 4)],
    ["", ""],
    ["OOT metric", "Scorecard Platt"],
    ["AUC", round(scorecard_oot_platt['auc'], 4)],
    ["Gini", round(scorecard_oot_platt['gini'], 4)],
    ["KS", round(scorecard_oot_platt['ks'], 4)],
    ["Brier", round(scorecard_oot_platt['brier'], 5)],
    ["ECE", round(scorecard_oot_platt['ece'], 5)],
    ["Mean predicted", round(scorecard_oot_platt['mean_pred'], 5)],
    ["Base rate", round(scorecard_oot_platt['base_rate'], 5)],
]
for r in rows:
    ws.append(r)
ws2 = wb.create_sheet("Comparison")
ws2.append(["model", "AUC", "Gini", "KS", "Brier", "ECE", "mean_pred"])
for name, m in [
    ("LR no-F6E raw",         lr_nof6e_oot_raw),
    ("Scorecard no-F6E raw",  scorecard_oot_raw),
    ("Scorecard no-F6E Platt",scorecard_oot_platt),
    ("LR full-F6E Platt",     lr_full_oot_platt),
    ("LightGBM tuned Platt",  lgb_oot_platt),
]:
    ws2.append([name,
                round(m['auc'], 4),
                round(m['gini'], 4),
                round(m.get('ks', 0), 4),
                round(m['brier'], 5),
                round(m.get('ece', 0), 5),
                round(m.get('mean_pred', 0), 5)])
ws3 = wb.create_sheet("Scorecard")
ws3.append(list(scorecard.columns))
for _, row in scorecard.iterrows():
    ws3.append([row[c] for c in scorecard.columns])
wb.save(OUT_DIR / "scorecard_summary.xlsx")
print(f"Saved: {OUT_DIR / 'scorecard_summary.xlsx'}")
print(f"\nWall (Phase 2B): {time.time()-T0:.1f}s")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\optbin_definitions.json
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\scorecard_table.csv


Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\woe_transformed_abt.parquet  (534,314 rows)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\scorecard_metrics.json


Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\scorecard_summary.xlsx

Wall (Phase 2B): 3.7s


## 8. (OPTIONAL) Robustness scorecard — 22-feature full-F6E

Only run if Section 1-7 succeeded. Same WoE workflow on the LR full-F6E final
features. Saved separately to `artifacts/scorecard/robustness_full_f6e/`.

In [10]:
import time
T1 = time.time()
ROB_DIR = OUT_DIR / "robustness_full_f6e"
ROB_DIR.mkdir(parents=True, exist_ok=True)

# Load 22-feature set from Phase 2A
final_features_full = pd.read_csv(REPO_ROOT / "artifacts/phase2_rerun_v2/final_feature_set.csv")["feature"].tolist()
print(f"Robustness: 22-feature LR full-F6E features:")
for f in final_features_full:
    print(f"  - {f}")

# Determine numeric vs categorical via catalog
cat_idx = catalog.set_index("feature_name")
KNOWN_CAT = {"app_nom_branch","app_nom_gender","app_nom_job_code",
             "app_nom_marital_status","app_nom_city","app_nom_home_status"}
ROB_NUMERIC = []; ROB_CAT = []
for f in final_features_full:
    if f in KNOWN_CAT:
        ROB_CAT.append(f)
    else:
        ROB_NUMERIC.append(f)
print(f"\nNumeric: {len(ROB_NUMERIC)}, Categorical: {len(ROB_CAT)}")

Robustness: 22-feature LR full-F6E features:
  - app_nom_job_code
  - synth_age_of_newest
  - synth_newest_account_months
  - synth_avg_account_age
  - synth_oldest_mortgage_months
  - synth_seg_app_nom_job_code_top1
  - synth_credit_age_years
  - marital_x_n_children
  - synth_int_inc_x_nchildren
  - synth_oldest_tradeline_v1
  - synth_oldest_card_months
  - synth_n_late_90
  - synth_avg_account_age_months
  - synth_seg_app_nom_marital_status_top2
  - synth_n_card_lines
  - synth_oldest_installment_months
  - median_income_by_job_code
  - synth_n_auto_lines
  - synth_n_late_30
  - synth_n_installment_lines
  - app_nom_branch
  - app_nom_city

Numeric: 19, Categorical: 3


In [11]:
# Load all 22 features + meta
df_rob = pd.read_parquet(ABT_PATH, columns=final_features_full + get_meta_columns())
df_rob["split_new"] = make_calibration_split(df_rob, TRAIN_FOR_MODEL, CALIB_PERIODS)
mask_tr2  = df_rob["split_new"] == "train_for_model"
mask_cal2 = df_rob["split_new"] == "calib"
mask_oot2 = df_rob["split_new"] == "oot"
y_tr2 = df_rob.loc[mask_tr2, "default_flag_12m"].astype(int)

binners_rob = {}
for f in ROB_NUMERIC:
    b = NumericBinner(feature=f, n_initial_bins=20, min_bin_size_frac=0.05, monotonic=True)
    b.fit(df_rob.loc[mask_tr2, f], y_tr2)
    binners_rob[f] = b
for f in ROB_CAT:
    b = CategoricalBinner(feature=f, min_bin_size_frac=0.05)
    b.fit(df_rob.loc[mask_tr2, f], y_tr2)
    binners_rob[f] = b
print("Per-feature IV (robustness):")
for f in final_features_full:
    print(f"  {f:<45}  IV={binners_rob[f].iv:.4f}  bins={len(binners_rob[f].bin_woe)}")

Per-feature IV (robustness):
  app_nom_job_code                               IV=0.3923  bins=3
  synth_age_of_newest                            IV=0.2585  bins=14
  synth_newest_account_months                    IV=0.2008  bins=16
  synth_avg_account_age                          IV=0.1819  bins=15
  synth_oldest_mortgage_months                   IV=0.1790  bins=18
  synth_seg_app_nom_job_code_top1                IV=0.2310  bins=2
  synth_credit_age_years                         IV=0.1708  bins=16
  marital_x_n_children                           IV=0.3937  bins=4
  synth_int_inc_x_nchildren                      IV=0.3299  bins=5
  synth_oldest_tradeline_v1                      IV=0.1406  bins=16
  synth_oldest_card_months                       IV=0.1101  bins=15
  synth_n_late_90                                IV=0.0994  bins=15
  synth_avg_account_age_months                   IV=0.0971  bins=15
  synth_seg_app_nom_marital_status_top2          IV=0.1304  bins=2
  synth_n_card_lines    

In [12]:
df_rob_woe = apply_woe(df_rob, binners_rob)
woe_cols_rob = [f"{f}_woe" for f in final_features_full]
X_tr_r = df_rob_woe.loc[mask_tr2, woe_cols_rob]
y_tr_r = df_rob_woe.loc[mask_tr2, "default_flag_12m"].astype(int)
X_tr_r_const = sm.add_constant(X_tr_r, has_constant="add")
model_rob = sm.Logit(y_tr_r, X_tr_r_const).fit(disp=False, maxiter=200)
print(f"Robustness LR refit: {len(final_features_full)} features")
# Show only significant coefficients
sig = model_rob.pvalues[model_rob.pvalues < 0.05]
print(f"Significant (p<0.05): {len(sig)} of {len(model_rob.pvalues)}")

feature_betas_rob = {f: float(model_rob.params[f"{f}_woe"]) for f in final_features_full}
intercept_rob = float(model_rob.params["const"])

scorecard_rob = build_scorecard(
    feature_betas=feature_betas_rob,
    intercept=intercept_rob,
    binners=binners_rob,
    base_score=BASE_SCORE,
    pdo=PDO,
    base_odds=BASE_ODDS,
)
print(f"Robustness scorecard rows: {len(scorecard_rob)}")

Robustness LR refit: 22 features
Significant (p<0.05): 17 of 23
Robustness scorecard rows: 219


In [13]:
# Score and Platt-calibrate
df_rob_woe_const = sm.add_constant(df_rob_woe[woe_cols_rob], has_constant="add")
df_rob["pd_rob_raw"] = model_rob.predict(df_rob_woe_const)
y_cal2 = df_rob.loc[mask_cal2, "default_flag_12m"].astype(int).to_numpy()
s_cal2 = df_rob.loc[mask_cal2, "pd_rob_raw"].to_numpy()
platt_rob = fit_platt(s_cal2, y_cal2)
df_rob["pd_rob_platt"] = apply_calibrator(df_rob["pd_rob_raw"].to_numpy(), platt_rob, "platt")

y_oot2 = df_rob.loc[mask_oot2, "default_flag_12m"].astype(int).to_numpy()
rob_raw   = metrics_at(y_oot2, df_rob.loc[mask_oot2, "pd_rob_raw"].to_numpy())
rob_platt = metrics_at(y_oot2, df_rob.loc[mask_oot2, "pd_rob_platt"].to_numpy())

print("Robustness scorecard OOT metrics:")
print(pd.DataFrame({"raw": rob_raw, "platt": rob_platt}).round(5).to_string())

# Save robustness artifacts
scorecard_rob.to_csv(ROB_DIR / "scorecard_table.csv", index=False)
with open(ROB_DIR / "scorecard_metrics.json", "w") as f:
    json.dump({
        "n_features": len(final_features_full),
        "features": final_features_full,
        "iv_per_feature": {f: float(binners_rob[f].iv) for f in final_features_full},
        "betas": feature_betas_rob,
        "intercept": intercept_rob,
        "oot_metrics": {"raw": rob_raw, "platt": rob_platt},
    }, f, indent=2, default=str)
print(f"\nSaved: {ROB_DIR / 'scorecard_table.csv'}")
print(f"Saved: {ROB_DIR / 'scorecard_metrics.json'}")
print(f"Robustness wall: {time.time()-T1:.1f}s")
print(f"\nTotal Phase 2B wall: {time.time()-T0:.1f}s")

Robustness scorecard OOT metrics:
               raw    platt
auc        0.81657  0.81657
gini       0.63314  0.63314
ks         0.47444  0.47444
brier      0.00838  0.00823
ece        0.00872  0.00104
mean_pred  0.01719  0.00931
base_rate  0.00847  0.00847

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\robustness_full_f6e\scorecard_table.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\scorecard\robustness_full_f6e\scorecard_metrics.json
Robustness wall: 4.1s

Total Phase 2B wall: 7.9s


## 9. Done

Primary scorecard (7-feature LR no-F6E) and optional robustness scorecard
(22-feature LR full-F6E) both built. See `artifacts/scorecard/` for outputs.

# Section 5 - 02_economic_framework.ipynb

**Section metadata**

- **Purpose**: Phase 3.1B - base economic computation using the locked Lifetime Net Margin formulas on the 235,968 economic-analysis rows.
- **Main artifacts read**: Calibrated PD predictions; `src/economics.py` locked formulas.
- **Main artifacts written**: `artifacts/economic_framework/economics_per_account.parquet` and base-scenario reports.
- **Expensive rerun expected**: Moderate (per-account vectorised formula evaluation).

---


# Phase 3.1B — Full Production Economics Run

**Locked conditions** (per `phase3_formula_lock.md` + Phase 3.1A acceptance):
- Exclude 12-month loans (structural artifact). Population = 24m + 36m only ≈ 235,968 rows.
- Primary PD: LightGBM tuned + Platt-calibrated.
- Robustness PDs: LR full-F6E + Platt, LR no-F6E + Platt, Scorecard no-F6E + Platt.
- Use locked formulas (`src.economics`).
- APR scenarios: 1 tiered (locked tier table) + 4 fixed (12%, 18%, 24%, 30%).
- LGD grid: {0.45, 0.55, 0.65, 0.75, 0.85}.
- Discount grid: {0.00, 0.05}; op_cost grid: {0.00, 0.01, 0.02}.
- Sensitivity total: 5 × 5 × 2 × 3 = 150 cells (primary PD only).
- Base scenario (tiered APR, LGD=0.65, disc=0, op=0): all 4 PD models.
- ASB one-period profit is benchmark only.

In [1]:
import sys, time, json, joblib, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.governance import filter_pd_eligible, get_meta_columns
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
)
from src.economics import (
    amortization_schedule, monthly_hazard_from_pd12, marginal_pd_schedule,
    lifetime_expected_loss, expected_interest_margin, expected_net_profit,
    apr_tier_lookup, apr_tier_lookup_vec,
    asb_one_period_profit_reference, asb_one_period_profit_reference_vec,
    batch_lifetime_economics,
)
from src.evaluation import compute_gini

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
LGB_MODEL    = REPO_ROOT / "artifacts/pd_model/lightgbm_tuned_model.pkl"
WOE_ABT      = REPO_ROOT / "artifacts/scorecard/woe_transformed_abt.parquet"
DIAG_T1      = REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test1_f6e_ablation.json"
FINAL_FEATS  = REPO_ROOT / "artifacts/phase2_rerun_v2/final_feature_set.csv"

OUT_DIR = REPO_ROOT / "artifacts/economic_framework"
OUT_DIR.mkdir(parents=True, exist_ok=True)
T0 = time.time()

## 1. Load economics population (24m + 36m only)

In [2]:
catalog = pd.read_csv(CATALOG_PATH)
pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible features: {len(pd_eligible)}")

# Load only the columns we need for ALL PD models (union of all features)
LR_FULL_F6E_FEATS = pd.read_csv(FINAL_FEATS)["feature"].tolist()  # 22 features
LR_NO_F6E_FEATS = json.load(open(DIAG_T1))["final_features"]  # 7 features
print(f"LR full-F6E features: {len(LR_FULL_F6E_FEATS)}")
print(f"LR no-F6E features: {len(LR_NO_F6E_FEATS)}")

cols_needed = sorted(set(pd_eligible) | set(LR_FULL_F6E_FEATS) | set(LR_NO_F6E_FEATS) | set(get_meta_columns()) | {"n_installments", "loan_amount"})
print(f"Total columns to load: {len(cols_needed)}")

t = time.time()
df = pd.read_parquet(ABT_PATH, columns=cols_needed)
print(f"Loaded ABT: {df.shape} in {time.time()-t:.1f}s")

# Filter to 24m + 36m
df_eco = df[df["n_installments"].isin([24, 36])].copy().reset_index(drop=True)
print(f"\nEconomics population (24m + 36m): {df_eco.shape}")
print(f"  24m: {(df_eco['n_installments']==24).sum():,}")
print(f"  36m: {(df_eco['n_installments']==36).sum():,}")

# Confirm 12m excluded
assert (df_eco['n_installments'] == 12).sum() == 0, "12m loans found in eco pop!"
print("✓ 12-month loans EXCLUDED")

# Categorical conversion for LightGBM
CAT_FEATURES = ["app_nom_branch","app_nom_gender","app_nom_job_code",
                "app_nom_marital_status","app_nom_city","app_nom_home_status"]
cat_in_pool = [c for c in CAT_FEATURES if c in pd_eligible]
for c in cat_in_pool:
    df_eco[c] = df_eco[c].astype("category")

# Set up temporal split for refitting LR variants
TRAIN_FOR_MODEL = list(range(202509, 202611))
CALIB_PERIODS   = [202611, 202612]
df_eco["split_new"] = make_calibration_split(df_eco, TRAIN_FOR_MODEL, CALIB_PERIODS)
print("\nNew split counts (eco pop):")
print(df_eco["split_new"].value_counts().to_string())

PD-eligible features: 435
LR full-F6E features: 22
LR no-F6E features: 7
Total columns to load: 441


Loaded ABT: (534314, 441) in 0.8s



Economics population (24m + 36m): (235968, 441)
  24m: 135,115
  36m: 100,853
✓ 12-month loans EXCLUDED

New split counts (eco pop):


split_new
train_for_model    150476
oot                 64027
calib               21465


## 2. Score with 4 PD models

Primary: LightGBM tuned + Platt. Three robustness variants refit on
202509-202610, Platt on 202611-202612.

In [3]:
def score_lightgbm(df_in):
    """Score with the saved tuned LightGBM model + Platt."""
    pkg = joblib.load(LGB_MODEL)
    booster = pkg["booster"]
    feat_list = pkg["feature_list"]
    platt = pkg["platt"]
    # Score
    s_raw = booster.predict(df_in[feat_list], num_iteration=pkg["best_iteration"])
    s_platt = apply_calibrator(s_raw, platt, "platt")
    return s_raw, s_platt

print("Scoring with LightGBM tuned...")
t = time.time()
df_eco["pd_lgb_raw"], df_eco["pd_lgb_platt"] = score_lightgbm(df_eco)
print(f"  done in {time.time()-t:.1f}s")
print(f"  pd_lgb_platt mean: {df_eco['pd_lgb_platt'].mean():.4%}")

Scoring with LightGBM tuned...


  done in 0.7s
  pd_lgb_platt mean: 0.9486%


In [4]:
def refit_lr_with_platt(df_in, features, label):
    """Refit LR on train_for_model + Platt on calib; score full df_in."""
    mask_tr  = df_in["split_new"] == "train_for_model"
    mask_cal = df_in["split_new"] == "calib"
    medians = df_in.loc[mask_tr, features].median(numeric_only=True)
    X_tr = df_in.loc[mask_tr, features].fillna(medians)
    y_tr = df_in.loc[mask_tr, "default_flag_12m"].astype(int)
    keep = ~X_tr.isna().any(axis=1)
    X_tr_const = sm.add_constant(X_tr.loc[keep], has_constant="add")
    model = sm.Logit(y_tr.loc[keep], X_tr_const).fit(disp=False, maxiter=200)
    # Score full population
    X_full = sm.add_constant(df_in[features].fillna(medians), has_constant="add")
    s_raw = model.predict(X_full).to_numpy()
    # Platt on calib
    s_cal = s_raw[mask_cal.to_numpy()]
    y_cal = df_in.loc[mask_cal, "default_flag_12m"].astype(int).to_numpy()
    platt = fit_platt(s_cal, y_cal)
    s_platt = apply_calibrator(s_raw, platt, "platt")
    print(f"  {label}: refit OK, OOT mean_pred={s_platt[df_in['split_new']=='oot'].mean():.4%}")
    return s_raw, s_platt

print("Refitting LR full-F6E + Platt...")
t = time.time()
df_eco["pd_lr_full_raw"], df_eco["pd_lr_full_platt"] = refit_lr_with_platt(
    df_eco, LR_FULL_F6E_FEATS, "LR_full_F6E")
print(f"  wall: {time.time()-t:.1f}s")

print("\nRefitting LR no-F6E + Platt...")
t = time.time()
df_eco["pd_lr_nof6e_raw"], df_eco["pd_lr_nof6e_platt"] = refit_lr_with_platt(
    df_eco, LR_NO_F6E_FEATS, "LR_no_F6E")
print(f"  wall: {time.time()-t:.1f}s")

Refitting LR full-F6E + Platt...


  LR_full_F6E: refit OK, OOT mean_pred=2.0702%
  wall: 0.7s

Refitting LR no-F6E + Platt...


  LR_no_F6E: refit OK, OOT mean_pred=2.0755%
  wall: 0.3s


In [5]:
# Scorecard PD: load from woe_transformed_abt.parquet (Phase 2B output)
print("Joining scorecard PD from Phase 2B...")
sc_df = pd.read_parquet(WOE_ABT, columns=["aid", "pd_woe_raw", "pd_woe_platt"])
sc_df = sc_df.rename(columns={"pd_woe_raw": "pd_sc_raw", "pd_woe_platt": "pd_sc_platt"})
df_eco = df_eco.merge(sc_df, on="aid", how="left")
print(f"  scorecard PD null after merge: {int(df_eco['pd_sc_platt'].isna().sum()):,}")
print(f"  pd_sc_platt mean: {df_eco['pd_sc_platt'].mean():.4%}")

Joining scorecard PD from Phase 2B...


  scorecard PD null after merge: 0
  pd_sc_platt mean: 0.9222%


In [6]:
# Multi-model PD comparison (sanity check)
PD_COLS = ["pd_lgb_platt", "pd_lr_full_platt", "pd_lr_nof6e_platt", "pd_sc_platt"]
oot = df_eco[df_eco["split_new"] == "oot"]
print(f"OOT rows: {len(oot):,}, OOT base rate: {(oot['default_flag_12m']==1).mean():.4%}")
print()
print(f"{'PD model':<22}  {'mean_pred':>10}  {'OOT_AUC':>8}  {'OOT_Gini':>8}")
for c in PD_COLS:
    if oot[c].notna().all():
        gini = compute_gini(oot["default_flag_12m"], oot[c])
        print(f"{c:<22}  {oot[c].mean():>10.4%}  {(gini+1)/2:>8.4f}  {gini:>8.4f}")

OOT rows: 64,027, OOT base rate: 1.9164%

PD model                 mean_pred   OOT_AUC  OOT_Gini
pd_lgb_platt               0.9386%    0.9022    0.8044
pd_lr_full_platt           2.0702%    0.8640    0.7281


pd_lr_nof6e_platt          2.0755%    0.8400    0.6800
pd_sc_platt                0.9179%    0.8030    0.6061


## 3. Base economics (primary LightGBM PD, tiered APR, LGD=0.65, disc=0, op=0)

In [7]:
LGD_BASE = 0.65
DISC_BASE = 0.0
OP_BASE = 0.0
PRIMARY_PD = "pd_lgb_platt"

# Per-row APR via tiered lookup (locked Section 10)
df_eco["apr_tiered"] = apr_tier_lookup_vec(df_eco[PRIMARY_PD].fillna(0).to_numpy())

# Base economics (primary PD, tiered APR, LGD=0.65)
print(f"Computing base economics (primary={PRIMARY_PD}, tiered APR, LGD={LGD_BASE})...")
t = time.time()
base = batch_lifetime_economics(
    pd_12m=df_eco[PRIMARY_PD].fillna(0).to_numpy(),
    loan_amount=df_eco["loan_amount"].to_numpy(),
    n_installments=df_eco["n_installments"].to_numpy(),
    apr=df_eco["apr_tiered"].to_numpy(),
    lgd=LGD_BASE, op_cost_annual=OP_BASE, discount_annual=DISC_BASE,
)
df_eco["LT_EL_base"] = base["LT_EL"]
df_eco["LT_margin_base"] = base["LT_margin"]
df_eco["Expected_Profit_base"] = base["Expected_Profit"]
print(f"  wall: {time.time()-t:.1f}s")
print()
print("Base scenario distribution (full eco pop):")
for col in ["LT_EL_base", "LT_margin_base", "Expected_Profit_base"]:
    s = df_eco[col]
    print(f"  {col:<22}  mean={s.mean():>10.2f}  median={s.median():>10.2f}  "
          f"std={s.std():>10.2f}  p1={s.quantile(0.01):>10.2f}  "
          f"p99={s.quantile(0.99):>10.2f}")

# Profit > 0 share
print(f"\nProfit > 0 share: {(df_eco['Expected_Profit_base'] > 0).mean():.4%}")

Computing base economics (primary=pd_lgb_platt, tiered APR, LGD=0.65)...


  wall: 0.2s

Base scenario distribution (full eco pop):
  LT_EL_base              mean=     62.95  median=     13.00  std=    164.85  p1=      0.18  p99=    817.94
  LT_margin_base          mean=   1556.02  median=   1179.79  std=   1244.45  p1=    295.72  p99=   6366.36
  Expected_Profit_base    mean=   1493.07  median=   1150.71  std=   1150.08  p1=    293.22  p99=   5831.05

Profit > 0 share: 100.0000%


In [8]:
# Multi-PD comparison at base scenario (tiered APR, LGD=0.65, disc=0, op=0)
print("\n=== Base-scenario multi-PD comparison ===")
multi_pd_results = {}
for pd_col in PD_COLS:
    pd_arr = df_eco[pd_col].fillna(0).to_numpy()
    apr_arr = apr_tier_lookup_vec(pd_arr)
    out = batch_lifetime_economics(
        pd_12m=pd_arr,
        loan_amount=df_eco["loan_amount"].to_numpy(),
        n_installments=df_eco["n_installments"].to_numpy(),
        apr=apr_arr, lgd=LGD_BASE,
    )
    multi_pd_results[pd_col] = {
        "mean_pd": float(np.nanmean(pd_arr)),
        "mean_LT_EL": float(out["LT_EL"].mean()),
        "mean_LT_margin": float(out["LT_margin"].mean()),
        "mean_Expected_Profit": float(out["Expected_Profit"].mean()),
        "total_Expected_Profit": float(out["Expected_Profit"].sum()),
        "share_profit_gt_0": float((out["Expected_Profit"] > 0).mean()),
    }
print(pd.DataFrame(multi_pd_results).T.round(4).to_string())


=== Base-scenario multi-PD comparison ===


                   mean_pd  mean_LT_EL  mean_LT_margin  mean_Expected_Profit  total_Expected_Profit  share_profit_gt_0
pd_lgb_platt        0.0095     62.9527       1556.0200             1493.0673           3.523161e+08             1.0000
pd_lr_full_platt    0.0208    134.5016       1940.0882             1805.5865           4.260606e+08             0.9970
pd_lr_nof6e_platt   0.0208    134.3763       1993.4515             1859.0752           4.386823e+08             0.9967
pd_sc_platt         0.0092     60.7759       1643.1129             1582.3370           3.733809e+08             1.0000


## 4. Sensitivity grid (150 cells, primary PD only)

In [9]:
APR_SCENARIOS = {
    "tiered": None,  # use apr_tier_lookup_vec
    "fixed_12": 0.12,
    "fixed_18": 0.18,
    "fixed_24": 0.24,
    "fixed_30": 0.30,
}
LGD_GRID = [0.45, 0.55, 0.65, 0.75, 0.85]
DISC_GRID = [0.00, 0.05]
OP_GRID = [0.00, 0.01, 0.02]

print(f"Grid size: {len(APR_SCENARIOS)} × {len(LGD_GRID)} × {len(DISC_GRID)} × {len(OP_GRID)} = "
      f"{len(APR_SCENARIOS) * len(LGD_GRID) * len(DISC_GRID) * len(OP_GRID)}")

pd_arr_primary = df_eco[PRIMARY_PD].fillna(0).to_numpy()
L_arr = df_eco["loan_amount"].to_numpy()
n_arr = df_eco["n_installments"].to_numpy()

grid_rows = []
t = time.time()
for apr_label, apr_val in APR_SCENARIOS.items():
    if apr_val is None:
        apr_arr = apr_tier_lookup_vec(pd_arr_primary)
    else:
        apr_arr = np.full_like(L_arr, apr_val, dtype=np.float64)
    for lgd in LGD_GRID:
        for disc in DISC_GRID:
            for op in OP_GRID:
                out = batch_lifetime_economics(
                    pd_12m=pd_arr_primary, loan_amount=L_arr,
                    n_installments=n_arr, apr=apr_arr, lgd=lgd,
                    op_cost_annual=op, discount_annual=disc,
                )
                grid_rows.append({
                    "apr_scenario": apr_label,
                    "lgd": lgd,
                    "discount_annual": disc,
                    "op_cost_annual": op,
                    "mean_LT_EL": float(out["LT_EL"].mean()),
                    "mean_LT_margin": float(out["LT_margin"].mean()),
                    "mean_Expected_Profit": float(out["Expected_Profit"].mean()),
                    "total_Expected_Profit": float(out["Expected_Profit"].sum()),
                    "share_profit_gt_0": float((out["Expected_Profit"] > 0).mean()),
                    "n_rows": len(out["LT_EL"]),
                })
sensitivity = pd.DataFrame(grid_rows)
print(f"Grid wall: {time.time()-t:.1f}s; rows: {len(sensitivity)}")
print()
print("Top 10 most-profitable cells (mean Expected_Profit):")
print(sensitivity.nlargest(10, "mean_Expected_Profit").round(2).to_string(index=False))
print()
print("Bottom 10 (mean Expected_Profit):")
print(sensitivity.nsmallest(10, "mean_Expected_Profit").round(2).to_string(index=False))

Grid size: 5 × 5 × 2 × 3 = 150


Grid wall: 1544.6s; rows: 150

Top 10 most-profitable cells (mean Expected_Profit):
apr_scenario  lgd  discount_annual  op_cost_annual  mean_LT_EL  mean_LT_margin  mean_Expected_Profit  total_Expected_Profit  share_profit_gt_0  n_rows
    fixed_30 0.45             0.00            0.00       44.43         3156.39               3111.96           734323054.77                1.0  235968
    fixed_30 0.55             0.00            0.00       54.30         3156.39               3102.09           731993413.29                1.0  235968
    fixed_30 0.65             0.00            0.00       64.17         3156.39               3092.21           729663771.81                1.0  235968
    fixed_30 0.75             0.00            0.00       74.05         3156.39               3082.34           727334130.32                1.0  235968
    fixed_30 0.85             0.00            0.00       83.92         3156.39               3072.47           725004488.84                1.0  235968
    fixed_

## 5. Cut-off analysis

For each PD model and APR scenario, find the profit-maximizing PD threshold
and compare with Youden's J cutoff (TPR-FPR) on OOT.

In [10]:
def youden_threshold(y_true, y_score):
    """Find threshold maximizing TPR - FPR."""
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y_true, y_score)
    j = tpr - fpr
    return float(thr[np.argmax(j)])

# Limit thresholds: percentiles 50, 60, 70, 75, 80, 85, 90, 95, 99
thresholds_pct = [50, 60, 70, 75, 80, 85, 90, 95, 99, 100]

cutoff_rows = []
optimal_cutoffs = {}
for pd_col in PD_COLS:
    pd_arr = df_eco[pd_col].fillna(1.0).to_numpy()  # NaN → reject (PD=1)
    L = df_eco["loan_amount"].to_numpy()
    n = df_eco["n_installments"].to_numpy()
    for apr_label, apr_val in APR_SCENARIOS.items():
        apr_arr = apr_tier_lookup_vec(pd_arr) if apr_val is None else np.full_like(L, apr_val, dtype=np.float64)
        out = batch_lifetime_economics(
            pd_12m=pd_arr, loan_amount=L, n_installments=n, apr=apr_arr, lgd=LGD_BASE,
        )
        profit = out["Expected_Profit"]
        # Sort by PD ascending (lowest PD first = best)
        order = np.argsort(pd_arr)
        pd_sorted = pd_arr[order]
        profit_sorted = profit[order]
        cum_profit = np.cumsum(profit_sorted)
        for tp in thresholds_pct:
            k = int(len(pd_arr) * tp / 100)
            if k <= 0: continue
            cutoff_pd = pd_sorted[k - 1]
            accepted = k
            total_p = cum_profit[k - 1]
            cutoff_rows.append({
                "pd_model": pd_col,
                "apr_scenario": apr_label,
                "approve_pct": tp,
                "n_accepted": int(accepted),
                "cutoff_pd": float(cutoff_pd),
                "total_Expected_Profit": float(total_p),
                "mean_Expected_Profit_accepted": float(total_p / max(accepted, 1)),
            })
        # Optimal threshold = max cumulative profit
        k_star = int(np.argmax(cum_profit)) + 1
        cutoff_pd_star = pd_sorted[k_star - 1]
        # Youden (only on OOT)
        oot_mask = (df_eco["split_new"] == "oot").to_numpy()
        if oot_mask.any():
            try:
                youden = youden_threshold(
                    df_eco.loc[oot_mask, "default_flag_12m"].to_numpy(),
                    pd_arr[oot_mask],
                )
            except Exception:
                youden = float("nan")
        else:
            youden = float("nan")
        optimal_cutoffs[f"{pd_col}__{apr_label}"] = {
            "k_star": int(k_star),
            "approve_pct_star": round(100 * k_star / len(pd_arr), 2),
            "cutoff_pd_star": float(cutoff_pd_star),
            "max_total_profit": float(cum_profit[k_star - 1]),
            "youden_threshold_oot": float(youden),
            "youden_approve_pct": round(100 * (pd_arr <= youden).mean(), 2) if not np.isnan(youden) else float("nan"),
        }

cutoff_df = pd.DataFrame(cutoff_rows)
print(f"Cut-off result rows: {len(cutoff_df)}")
print()
print("Sample cut-off table (LightGBM tuned, tiered APR):")
sample = cutoff_df[(cutoff_df["pd_model"]=="pd_lgb_platt") & (cutoff_df["apr_scenario"]=="tiered")]
print(sample.round(2).to_string(index=False))
print()
print("Optimal cut-offs across all (PD model × APR scenario):")
for k, v in list(optimal_cutoffs.items())[:6]:
    print(f"  {k:<35}  k*={v['approve_pct_star']:>6.2f}%  PD*={v['cutoff_pd_star']:.4f}  "
          f"profit={v['max_total_profit']:>14,.2f}  Youden_thr={v['youden_threshold_oot']:.4f}")

Cut-off result rows: 200

Sample cut-off table (LightGBM tuned, tiered APR):
    pd_model apr_scenario  approve_pct  n_accepted  cutoff_pd  total_Expected_Profit  mean_Expected_Profit_accepted
pd_lgb_platt       tiered           50      117984       0.00           139041479.17                        1178.48
pd_lgb_platt       tiered           60      141580       0.00           166268682.87                        1174.38
pd_lgb_platt       tiered           70      165177       0.01           199535994.58                        1208.01
pd_lgb_platt       tiered           75      176976       0.01           220654245.20                        1246.80
pd_lgb_platt       tiered           80      188774       0.01           243140969.01                        1288.00
pd_lgb_platt       tiered           85      200572       0.02           269019371.29                        1341.26
pd_lgb_platt       tiered           90      212371       0.02           295894395.99                        139

## 6. Tenor-stratified analysis (24m vs 36m vs combined)

In [11]:
tenor_rows = []
for tenor_set, label in [([24], "24m"), ([36], "36m"), ([24, 36], "24m+36m")]:
    sub = df_eco[df_eco["n_installments"].isin(tenor_set)]
    if len(sub) == 0: continue
    pd_arr = sub[PRIMARY_PD].fillna(0).to_numpy()
    apr_arr = apr_tier_lookup_vec(pd_arr)
    out = batch_lifetime_economics(
        pd_12m=pd_arr, loan_amount=sub["loan_amount"].to_numpy(),
        n_installments=sub["n_installments"].to_numpy(), apr=apr_arr,
        lgd=LGD_BASE,
    )
    tenor_rows.append({
        "tenor": label,
        "n_rows": len(sub),
        "mean_pd": float(np.nanmean(pd_arr)),
        "mean_loan_amount": float(sub["loan_amount"].mean()),
        "mean_LT_EL": float(out["LT_EL"].mean()),
        "mean_LT_margin": float(out["LT_margin"].mean()),
        "mean_Expected_Profit": float(out["Expected_Profit"].mean()),
        "total_Expected_Profit": float(out["Expected_Profit"].sum()),
        "share_profit_gt_0": float((out["Expected_Profit"] > 0).mean()),
    })
tenor_df = pd.DataFrame(tenor_rows)
print("Tenor-stratified (base scenario, primary PD):")
print(tenor_df.round(2).to_string(index=False))

Tenor-stratified (base scenario, primary PD):
  tenor  n_rows  mean_pd  mean_loan_amount  mean_LT_EL  mean_LT_margin  mean_Expected_Profit  total_Expected_Profit  share_profit_gt_0
    24m  135115     0.01           5946.51       40.89         1006.13                965.25           130419120.25                1.0
    36m  100853     0.01           8944.29       92.51         2292.72               2200.20           221896988.57                1.0
24m+36m  235968     0.01           7227.77       62.95         1556.02               1493.07           352316108.82                1.0


## 7. ASB benchmark vs tenor-aware Expected_Profit

In [12]:
df_eco["ASB_profit"] = asb_one_period_profit_reference_vec(
    pd_12m=df_eco[PRIMARY_PD].fillna(0).to_numpy(),
    loan_amount=df_eco["loan_amount"].to_numpy(),
    apr=df_eco["apr_tiered"].to_numpy(),
    lgd=LGD_BASE,
)
df_eco["ASB_minus_LT"] = df_eco["ASB_profit"] - df_eco["Expected_Profit_base"]
df_eco["ASB_pct_bias"] = (df_eco["ASB_profit"] - df_eco["Expected_Profit_base"]) / df_eco["Expected_Profit_base"].abs()

asb_rows = []
for tenor_set, label in [([24], "24m"), ([36], "36m"), ([24, 36], "24m+36m")]:
    sub = df_eco[df_eco["n_installments"].isin(tenor_set)]
    asb_rows.append({
        "tenor": label,
        "n_rows": len(sub),
        "mean_LT_profit": float(sub["Expected_Profit_base"].mean()),
        "mean_ASB_profit": float(sub["ASB_profit"].mean()),
        "mean_ASB_minus_LT": float(sub["ASB_minus_LT"].mean()),
        "median_ASB_pct_bias": float(sub["ASB_pct_bias"].median()),
        "ASB_total_profit": float(sub["ASB_profit"].sum()),
        "LT_total_profit": float(sub["Expected_Profit_base"].sum()),
        "ASB_LT_total_ratio": float(sub["ASB_profit"].sum() / sub["Expected_Profit_base"].sum()),
    })
asb_df = pd.DataFrame(asb_rows)
print("ASB benchmark vs Lifetime Expected_Profit (base scenario):")
print(asb_df.round(4).to_string(index=False))

ASB benchmark vs Lifetime Expected_Profit (base scenario):
  tenor  n_rows  mean_LT_profit  mean_ASB_profit  mean_ASB_minus_LT  median_ASB_pct_bias  ASB_total_profit  LT_total_profit  ASB_LT_total_ratio
    24m  135115        965.2453         873.9911           -91.2542              -0.0761      1.180893e+08     1.304191e+08              0.9055
    36m  100853       2200.2022        1317.7067          -882.4954              -0.3869      1.328947e+08     2.218970e+08              0.5989
24m+36m  235968       1493.0673        1063.6357          -429.4316              -0.1086      2.509840e+08     3.523161e+08              0.7124


## 8. Save artifacts

In [13]:
# economics_per_account.parquet (slim, primary economics)
slim_cols = ["aid", "n_installments", "loan_amount", "default_flag_12m", "split", "split_new",
             PRIMARY_PD, "pd_lr_full_platt", "pd_lr_nof6e_platt", "pd_sc_platt",
             "apr_tiered", "LT_EL_base", "LT_margin_base", "Expected_Profit_base", "ASB_profit"]
slim = df_eco[slim_cols].copy()
slim.to_parquet(OUT_DIR / "economics_per_account.parquet", index=False)
print(f"Saved: {OUT_DIR / 'economics_per_account.parquet'}  ({len(slim):,} rows × {len(slim.columns)} cols)")

# sensitivity_grid.parquet
sensitivity.to_parquet(OUT_DIR / "sensitivity_grid.parquet", index=False)
print(f"Saved: {OUT_DIR / 'sensitivity_grid.parquet'}  ({len(sensitivity)} cells)")

# cutoff_results.csv
cutoff_df.to_csv(OUT_DIR / "cutoff_results.csv", index=False)
print(f"Saved: {OUT_DIR / 'cutoff_results.csv'}  ({len(cutoff_df)} rows)")

# optimal_cutoffs.json
with open(OUT_DIR / "optimal_cutoffs.json", "w") as f:
    json.dump(optimal_cutoffs, f, indent=2)
print(f"Saved: {OUT_DIR / 'optimal_cutoffs.json'}")

# asb_comparison.csv
asb_df.to_csv(OUT_DIR / "asb_comparison.csv", index=False)
print(f"Saved: {OUT_DIR / 'asb_comparison.csv'}")

# tenor_economics.csv
tenor_df.to_csv(OUT_DIR / "tenor_economics.csv", index=False)
print(f"Saved: {OUT_DIR / 'tenor_economics.csv'}")

# Multi-model results
with open(OUT_DIR / "multi_pd_base_scenario.json", "w") as f:
    json.dump(multi_pd_results, f, indent=2)
print(f"Saved: {OUT_DIR / 'multi_pd_base_scenario.json'}")

print(f"\nTotal Phase 3.1B wall: {time.time()-T0:.1f}s")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\economics_per_account.parquet  (235,968 rows × 15 cols)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\sensitivity_grid.parquet  (150 cells)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\cutoff_results.csv  (200 rows)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\optimal_cutoffs.json
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\asb_comparison.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\tenor_economics.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\multi_pd_base_scenario.json

Total Phase 3.1B wall: 1557.8s


# Section 6 - 03_economic_stress_test.ipynb

**Section metadata**

- **Purpose**: Phase 3.2 - 576-cell economic stress grid crossing five dimensions (PD multiplier, APR strategy, acquisition cost, LGD, cost of funds).
- **Main artifacts read**: `economics_per_account.parquet` plus the stress configuration.
- **Main artifacts written**: `artifacts/economic_framework/phase3_2_stress_grid.parquet` and driver hierarchy.
- **Expensive rerun expected**: Moderate (576 cells x per-account evaluation).

---


# Phase 3.2 — Economic Realism Stress Test

**Goal**: identify whether profit-optimal cutoffs remain approve-all (Phase 3.1B
finding) or become interior under realistic cost / APR / LGD / PD stress.

**Stress dimensions** (locked by user prompt):
- PD multipliers: {1, 2, 3, 5}
- cost_of_funds_annual: {0.00, 0.03, 0.06}
- acquisition_cost (per approved loan): {0, 250, 500}
- LGD: {0.55, 0.65, 0.75, 0.85}
- APR strategy: {tiered_uncapped, tiered_cap_24, flat_18, flat_24}

Combined grid = 4 × 3 × 3 × 4 × 4 = **576 cells** (primary PD only).

**Per cell**: mean / total / share>0 of Expected_Profit, profit-optimal cutoff,
Youden cutoff (computed once per PD model on OOT), gap.

**Classification**:
- A. approve-all: profit-optimal k* >= 99%
- B. interior-optimum: 50% <= k* < 99%
- C. reject-most: k* < 50%

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.economics import (
    batch_lifetime_economics,
    apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)
from src.evaluation import compute_gini

# Load Phase 3.1B per-account economics (already filtered to 24m+36m,
# already scored with all 4 PD models)
ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
eco = pd.read_parquet(ECO_PATH)
print(f"Loaded eco data: {eco.shape}")
print(f"  splits: {eco['split_new'].value_counts().to_dict()}")

PRIMARY_PD = "pd_lgb_platt"
PD_COLS = ["pd_lgb_platt", "pd_lr_full_platt", "pd_lr_nof6e_platt", "pd_sc_platt"]

T0 = time.time()

Loaded eco data: (235968, 15)
  splits: {'train_for_model': 150476, 'oot': 64027, 'calib': 21465}


## 1. APR strategy helpers

In [2]:
def apr_array(pd_arr, strategy):
    """Return per-row APR array given a strategy name."""
    if strategy == "tiered_uncapped":
        return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy == "tiered_cap_18":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.18)
    if strategy.startswith("flat_"):
        v = int(strategy.split("_")[1]) / 100.0
        return np.full_like(pd_arr, v, dtype=np.float64)
    raise ValueError(f"unknown strategy {strategy}")

# Sanity
test = np.array([0.001, 0.01, 0.04, 0.10])
for s in ["tiered_uncapped","tiered_cap_24","tiered_cap_18","flat_18","flat_24"]:
    print(f"  {s:<18}: {apr_array(test, s)}")

  tiered_uncapped   : [0.12 0.22 0.26 0.3 ]
  tiered_cap_24     : [0.12 0.22 0.24 0.24]
  tiered_cap_18     : [0.12 0.18 0.18 0.18]
  flat_18           : [0.18 0.18 0.18 0.18]
  flat_24           : [0.24 0.24 0.24 0.24]


## 2. Youden cutoffs (per PD model, computed once on OOT)

In [3]:
from sklearn.metrics import roc_curve

oot = eco[eco["split_new"] == "oot"].copy()
print(f"OOT rows: {len(oot):,}")
youden_per_pd = {}
for pd_col in PD_COLS:
    if oot[pd_col].notna().all():
        fpr, tpr, thr = roc_curve(oot["default_flag_12m"], oot[pd_col])
        j = tpr - fpr
        thr_star = float(thr[np.argmax(j)])
        oot_approve_pct = float((oot[pd_col] <= thr_star).mean() * 100)
        # Apply same threshold on full eco pop
        full_approve_pct = float((eco[pd_col].fillna(1.0) <= thr_star).mean() * 100)
        youden_per_pd[pd_col] = {
            "threshold": thr_star,
            "oot_approve_pct": oot_approve_pct,
            "full_approve_pct": full_approve_pct,
        }
        print(f"  {pd_col:<22}  Youden_thr={thr_star:.4f}  OOT_acc={oot_approve_pct:.2f}%  full_acc={full_approve_pct:.2f}%")

OOT rows: 64,027
  pd_lgb_platt            Youden_thr=0.0106  OOT_acc=79.59%  full_acc=79.41%
  pd_lr_full_platt        Youden_thr=0.0213  OOT_acc=77.04%  full_acc=76.72%
  pd_lr_nof6e_platt       Youden_thr=0.0226  OOT_acc=80.62%  full_acc=80.59%
  pd_sc_platt             Youden_thr=0.0185  OOT_acc=87.92%  full_acc=87.90%


## 3. Combined stress grid (576 cells)

In [4]:
PD_MULT_GRID  = [1.0, 2.0, 3.0, 5.0]
COF_GRID      = [0.00, 0.03, 0.06]
ACQ_GRID      = [0, 250, 500]
LGD_GRID      = [0.55, 0.65, 0.75, 0.85]
APR_STRATS    = ["tiered_uncapped", "tiered_cap_24", "flat_18", "flat_24"]
DISC_BASE     = 0.0
OP_COST_BASE  = 0.0  # holding op_cost separate; the user's spec says discount/op grids
                     # were Phase 3.1B; this stage focuses on COF/acq/PD-mult/LGD/APR

print(f"Grid size: {len(PD_MULT_GRID)} × {len(COF_GRID)} × {len(ACQ_GRID)} × {len(LGD_GRID)} × {len(APR_STRATS)} = {len(PD_MULT_GRID)*len(COF_GRID)*len(ACQ_GRID)*len(LGD_GRID)*len(APR_STRATS)}")

L = eco["loan_amount"].to_numpy()
n = eco["n_installments"].to_numpy()
pd_base = eco[PRIMARY_PD].fillna(0.0).to_numpy()

results = []
t = time.time()
for mult in PD_MULT_GRID:
    pd_stressed = np.clip(pd_base * mult, 0.0, 0.999)
    for strat in APR_STRATS:
        apr_arr = apr_array(pd_stressed, strat)  # APR strategy uses STRESSED PD for tier
        for cof in COF_GRID:
            for acq in ACQ_GRID:
                for lgd in LGD_GRID:
                    out = batch_lifetime_economics(
                        pd_12m=pd_stressed,
                        loan_amount=L,
                        n_installments=n,
                        apr=apr_arr,
                        lgd=lgd,
                        op_cost_annual=OP_COST_BASE,
                        discount_annual=DISC_BASE,
                        cost_of_funds_annual=cof,
                        acquisition_cost=float(acq),
                    )
                    profit = out["Expected_Profit"]
                    # Cut-off: sort by stressed PD ascending, find max cumulative profit
                    order = np.argsort(pd_stressed)
                    profit_sorted = profit[order]
                    cum_profit = np.cumsum(profit_sorted)
                    if cum_profit.max() <= 0:
                        # No positive cumulative profit anywhere -> reject all
                        k_star = 0
                        cutoff_pd_star = 0.0
                        max_profit = 0.0
                    else:
                        k_star = int(np.argmax(cum_profit)) + 1
                        cutoff_pd_star = float(pd_stressed[order[k_star - 1]])
                        max_profit = float(cum_profit[k_star - 1])
                    approve_pct_star = 100.0 * k_star / len(profit)

                    youden_full_pct = youden_per_pd[PRIMARY_PD]["full_approve_pct"]
                    cutoff_gap = approve_pct_star - youden_full_pct

                    if approve_pct_star >= 99.0:
                        category = "approve_all"
                    elif approve_pct_star >= 50.0:
                        category = "interior"
                    else:
                        category = "reject_most"

                    results.append({
                        "pd_multiplier": mult,
                        "apr_strategy": strat,
                        "cost_of_funds_annual": cof,
                        "acquisition_cost": acq,
                        "lgd": lgd,
                        "mean_pd_stressed": float(pd_stressed.mean()),
                        "mean_LT_EL": float(out["LT_EL"].mean()),
                        "mean_LT_margin": float(out["LT_margin"].mean()),
                        "mean_Expected_Profit": float(profit.mean()),
                        "total_Expected_Profit": float(profit.sum()),
                        "share_profit_gt_0": float((profit > 0).mean()),
                        "k_star_approve_pct": approve_pct_star,
                        "cutoff_pd_star": cutoff_pd_star,
                        "max_total_profit_at_k_star": max_profit,
                        "youden_approve_pct_full": youden_full_pct,
                        "cutoff_gap_profit_minus_youden": cutoff_gap,
                        "category": category,
                    })
stress = pd.DataFrame(results)
print(f"\nGrid wall: {time.time()-t:.1f}s")
print(f"Total cells: {len(stress)}")

Grid size: 4 × 3 × 3 × 4 × 4 = 576



Grid wall: 135.6s
Total cells: 576


## 4. Classification distribution

In [5]:
cat_counts = stress["category"].value_counts()
print("Scenario classification:")
for c in ["approve_all", "interior", "reject_most"]:
    n = int(cat_counts.get(c, 0))
    pct = 100.0 * n / len(stress)
    print(f"  {c:<14}: {n:>4} cells ({pct:.1f}%)")

print("\nClassification x APR strategy:")
print(stress.pivot_table(index="apr_strategy", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x PD multiplier:")
print(stress.pivot_table(index="pd_multiplier", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x cost_of_funds:")
print(stress.pivot_table(index="cost_of_funds_annual", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x acquisition_cost:")
print(stress.pivot_table(index="acquisition_cost", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

print("\nClassification x lgd:")
print(stress.pivot_table(index="lgd", columns="category",
                          values="mean_Expected_Profit", aggfunc="count",
                          fill_value=0).to_string())

Scenario classification:
  approve_all   :  220 cells (38.2%)
  interior      :  356 cells (61.8%)
  reject_most   :    0 cells (0.0%)

Classification x APR strategy:
category         approve_all  interior
apr_strategy                          
flat_18                   34       110
flat_24                   55        89
tiered_cap_24             55        89
tiered_uncapped           76        68

Classification x PD multiplier:
category       approve_all  interior
pd_multiplier                       
1.0                    138         6
2.0                     72        72
3.0                     10       134
5.0                      0       144

Classification x cost_of_funds:
category              approve_all  interior
cost_of_funds_annual                       
0.00                           87       105
0.03                           74       118
0.06                           59       133

Classification x acquisition_cost:
category          approve_all  interior
acquisition_cos

## 5. Drivers — sensitivity of k_star to each parameter

In [6]:
# How much does k_star vary when each parameter moves, holding others at central values?
# Central: mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24
def vary(param, values, fix=None):
    fix = fix or {}
    base_filter = (
        (stress["pd_multiplier"]    == fix.get("pd_multiplier",   2.0)) &
        (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
        (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
        (stress["lgd"]              == fix.get("lgd",             0.65)) &
        (stress["apr_strategy"]     == fix.get("apr_strategy",    "tiered_cap_24"))
    )
    # Replace param's filter with "any of values"
    if param == "pd_multiplier":
        slc = stress[
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["pd_multiplier"].isin(values))
        ]
    elif param == "cost_of_funds_annual":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["cost_of_funds_annual"].isin(values))
        ]
    elif param == "acquisition_cost":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["acquisition_cost"].isin(values))
        ]
    elif param == "lgd":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["apr_strategy"] == fix.get("apr_strategy", "tiered_cap_24")) &
            (stress["lgd"].isin(values))
        ]
    elif param == "apr_strategy":
        slc = stress[
            (stress["pd_multiplier"] == fix.get("pd_multiplier", 2.0)) &
            (stress["cost_of_funds_annual"] == fix.get("cost_of_funds_annual", 0.03)) &
            (stress["acquisition_cost"] == fix.get("acquisition_cost", 250)) &
            (stress["lgd"] == fix.get("lgd", 0.65)) &
            (stress["apr_strategy"].isin(values))
        ]
    return slc[[param, "k_star_approve_pct", "mean_Expected_Profit", "category"]].sort_values(param)

print("Holding other params at center (mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24):")
print()
for p, vals in [
    ("pd_multiplier",         PD_MULT_GRID),
    ("cost_of_funds_annual",  COF_GRID),
    ("acquisition_cost",      ACQ_GRID),
    ("lgd",                   LGD_GRID),
    ("apr_strategy",          APR_STRATS),
]:
    print(f"\nVarying {p}:")
    print(vary(p, vals).to_string(index=False))

Holding other params at center (mult=2, COF=0.03, acq=250, LGD=0.65, APR=tiered_cap_24):


Varying pd_multiplier:
 pd_multiplier  k_star_approve_pct  mean_Expected_Profit    category
           1.0          100.000000            900.808042 approve_all
           2.0           99.253289            999.636277 approve_all
           3.0           97.733591           1037.032965    interior
           5.0           95.558296           1036.985057    interior

Varying cost_of_funds_annual:
 cost_of_funds_annual  k_star_approve_pct  mean_Expected_Profit    category
                 0.00           99.652495           1297.602202 approve_all
                 0.03           99.253289            999.636277 approve_all
                 0.06           98.701519            701.670351    interior

Varying acquisition_cost:
 acquisition_cost  k_star_approve_pct  mean_Expected_Profit    category
                0           99.690636           1249.636277 approve_all
              250           99.2532

## 6. Profit-vs-Youden cutoff finding under stress

In [7]:
# Distribution of cutoff_gap (k_star - youden_pct) across all 576 cells
gap = stress["cutoff_gap_profit_minus_youden"]
print("cutoff_gap = k_star_approve_pct - youden_approve_pct  (PRIMARY=LightGBM)")
print(f"  min:    {gap.min():>+7.2f} pp")
print(f"  p10:    {gap.quantile(0.10):>+7.2f} pp")
print(f"  median: {gap.median():>+7.2f} pp")
print(f"  p90:    {gap.quantile(0.90):>+7.2f} pp")
print(f"  max:    {gap.max():>+7.2f} pp")
print(f"  mean:   {gap.mean():>+7.2f} pp")
print()
print(f"  Cells where profit cutoff is MORE PERMISSIVE than Youden: {(gap > 0).sum()} / {len(gap)} = {(gap > 0).mean():.1%}")
print(f"  Cells where profit cutoff is LESS PERMISSIVE than Youden: {(gap < 0).sum()} / {len(gap)} = {(gap < 0).mean():.1%}")
print(f"  Cells where they match (within 1 pp):                     {(gap.abs() <= 1).sum()} / {len(gap)} = {(gap.abs() <= 1).mean():.1%}")

# By category
print()
print("Mean cutoff_gap by category:")
print(stress.groupby("category")["cutoff_gap_profit_minus_youden"].agg(["mean", "median", "min", "max"]).round(2).to_string())

cutoff_gap = k_star_approve_pct - youden_approve_pct  (PRIMARY=LightGBM)
  min:      +5.33 pp
  p10:     +15.00 pp
  median:  +18.83 pp
  p90:     +20.59 pp
  max:     +20.59 pp
  mean:    +18.23 pp

  Cells where profit cutoff is MORE PERMISSIVE than Youden: 576 / 576 = 100.0%
  Cells where profit cutoff is LESS PERMISSIVE than Youden: 0 / 576 = 0.0%
  Cells where they match (within 1 pp):                     0 / 576 = 0.0%

Mean cutoff_gap by category:
              mean  median    min    max
category                                
approve_all  20.41   20.58  19.63  20.59
interior     16.89   17.32   5.33  19.58


## 7. Three anchor scenarios

In [8]:
# OPTIMISTIC base: mult=1, COF=0, acq=0, LGD=0.55, tiered_uncapped
opt = stress[
    (stress["pd_multiplier"]==1.0) & (stress["cost_of_funds_annual"]==0.0) &
    (stress["acquisition_cost"]==0) & (stress["lgd"]==0.55) &
    (stress["apr_strategy"]=="tiered_uncapped")
].iloc[0]

# REALISTIC central: mult=2, COF=0.03, acq=250, LGD=0.65, tiered_cap_24
real = stress[
    (stress["pd_multiplier"]==2.0) & (stress["cost_of_funds_annual"]==0.03) &
    (stress["acquisition_cost"]==250) & (stress["lgd"]==0.65) &
    (stress["apr_strategy"]=="tiered_cap_24")
].iloc[0]

# ADVERSE stress: mult=5, COF=0.06, acq=500, LGD=0.85, flat_18
adv = stress[
    (stress["pd_multiplier"]==5.0) & (stress["cost_of_funds_annual"]==0.06) &
    (stress["acquisition_cost"]==500) & (stress["lgd"]==0.85) &
    (stress["apr_strategy"]=="flat_18")
].iloc[0]

anchors = {
    "optimistic_base": opt.to_dict(),
    "realistic_central": real.to_dict(),
    "adverse_stress": adv.to_dict(),
}
print("3 anchor scenarios:")
for name, vals in anchors.items():
    print(f"\n{name}:")
    for k, v in vals.items():
        if isinstance(v, float):
            print(f"  {k:<35} {v:>15.4f}")
        else:
            print(f"  {k:<35} {v}")

3 anchor scenarios:

optimistic_base:
  pd_multiplier                                1.0000
  apr_strategy                        tiered_uncapped
  cost_of_funds_annual                         0.0000
  acquisition_cost                    0
  lgd                                          0.5500
  mean_pd_stressed                             0.0095
  mean_LT_EL                                  53.2676
  mean_LT_margin                            1556.0200
  mean_Expected_Profit                      1502.7523
  total_Expected_Profit                354601464.4394
  share_profit_gt_0                            1.0000
  k_star_approve_pct                         100.0000
  cutoff_pd_star                               0.1916
  max_total_profit_at_k_star           354601464.4394
  youden_approve_pct_full                     79.4057
  cutoff_gap_profit_minus_youden              20.5943
  category                            approve_all

realistic_central:
  pd_multiplier                           

## 8. Save artifacts

In [9]:
OUT_DIR = REPO_ROOT / "artifacts/economic_framework"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stress grid as parquet
stress.to_parquet(OUT_DIR / "economic_stress_grid.parquet", index=False)
print(f"Saved: {OUT_DIR / 'economic_stress_grid.parquet'}  ({len(stress)} cells)")

# Cut-off-only CSV (compact)
stress[["pd_multiplier","apr_strategy","cost_of_funds_annual","acquisition_cost","lgd",
         "k_star_approve_pct","cutoff_pd_star","max_total_profit_at_k_star",
         "youden_approve_pct_full","cutoff_gap_profit_minus_youden","category"]].to_csv(
    OUT_DIR / "stress_cutoff_results.csv", index=False)
print(f"Saved: {OUT_DIR / 'stress_cutoff_results.csv'}")

# Anchor scenarios JSON
with open(OUT_DIR / "anchor_scenarios.json", "w") as f:
    json.dump({k: {kk: (float(vv) if hasattr(vv, "item") else vv) for kk, vv in v.items()}
               for k, v in anchors.items()}, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'anchor_scenarios.json'}")

print(f"\nTotal Phase 3.2 wall: {time.time()-T0:.1f}s")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\economic_stress_grid.parquet  (576 cells)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\stress_cutoff_results.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\anchor_scenarios.json

Total Phase 3.2 wall: 135.9s


# Section 7 - 04_bootstrap_cutoff_ci.ipynb

**Section metadata**

- **Purpose**: Phase 4.1 - 1,000-resample stratified bootstrap on the 64,027-row OOT economics population at four anchor scenarios (4,000 anchor x resample combinations).
- **Main artifacts read**: `economics_per_account.parquet` filtered to the OOT split.
- **Main artifacts written**: `artifacts/economic_framework/bootstrap_anchor_results.parquet` and bootstrap CI summary.
- **Expensive rerun expected**: Moderate-to-expensive (4,000 stratified resamples).

---


# Phase 4.1 — Bootstrap Confidence Intervals on Anchor Scenarios

**Goal**: 1000-resample bootstrap CIs around profit-optimal cut-offs and
profit-vs-Youden gaps for 4 anchor scenarios. Stratified by tenor, on the
OOT economics-eligible subset (24m + 36m only).

**Pre-checks** (Sections 1-2):
- CHECK 1: refine anchors (add `moderate_interior`)
- CHECK 2: op_cost ablation (Phase 3.2 grid omitted op_cost; run small ablation)

**Bootstrap** (Sections 3-6):
- N = 1000 resamples
- Stratified by `n_installments ∈ {24, 36}`
- Primary PD: LightGBM tuned + Platt
- Per-iteration metrics: total profit at k*, k*, PD threshold at k*,
  total profit at Youden, Youden approve %, cutoff gap, profit uplift, uplift %

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.economics import (
    batch_lifetime_economics, apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)
from src.evaluation import compute_gini

ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
STRESS_PATH = REPO_ROOT / "artifacts/economic_framework/economic_stress_grid.parquet"
OUT_DIR  = REPO_ROOT / "artifacts/economic_framework"

eco = pd.read_parquet(ECO_PATH)
stress = pd.read_parquet(STRESS_PATH)
print(f"eco: {eco.shape}; stress grid: {stress.shape}")
PRIMARY_PD = "pd_lgb_platt"
T0 = time.time()

eco: (235968, 15); stress grid: (576, 17)


## 1. CHECK 1 — Anchor refinement

`realistic_central` previously classified as approve_all (k*=99.25%) — relabel as
`realistic_central_boundary`. Search the stress grid for a `moderate_interior`
anchor with `94% <= k* <= 97%`, total profit > 0, gap > 0, and balanced
parameter levels (PD multiplier in {2, 3}, LGD in {0.65, 0.75}).

In [2]:
mask = (
    (stress["k_star_approve_pct"] >= 94)
    & (stress["k_star_approve_pct"] <= 97)
    & (stress["total_Expected_Profit"] > 0)
    & (stress["cutoff_gap_profit_minus_youden"] > 0)
    & (stress["pd_multiplier"].isin([2.0, 3.0]))
    & (stress["lgd"].isin([0.65, 0.75]))
)
candidates = stress[mask].copy()
print(f"Balanced moderate_interior candidates: {len(candidates)}")

# Pick the most representative — closest to k=95.5, prefer flat_18 + LGD=0.65 + COF=3%
def score(row):
    s = abs(row["k_star_approve_pct"] - 95.5) * 10
    s += 5 if row["apr_strategy"] != "flat_18" else 0  # prefer flat_18
    s += 5 if row["lgd"] != 0.65 else 0
    s += 3 if row["cost_of_funds_annual"] != 0.03 else 0
    s += 3 if row["acquisition_cost"] != 250 else 0
    s += 3 if row["pd_multiplier"] != 3.0 else 0
    return s
candidates["sel_score"] = candidates.apply(score, axis=1)
chosen = candidates.sort_values("sel_score").iloc[0]
print()
print("Chosen moderate_interior:")
for k in ["pd_multiplier","apr_strategy","cost_of_funds_annual","acquisition_cost","lgd",
          "k_star_approve_pct","mean_Expected_Profit","total_Expected_Profit",
          "share_profit_gt_0","cutoff_gap_profit_minus_youden","category"]:
    v = chosen[k]
    if isinstance(v, float):
        print(f"  {k:<35} {v:>15.4f}")
    else:
        print(f"  {k:<35} {v}")

MODERATE_INTERIOR = chosen.to_dict()

Balanced moderate_interior candidates: 31

Chosen moderate_interior:
  pd_multiplier                                3.0000
  apr_strategy                        flat_18
  cost_of_funds_annual                         0.0300
  acquisition_cost                    500
  lgd                                          0.6500
  k_star_approve_pct                          95.2939
  mean_Expected_Profit                       808.9267
  total_Expected_Profit                190880809.4118
  share_profit_gt_0                            0.8406
  cutoff_gap_profit_minus_youden              15.8882
  category                            interior


In [3]:
# Final anchor set (4 scenarios)
ANCHORS = {
    "optimistic_base": {
        "pd_multiplier": 1.0, "cost_of_funds_annual": 0.00,
        "acquisition_cost": 0, "lgd": 0.55,
        "apr_strategy": "tiered_uncapped", "op_cost_annual": 0.0,
    },
    "realistic_central_boundary": {
        "pd_multiplier": 2.0, "cost_of_funds_annual": 0.03,
        "acquisition_cost": 250, "lgd": 0.65,
        "apr_strategy": "tiered_cap_24", "op_cost_annual": 0.0,
    },
    "moderate_interior": {
        "pd_multiplier": float(MODERATE_INTERIOR["pd_multiplier"]),
        "cost_of_funds_annual": float(MODERATE_INTERIOR["cost_of_funds_annual"]),
        "acquisition_cost": int(MODERATE_INTERIOR["acquisition_cost"]),
        "lgd": float(MODERATE_INTERIOR["lgd"]),
        "apr_strategy": MODERATE_INTERIOR["apr_strategy"],
        "op_cost_annual": 0.0,
    },
    "adverse_stress": {
        "pd_multiplier": 5.0, "cost_of_funds_annual": 0.06,
        "acquisition_cost": 500, "lgd": 0.85,
        "apr_strategy": "flat_18", "op_cost_annual": 0.0,
    },
}
print("Final anchor set:")
for name, params in ANCHORS.items():
    print(f"\n  {name}:")
    for k, v in params.items():
        print(f"    {k}: {v}")

Final anchor set:

  optimistic_base:
    pd_multiplier: 1.0
    cost_of_funds_annual: 0.0
    acquisition_cost: 0
    lgd: 0.55
    apr_strategy: tiered_uncapped
    op_cost_annual: 0.0

  realistic_central_boundary:
    pd_multiplier: 2.0
    cost_of_funds_annual: 0.03
    acquisition_cost: 250
    lgd: 0.65
    apr_strategy: tiered_cap_24
    op_cost_annual: 0.0

  moderate_interior:
    pd_multiplier: 3.0
    cost_of_funds_annual: 0.03
    acquisition_cost: 500
    lgd: 0.65
    apr_strategy: flat_18
    op_cost_annual: 0.0

  adverse_stress:
    pd_multiplier: 5.0
    cost_of_funds_annual: 0.06
    acquisition_cost: 500
    lgd: 0.85
    apr_strategy: flat_18
    op_cost_annual: 0.0


## 2. CHECK 2 — op_cost ablation

The 576-cell Phase 3.2 grid held `op_cost_annual = 0` to keep grid size
manageable (focus was on PD/COF/acq/LGD/APR). Phase 3.1B had already covered
op_cost ∈ {0, 0.01, 0.02} in its 150-cell grid. Here we run a focused ablation
on `realistic_central_boundary` and `adverse_stress` over `op_cost_annual ∈
{0.00, 0.01, 0.02, 0.04}` to confirm the effect.

In [4]:
def apr_array(pd_arr, strategy):
    if strategy == "tiered_uncapped":
        return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy == "tiered_cap_18":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.18)
    if strategy.startswith("flat_"):
        v = int(strategy.split("_")[1]) / 100.0
        return np.full_like(pd_arr, v, dtype=np.float64)
    raise ValueError(f"unknown strategy {strategy}")

def compute_anchor_economics(eco_subset, anchor, pd_col=PRIMARY_PD,
                              op_cost_override=None):
    pd_base = eco_subset[pd_col].fillna(0.0).to_numpy()
    pd_stressed = np.clip(pd_base * anchor["pd_multiplier"], 0.0, 0.999)
    apr_arr = apr_array(pd_stressed, anchor["apr_strategy"])
    op = op_cost_override if op_cost_override is not None else anchor.get("op_cost_annual", 0.0)
    out = batch_lifetime_economics(
        pd_12m=pd_stressed,
        loan_amount=eco_subset["loan_amount"].to_numpy(),
        n_installments=eco_subset["n_installments"].to_numpy(),
        apr=apr_arr, lgd=anchor["lgd"],
        op_cost_annual=op,
        cost_of_funds_annual=anchor["cost_of_funds_annual"],
        acquisition_cost=float(anchor["acquisition_cost"]),
    )
    return pd_stressed, out

# Run ablation on 2 anchors x 4 op_cost levels
ablation = []
for name in ["realistic_central_boundary", "adverse_stress"]:
    a = ANCHORS[name]
    for op in [0.00, 0.01, 0.02, 0.04]:
        pd_str, out = compute_anchor_economics(eco, a, op_cost_override=op)
        profit = out["Expected_Profit"]
        order = np.argsort(pd_str)
        cum = np.cumsum(profit[order])
        if cum.max() <= 0:
            k_star, max_p = 0, 0.0
        else:
            k_star = int(np.argmax(cum)) + 1
            max_p = float(cum[k_star - 1])
        ablation.append({
            "anchor": name,
            "op_cost_annual": op,
            "mean_profit": float(profit.mean()),
            "total_profit": float(profit.sum()),
            "share_profit_gt_0": float((profit > 0).mean()),
            "k_star_approve_pct": 100.0 * k_star / len(profit),
            "max_total_profit_at_k_star": max_p,
        })
ab_df = pd.DataFrame(ablation)
print("op_cost ablation (4 op_cost levels x 2 anchors):")
print(ab_df.round(2).to_string(index=False))
ab_df.to_csv(OUT_DIR / "op_cost_ablation.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'op_cost_ablation.csv'}")

op_cost ablation (4 op_cost levels x 2 anchors):
                    anchor  op_cost_annual  mean_profit  total_profit  share_profit_gt_0  k_star_approve_pct  max_total_profit_at_k_star
realistic_central_boundary            0.00       999.64  235882172.93               0.97               99.25                236363994.03
realistic_central_boundary            0.01       821.10  193754455.93               0.93               98.89                194498265.45
realistic_central_boundary            0.02       642.57  151626738.94               0.87               98.68                152731726.74
realistic_central_boundary            0.04       285.51   67371304.94               0.53               97.81                 69576649.90
            adverse_stress            0.00       300.59   70930376.02               0.67               84.73                112939775.59
            adverse_stress            0.01       127.83   30164459.82               0.59               81.16                 7782

## 3. Bootstrap setup

- Population: OOT economics-eligible (24m + 36m only)
- N_BOOTSTRAP = 1000
- Stratified by tenor (24m, 36m)
- random_state = 42
- Primary PD: LightGBM Platt

In [5]:
N_BOOTSTRAP = 1000
SEED = 42

# OOT subset (already 24m+36m only since eco was filtered in Phase 3.1B)
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
print(f"OOT rows: {len(oot):,}")
print(f"  tenor 24m: {(oot['n_installments']==24).sum():,}")
print(f"  tenor 36m: {(oot['n_installments']==36).sum():,}")
print(f"  default rate: {(oot['default_flag_12m']==1).mean():.4%}")

# Pre-compute per-row Expected_Profit per anchor (constant across bootstrap iters)
print("\nPre-computing per-row Expected_Profit + stressed PD per anchor...")
anchor_eco = {}
for name, a in ANCHORS.items():
    pd_str, out = compute_anchor_economics(oot, a)
    anchor_eco[name] = {
        "pd_stressed": pd_str,
        "profit": out["Expected_Profit"],
        "lt_el": out["LT_EL"],
        "lt_margin": out["LT_margin"],
    }
    pos_share = (out["Expected_Profit"] > 0).mean()
    print(f"  {name:<28}  share_profit>0={pos_share:.4%}  mean_profit={out['Expected_Profit'].mean():>10.2f}  "
          f"mean_pd_stressed={pd_str.mean():.4%}")

# Pre-compute Youden threshold ONCE per anchor (uses stressed PD vs default flag)
from sklearn.metrics import roc_curve
youden_per_anchor = {}
for name, a in ANCHORS.items():
    pd_str = anchor_eco[name]["pd_stressed"]
    fpr, tpr, thr = roc_curve(oot["default_flag_12m"], pd_str)
    j_idx = int(np.argmax(tpr - fpr))
    thr_y = float(thr[j_idx])
    youden_per_anchor[name] = thr_y
    print(f"  Youden thr {name}: {thr_y:.4f}  approve%={float((pd_str <= thr_y).mean()*100):.2f}%")

OOT rows: 64,027
  tenor 24m: 36,775
  tenor 36m: 27,252
  default rate: 1.9164%

Pre-computing per-row Expected_Profit + stressed PD per anchor...


  optimistic_base               share_profit>0=100.0000%  mean_profit=   1501.04  mean_pd_stressed=0.9386%


  realistic_central_boundary    share_profit>0=96.7561%  mean_profit=    997.68  mean_pd_stressed=1.8772%


  moderate_interior             share_profit>0=84.2348%  mean_profit=    811.41  mean_pd_stressed=2.8158%
  adverse_stress                share_profit>0=67.2654%  mean_profit=    304.53  mean_pd_stressed=4.6929%
  Youden thr optimistic_base: 0.0106  approve%=79.59%
  Youden thr realistic_central_boundary: 0.0211  approve%=79.59%


  Youden thr moderate_interior: 0.0317  approve%=79.59%
  Youden thr adverse_stress: 0.0528  approve%=79.59%


## 4. Run bootstrap (4 anchors × 1000 resamples)

In [6]:
def stratified_bootstrap_indices(n_per_stratum, rng):
    """Return concatenated indices for a stratified resample with replacement."""
    parts = []
    for indices in n_per_stratum:
        parts.append(rng.choice(indices, size=len(indices), replace=True))
    return np.concatenate(parts)

# Build stratum index lists
idx_24 = oot.index[oot["n_installments"] == 24].to_numpy()
idx_36 = oot.index[oot["n_installments"] == 36].to_numpy()
print(f"Stratum sizes: 24m={len(idx_24):,}, 36m={len(idx_36):,}")
strata = [idx_24, idx_36]

rng = np.random.default_rng(SEED)
all_results = []

t0 = time.time()
for b in range(N_BOOTSTRAP):
    sample_idx = stratified_bootstrap_indices(strata, rng)
    for name in ANCHORS:
        pd_str = anchor_eco[name]["pd_stressed"][sample_idx]
        profit = anchor_eco[name]["profit"][sample_idx]
        # Profit-optimal cutoff: sort by stressed PD, find max cumulative profit
        order = np.argsort(pd_str)
        cum_profit = np.cumsum(profit[order])
        if cum_profit.max() <= 0:
            k_star = 0
            cutoff_pd_star = 0.0
            profit_at_kstar = 0.0
        else:
            k_star = int(np.argmax(cum_profit)) + 1
            cutoff_pd_star = float(pd_str[order[k_star - 1]])
            profit_at_kstar = float(cum_profit[k_star - 1])
        approve_pct_kstar = 100.0 * k_star / len(profit)
        # Youden cutoff (precomputed on full OOT, applied to bootstrap sample)
        thr_y = youden_per_anchor[name]
        accepted_y = pd_str <= thr_y
        profit_at_youden = float(profit[accepted_y].sum())
        approve_pct_y = float(accepted_y.mean() * 100)
        all_results.append({
            "bootstrap": b,
            "anchor": name,
            "profit_at_kstar": profit_at_kstar,
            "approve_pct_kstar": approve_pct_kstar,
            "cutoff_pd_star": cutoff_pd_star,
            "profit_at_youden": profit_at_youden,
            "approve_pct_youden": approve_pct_y,
            "cutoff_gap": approve_pct_kstar - approve_pct_y,
            "profit_uplift": profit_at_kstar - profit_at_youden,
            "profit_uplift_pct": (profit_at_kstar - profit_at_youden) / abs(profit_at_youden) if profit_at_youden != 0 else float("nan"),
        })
    if (b + 1) % 200 == 0:
        print(f"  {b+1}/{N_BOOTSTRAP} done ({time.time()-t0:.1f}s elapsed)")

bs = pd.DataFrame(all_results)
print(f"\nBootstrap wall: {time.time()-t0:.1f}s")
print(f"Result rows: {len(bs):,} ({N_BOOTSTRAP} × {len(ANCHORS)})")

Stratum sizes: 24m=36,775, 36m=27,252


  200/1000 done (6.3s elapsed)


  400/1000 done (12.9s elapsed)


  600/1000 done (19.2s elapsed)


  800/1000 done (25.6s elapsed)


  1000/1000 done (32.3s elapsed)

Bootstrap wall: 32.3s
Result rows: 4,000 (1000 × 4)


## 5. Summary statistics + CIs per anchor

In [7]:
metrics = ["profit_at_kstar","approve_pct_kstar","cutoff_pd_star",
           "profit_at_youden","approve_pct_youden","cutoff_gap",
           "profit_uplift","profit_uplift_pct"]
summary_rows = []
for anchor in ANCHORS:
    sub = bs[bs["anchor"] == anchor]
    for m in metrics:
        s = sub[m]
        summary_rows.append({
            "anchor": anchor,
            "metric": m,
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std()),
            "ci_lo_2_5": float(s.quantile(0.025)),
            "ci_hi_97_5": float(s.quantile(0.975)),
        })
summary = pd.DataFrame(summary_rows)
print("=== Bootstrap summary (4 anchors × 8 metrics × 5 stats) ===")
for anchor in ANCHORS:
    print(f"\n{anchor}:")
    sub = summary[summary["anchor"] == anchor].drop(columns="anchor")
    print(sub.round(4).to_string(index=False))

=== Bootstrap summary (4 anchors × 8 metrics × 5 stats) ===

optimistic_base:
            metric         mean       median         std    ci_lo_2_5   ci_hi_97_5
   profit_at_kstar 9.611159e+07 9.610937e+07 250755.5303 9.563645e+07 9.663242e+07
 approve_pct_kstar 1.000000e+02 1.000000e+02      0.0000 1.000000e+02 1.000000e+02
    cutoff_pd_star 1.895000e-01 1.902000e-01      0.0016 1.854000e-01 1.902000e-01
  profit_at_youden 6.535986e+07 6.536700e+07 230245.9590 6.492469e+07 6.580849e+07
approve_pct_youden 7.959310e+01 7.959850e+01      0.1565 7.929150e+01 7.990230e+01
        cutoff_gap 2.040690e+01 2.040150e+01      0.1565 2.009770e+01 2.070850e+01
     profit_uplift 3.075174e+07 3.073931e+07 295771.6329 3.017443e+07 3.131378e+07
 profit_uplift_pct 4.705000e-01 4.704000e-01      0.0056 4.599000e-01 4.809000e-01

realistic_central_boundary:
            metric         mean       median         std    ci_lo_2_5   ci_hi_97_5
   profit_at_kstar 6.400817e+07 6.401252e+07 214444.2851 6.3583

In [8]:
# Special probabilities
print("\n=== SPECIAL PROBABILITIES ===")
prob_rows = []
for anchor in ANCHORS:
    sub = bs[bs["anchor"] == anchor]
    p_approve_all = float((sub["approve_pct_kstar"] >= 99).mean())
    p_interior = float(((sub["approve_pct_kstar"] >= 50) & (sub["approve_pct_kstar"] < 99)).mean())
    p_reject_most = float((sub["approve_pct_kstar"] < 50).mean())
    p_profit_more_permissive = float((sub["cutoff_gap"] > 0).mean())
    p_uplift_pos = float((sub["profit_uplift"] > 0).mean())
    prob_rows.append({
        "anchor": anchor,
        "P(k* >= 99% / approve_all)": p_approve_all,
        "P(50% <= k* < 99% / interior)": p_interior,
        "P(k* < 50% / reject_most)": p_reject_most,
        "P(profit cutoff > Youden)": p_profit_more_permissive,
        "P(profit uplift > 0)": p_uplift_pos,
    })
prob_df = pd.DataFrame(prob_rows)
print(prob_df.round(4).to_string(index=False))


=== SPECIAL PROBABILITIES ===
                    anchor  P(k* >= 99% / approve_all)  P(50% <= k* < 99% / interior)  P(k* < 50% / reject_most)  P(profit cutoff > Youden)  P(profit uplift > 0)
           optimistic_base                         1.0                            0.0                        0.0                        1.0                   1.0
realistic_central_boundary                         1.0                            0.0                        0.0                        1.0                   1.0
         moderate_interior                         0.0                            1.0                        0.0                        1.0                   1.0
            adverse_stress                         0.0                            1.0                        0.0                        1.0                   1.0


## 6. Save artifacts

In [9]:
# Per-iteration parquet
bs.to_parquet(OUT_DIR / "bootstrap_anchor_results.parquet", index=False)
print(f"Saved: {OUT_DIR / 'bootstrap_anchor_results.parquet'}  ({len(bs):,} rows)")

# Summary CSV
summary.to_csv(OUT_DIR / "bootstrap_ci_summary.csv", index=False)
print(f"Saved: {OUT_DIR / 'bootstrap_ci_summary.csv'}")

# Cut-off distributions (k*, Youden % per anchor)
cutoff_dist_rows = []
for anchor in ANCHORS:
    sub = bs[bs["anchor"] == anchor]
    for col in ["approve_pct_kstar", "approve_pct_youden", "cutoff_gap"]:
        s = sub[col]
        cutoff_dist_rows.append({
            "anchor": anchor,
            "metric": col,
            "p2_5": float(s.quantile(0.025)),
            "p10": float(s.quantile(0.10)),
            "p25": float(s.quantile(0.25)),
            "p50": float(s.quantile(0.50)),
            "p75": float(s.quantile(0.75)),
            "p90": float(s.quantile(0.90)),
            "p97_5": float(s.quantile(0.975)),
            "mean": float(s.mean()),
            "std": float(s.std()),
        })
cd_df = pd.DataFrame(cutoff_dist_rows)
cd_df.to_csv(OUT_DIR / "bootstrap_cutoff_distributions.csv", index=False)
print(f"Saved: {OUT_DIR / 'bootstrap_cutoff_distributions.csv'}")

# Anchor definitions + special probs JSON
out_json = {
    "anchors": ANCHORS,
    "youden_thresholds": {k: float(v) for k, v in youden_per_anchor.items()},
    "special_probabilities": prob_df.to_dict(orient="records"),
    "n_bootstrap": N_BOOTSTRAP,
    "stratification": "by tenor (24m / 36m)",
    "primary_pd": PRIMARY_PD,
    "seed": SEED,
}
with open(OUT_DIR / "anchor_scenarios_v2.json", "w") as f:
    json.dump(out_json, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'anchor_scenarios_v2.json'}")

print(f"\nTotal Phase 4.1 wall: {time.time()-T0:.1f}s")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\bootstrap_anchor_results.parquet  (4,000 rows)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\bootstrap_ci_summary.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\bootstrap_cutoff_distributions.csv


Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\economic_framework\anchor_scenarios_v2.json

Total Phase 4.1 wall: 36.7s


# Section 8 - 05_pd_quality_opcost_stress.ipynb

**Section metadata**

- **Purpose**: Phase 4.2 - extended 64-cell stress space: PART A PD-signal stress (16 cells), PART B operating-cost grid (12 cells), PART C combined PD x op_cost grid (36 cells).
- **Main artifacts read**: `economics_per_account.parquet` plus perturbed PD distributions.
- **Main artifacts written**: `artifacts/economic_framework/phase4_2_stress_space.parquet` and PART A/B/C reports.
- **Expensive rerun expected**: Moderate.

---


# Phase 4.2 — PD-quality stress + op_cost robustness

**Goal**: test whether the profit-vs-Youden finding survives under (A) lower
PD discrimination and (B) higher operational-cost stress.

**Population**: OOT economics-eligible subset (24m + 36m only) — same as
Phase 4.1 bootstrap. **64,027 rows** (CHECK 0 correction: Phase 4.1 report had
a transcription error claiming 144,789).

**PD source**: LightGBM tuned + Platt-calibrated (raw OOT Gini ≈ 0.795).

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.economics import (
    batch_lifetime_economics, apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)
from src.evaluation import compute_gini
from src.calibration import perturb_to_target_gini

ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
PD_QUAL_DIR = REPO_ROOT / "artifacts/pd_quality_stress"
OP_DIR      = REPO_ROOT / "artifacts/op_cost_robustness"
ECO_DIR     = REPO_ROOT / "artifacts/economic_framework"
PD_QUAL_DIR.mkdir(parents=True, exist_ok=True)
OP_DIR.mkdir(parents=True, exist_ok=True)

eco = pd.read_parquet(ECO_PATH)
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
print(f"OOT rows: {len(oot):,}  (24m: {(oot['n_installments']==24).sum():,}; 36m: {(oot['n_installments']==36).sum():,})")
print(f"OOT base default rate: {(oot['default_flag_12m']==1).mean():.5%}")

PRIMARY_PD = "pd_lgb_platt"
T0 = time.time()

OOT rows: 64,027  (24m: 36,775; 36m: 27,252)
OOT base default rate: 1.91638%


## CHECK 0 — Phase 4.1 bootstrap population correction

The Phase 4.1 bootstrap report claimed 144,789 OOT rows. The actual code used
`eco[eco["split_new"] == "oot"]` which yields **64,027 rows** (24m: 36,775;
36m: 27,252). The 144,789 figure was a transcription error (it appears to
mix the OOT-of-full-wide-ABT count). The bootstrap results themselves are
correct (they were computed on the actual 64,027 rows). The label is:
**"OOT validation bootstrap (64,027 rows)"**, not portfolio-level.

In [2]:
# CHECK 0 confirmation cell
n_oot = len(oot)
print(f"CHECK 0 — Phase 4.1 bootstrap used split_new == 'oot' = {n_oot:,} rows")
print(f"Tenor split: 24m={(oot['n_installments']==24).sum():,}, 36m={(oot['n_installments']==27252).sum() if False else (oot['n_installments']==36).sum():,}")
print("Label: OOT validation bootstrap (NOT full portfolio).")

CHECK 0 — Phase 4.1 bootstrap used split_new == 'oot' = 64,027 rows
Tenor split: 24m=36,775, 36m=27,252
Label: OOT validation bootstrap (NOT full portfolio).


## PART A — PD-quality stress

Use `perturb_to_target_gini` to create PD variants at target Gini levels
[0.30, 0.45, 0.60, raw]. Each variant is re-calibrated to the OOT base rate
so the comparison isolates *discrimination* shift, not *level* shift.

In [3]:
y_oot = oot["default_flag_12m"].astype(int).to_numpy()
pd_raw = oot[PRIMARY_PD].fillna(0.0).to_numpy()
raw_gini = compute_gini(y_oot, pd_raw)
print(f"Raw OOT Gini (LightGBM Platt): {raw_gini:.4f}")

# Generate PD variants
TARGET_GINIS = [0.30, 0.45, 0.60]
pd_variants = {"raw": (pd_raw, {"sigma": 0.0, "achieved_gini": float(raw_gini), "note": "raw"})}
for tg in TARGET_GINIS:
    sp, meta = perturb_to_target_gini(pd_raw, y_oot, tg, seed=42, tolerance=0.005, sigma_max=5.0, n_iter=40, re_calibrate=True)
    pd_variants[f"gini_{int(tg*100)}"] = (sp, meta)
    print(f"  target Gini={tg:.2f}: sigma={meta['sigma']:.3f}, achieved={meta['achieved_gini']:.4f}, mean_pd={float(sp.mean()):.4%} (base_rate={float(y_oot.mean()):.4%})")
print()
# Save variants
variants_df = pd.DataFrame({"aid": oot["aid"].values, "n_installments": oot["n_installments"].values,
                             "loan_amount": oot["loan_amount"].values, "default_flag_12m": y_oot})
variants_df["pd_raw"] = pd_raw
for k, (sp, _) in pd_variants.items():
    if k != "raw":
        variants_df[f"pd_{k}"] = sp
print(f"PD variants shape: {variants_df.shape}")
variants_df.to_parquet(PD_QUAL_DIR / "pd_variants.parquet", index=False)
print(f"Saved: {PD_QUAL_DIR / 'pd_variants.parquet'}")

Raw OOT Gini (LightGBM Platt): 0.8044


  target Gini=0.30: sigma=4.831, achieved=0.3027, mean_pd=2.1767% (base_rate=1.9164%)
  target Gini=0.45: sigma=3.125, achieved=0.4456, mean_pd=1.9164% (base_rate=1.9164%)


  target Gini=0.60: sigma=1.875, achieved=0.6030, mean_pd=1.9164% (base_rate=1.9164%)

PD variants shape: (64027, 8)
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_quality_stress\pd_variants.parquet


In [4]:
# Anchor scenario definitions (4)
ANCHORS = {
    "optimistic_base": {
        "pd_multiplier": 1.0, "cost_of_funds_annual": 0.00,
        "acquisition_cost": 0, "lgd": 0.55,
        "apr_strategy": "tiered_uncapped",
    },
    "realistic_central_boundary": {
        "pd_multiplier": 2.0, "cost_of_funds_annual": 0.03,
        "acquisition_cost": 250, "lgd": 0.65,
        "apr_strategy": "tiered_cap_24",
    },
    "moderate_interior": {
        "pd_multiplier": 3.0, "cost_of_funds_annual": 0.03,
        "acquisition_cost": 250, "lgd": 0.65,
        "apr_strategy": "flat_18",
    },
    "adverse_stress": {
        "pd_multiplier": 5.0, "cost_of_funds_annual": 0.06,
        "acquisition_cost": 500, "lgd": 0.85,
        "apr_strategy": "flat_18",
    },
}

def apr_array(pd_arr, strategy):
    if strategy == "tiered_uncapped":
        return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy == "tiered_cap_18":
        return apr_tier_lookup_capped_vec(pd_arr, cap=0.18)
    if strategy.startswith("flat_"):
        v = int(strategy.split("_")[1]) / 100.0
        return np.full_like(pd_arr, v, dtype=np.float64)
    raise ValueError(strategy)

def compute_anchor_economics_arr(pd_base, anchor, op_cost_override=None):
    pd_str = np.clip(pd_base * anchor["pd_multiplier"], 0.0, 0.999)
    apr_arr = apr_array(pd_str, anchor["apr_strategy"])
    op = op_cost_override if op_cost_override is not None else 0.0
    out = batch_lifetime_economics(
        pd_12m=pd_str, loan_amount=oot["loan_amount"].to_numpy(),
        n_installments=oot["n_installments"].to_numpy(),
        apr=apr_arr, lgd=anchor["lgd"],
        op_cost_annual=op,
        cost_of_funds_annual=anchor["cost_of_funds_annual"],
        acquisition_cost=float(anchor["acquisition_cost"]),
    )
    return pd_str, out

def compute_cutoff_metrics(pd_str, profit, y_true):
    """Return dict of cutoff metrics."""
    from sklearn.metrics import roc_curve
    order = np.argsort(pd_str)
    cum_profit = np.cumsum(profit[order])
    if cum_profit.max() <= 0:
        k_star = 0
        cutoff_pd_star = 0.0
        profit_at_kstar = 0.0
    else:
        k_star = int(np.argmax(cum_profit)) + 1
        cutoff_pd_star = float(pd_str[order[k_star - 1]])
        profit_at_kstar = float(cum_profit[k_star - 1])
    approve_pct_kstar = 100.0 * k_star / len(profit)
    # Youden on stressed PD
    fpr, tpr, thr = roc_curve(y_true, pd_str)
    j_idx = int(np.argmax(tpr - fpr))
    thr_y = float(thr[j_idx])
    accepted_y = pd_str <= thr_y
    profit_at_youden = float(profit[accepted_y].sum())
    approve_pct_y = float(accepted_y.mean() * 100)
    cutoff_gap = approve_pct_kstar - approve_pct_y
    profit_uplift = profit_at_kstar - profit_at_youden
    profit_uplift_pct = profit_uplift / abs(profit_at_youden) if profit_at_youden != 0 else float("nan")
    if approve_pct_kstar >= 99:
        cat = "approve_all"
    elif approve_pct_kstar >= 50:
        cat = "interior"
    else:
        cat = "reject_most"
    return {
        "k_star_approve_pct": approve_pct_kstar,
        "cutoff_pd_star": cutoff_pd_star,
        "profit_at_kstar": profit_at_kstar,
        "profit_at_youden": profit_at_youden,
        "approve_pct_youden": approve_pct_y,
        "cutoff_gap": cutoff_gap,
        "profit_uplift": profit_uplift,
        "profit_uplift_pct": profit_uplift_pct,
        "category": cat,
    }

In [5]:
# PART A: PD variants × anchors (no op_cost stress; op=0)
part_a = []
for variant_name, (pd_base, meta) in pd_variants.items():
    for anchor_name, a in ANCHORS.items():
        pd_str, out = compute_anchor_economics_arr(pd_base, a)
        m = compute_cutoff_metrics(pd_str, out["Expected_Profit"], y_oot)
        m["pd_variant"] = variant_name
        m["pd_variant_gini"] = meta["achieved_gini"]
        m["anchor"] = anchor_name
        m["mean_profit"] = float(out["Expected_Profit"].mean())
        m["share_profit_gt_0"] = float((out["Expected_Profit"] > 0).mean())
        part_a.append(m)

pa_df = pd.DataFrame(part_a)
print("PART A — PD-quality stress (4 variants × 4 anchors = 16 cells):")
cols = ["pd_variant","pd_variant_gini","anchor","k_star_approve_pct",
        "approve_pct_youden","cutoff_gap","profit_at_kstar","profit_uplift",
        "profit_uplift_pct","share_profit_gt_0","category"]
print(pa_df[cols].round(4).to_string(index=False))
pa_df.to_csv(PD_QUAL_DIR / "cutoffs_by_gini.csv", index=False)
print(f"\nSaved: {PD_QUAL_DIR / 'cutoffs_by_gini.csv'}")

PART A — PD-quality stress (4 variants × 4 anchors = 16 cells):
pd_variant  pd_variant_gini                     anchor  k_star_approve_pct  approve_pct_youden  cutoff_gap  profit_at_kstar  profit_uplift  profit_uplift_pct  share_profit_gt_0    category
       raw           0.8044            optimistic_base            100.0000             79.5930     20.4070     9.610732e+07   3.075180e+07             0.4705             1.0000 approve_all
       raw           0.8044 realistic_central_boundary             99.2550             79.5930     19.6620     6.400525e+07   1.775092e+07             0.3838             0.9676 approve_all
       raw           0.8044          moderate_interior             96.3344             79.5930     16.7414     6.972650e+07   8.054153e+06             0.1306             0.9562    interior
       raw           0.8044             adverse_stress             84.8986             79.5930      5.3056     3.078831e+07   4.558479e+05             0.0150             0.6727    

## PART B — op_cost robustness

Apply op_cost_annual ∈ {0.00, 0.01, 0.02, 0.04} to 3 anchors using the raw PD.

In [6]:
OP_COST_GRID = [0.00, 0.01, 0.02, 0.04]
OP_ANCHORS = ["realistic_central_boundary", "moderate_interior", "adverse_stress"]

part_b = []
for anchor_name in OP_ANCHORS:
    a = ANCHORS[anchor_name]
    for op in OP_COST_GRID:
        pd_str, out = compute_anchor_economics_arr(pd_raw, a, op_cost_override=op)
        m = compute_cutoff_metrics(pd_str, out["Expected_Profit"], y_oot)
        m["anchor"] = anchor_name
        m["op_cost_annual"] = op
        m["mean_profit"] = float(out["Expected_Profit"].mean())
        m["total_profit"] = float(out["Expected_Profit"].sum())
        m["share_profit_gt_0"] = float((out["Expected_Profit"] > 0).mean())
        part_b.append(m)

pb_df = pd.DataFrame(part_b)
print("PART B — op_cost robustness (3 anchors × 4 op_cost = 12 cells):")
cols = ["anchor","op_cost_annual","k_star_approve_pct","approve_pct_youden",
        "cutoff_gap","mean_profit","total_profit","share_profit_gt_0",
        "profit_uplift","profit_uplift_pct","category"]
print(pb_df[cols].round(4).to_string(index=False))
pb_df.to_csv(OP_DIR / "cutoffs_by_op_cost.csv", index=False)
print(f"\nSaved: {OP_DIR / 'cutoffs_by_op_cost.csv'}")

PART B — op_cost robustness (3 anchors × 4 op_cost = 12 cells):
                    anchor  op_cost_annual  k_star_approve_pct  approve_pct_youden  cutoff_gap  mean_profit  total_profit  share_profit_gt_0  profit_uplift  profit_uplift_pct    category
realistic_central_boundary            0.00             99.2550              79.593     19.6620     997.6824  6.387861e+07             0.9676   1.775092e+07             0.3838 approve_all
realistic_central_boundary            0.01             98.9442              79.593     19.3512     819.0199  5.243939e+07             0.9311   1.561549e+07             0.4218    interior
realistic_central_boundary            0.02             98.7380              79.593     19.1450     640.3574  4.100017e+07             0.8738   1.350619e+07             0.4862    interior
realistic_central_boundary            0.04             97.7400              79.593     18.1470     283.0325  1.812172e+07             0.5281   9.390286e+06             1.0088    interior
 

## PART C — Combined Gini × op_cost × scenario mini-grid

In [7]:
GINI_KEYS = ["raw", "gini_60", "gini_45", "gini_30"]
OP_GRID_C = [0.00, 0.02, 0.04]
SCN_C = ["realistic_central_boundary", "moderate_interior", "adverse_stress"]

part_c = []
for variant_name in GINI_KEYS:
    pd_base, meta = pd_variants[variant_name]
    for op in OP_GRID_C:
        for anchor_name in SCN_C:
            a = ANCHORS[anchor_name]
            pd_str, out = compute_anchor_economics_arr(pd_base, a, op_cost_override=op)
            m = compute_cutoff_metrics(pd_str, out["Expected_Profit"], y_oot)
            m["pd_variant"] = variant_name
            m["pd_variant_gini"] = meta["achieved_gini"]
            m["op_cost_annual"] = op
            m["anchor"] = anchor_name
            m["mean_profit"] = float(out["Expected_Profit"].mean())
            m["share_profit_gt_0"] = float((out["Expected_Profit"] > 0).mean())
            part_c.append(m)

pc_df = pd.DataFrame(part_c)
print(f"PART C — Combined mini-grid: {len(pc_df)} cells (4 Gini × 3 op_cost × 3 scenarios)")
cols = ["pd_variant","pd_variant_gini","op_cost_annual","anchor",
        "k_star_approve_pct","approve_pct_youden","cutoff_gap",
        "profit_uplift","profit_uplift_pct","share_profit_gt_0","category"]
print(pc_df[cols].round(4).to_string(index=False))
pc_df.to_csv(ECO_DIR / "phase4_2_combined_grid.csv", index=False)
print(f"\nSaved: {ECO_DIR / 'phase4_2_combined_grid.csv'}")

PART C — Combined mini-grid: 36 cells (4 Gini × 3 op_cost × 3 scenarios)
pd_variant  pd_variant_gini  op_cost_annual                     anchor  k_star_approve_pct  approve_pct_youden  cutoff_gap  profit_uplift  profit_uplift_pct  share_profit_gt_0    category
       raw           0.8044            0.00 realistic_central_boundary             99.2550             79.5930     19.6620   1.775092e+07             0.3838             0.9676 approve_all
       raw           0.8044            0.00          moderate_interior             96.3344             79.5930     16.7414   8.054153e+06             0.1306             0.9562    interior
       raw           0.8044            0.00             adverse_stress             84.8986             79.5930      5.3056   4.558479e+05             0.0150             0.6727    interior
       raw           0.8044            0.02 realistic_central_boundary             98.7380             79.5930     19.1450   1.350619e+07             0.4862             0.8738

## Findings summary

In [8]:
print("=" * 80)
print("FINDINGS")
print("=" * 80)

# A: profit > Youden under low Gini?
print("\nPART A — Does profit cutoff remain MORE permissive than Youden under low Gini?")
for variant_name in GINI_KEYS:
    sub = pa_df[pa_df["pd_variant"] == variant_name]
    n_more = int((sub["cutoff_gap"] > 0).sum())
    n_total = len(sub)
    achieved = float(sub["pd_variant_gini"].iloc[0])
    print(f"  Gini ≈ {achieved:.3f}: profit > Youden in {n_more}/{n_total} anchors")

# B: op_cost tipping points
print("\nPART B — op_cost tipping points:")
for anchor_name in OP_ANCHORS:
    sub = pb_df[pb_df["anchor"] == anchor_name].sort_values("op_cost_annual")
    print(f"\n  {anchor_name}:")
    for _, row in sub.iterrows():
        print(f"    op_cost={row['op_cost_annual']:.2f}: k*={row['k_star_approve_pct']:>6.2f}%  cat={row['category']}  gap={row['cutoff_gap']:+.2f} pp  profit_uplift={row['profit_uplift']:>14,.0f}")

# C: How many cells have profit-cutoff > Youden?
print(f"\nPART C — Combined grid:")
n_more = int((pc_df["cutoff_gap"] > 0).sum())
n_eq = int((pc_df["cutoff_gap"] == 0).sum())
n_less = int((pc_df["cutoff_gap"] < 0).sum())
print(f"  cells where profit cutoff > Youden: {n_more}/{len(pc_df)}")
print(f"  cells where profit cutoff = Youden: {n_eq}/{len(pc_df)}")
print(f"  cells where profit cutoff < Youden: {n_less}/{len(pc_df)}")
print(f"\n  Category counts:")
print(pc_df["category"].value_counts().to_string())

# Final wall time
print(f"\nTotal Phase 4.2 wall: {time.time()-T0:.1f}s")

FINDINGS

PART A — Does profit cutoff remain MORE permissive than Youden under low Gini?
  Gini ≈ 0.804: profit > Youden in 4/4 anchors
  Gini ≈ 0.603: profit > Youden in 4/4 anchors
  Gini ≈ 0.446: profit > Youden in 4/4 anchors
  Gini ≈ 0.303: profit > Youden in 4/4 anchors

PART B — op_cost tipping points:

  realistic_central_boundary:
    op_cost=0.00: k*= 99.26%  cat=approve_all  gap=+19.66 pp  profit_uplift=    17,750,916
    op_cost=0.01: k*= 98.94%  cat=interior  gap=+19.35 pp  profit_uplift=    15,615,492
    op_cost=0.02: k*= 98.74%  cat=interior  gap=+19.15 pp  profit_uplift=    13,506,192
    op_cost=0.04: k*= 97.74%  cat=interior  gap=+18.15 pp  profit_uplift=     9,390,286

  moderate_interior:
    op_cost=0.00: k*= 96.33%  cat=interior  gap=+16.74 pp  profit_uplift=     8,054,153
    op_cost=0.01: k*= 95.72%  cat=interior  gap=+16.13 pp  profit_uplift=     6,265,229
    op_cost=0.02: k*= 94.88%  cat=interior  gap=+15.28 pp  profit_uplift=     4,547,765
    op_cost=0.04:

# Section 9 - 06_calibration_verification.ipynb

**Section metadata**

- **Purpose**: Phase 4.3 - calibration verification across the seven PD sources on the eco-OOT subset; feeds Table 3.4b in Chapter 3 section 3.3.
- **Main artifacts read**: All PD-model OOT predictions.
- **Main artifacts written**: `artifacts/pd_model/calibration_verification.csv`.
- **Expensive rerun expected**: Low.

---


# Phase 4.3 — Calibration Verification

7 PD sources:
1. LightGBM tuned + Platt (PRIMARY)
2. LR full-F6E + Platt
3. LR no-F6E + Platt
4. Scorecard no-F6E + Platt
5. Stressed Gini 0.30 (perturbed from #1)
6. Stressed Gini 0.45
7. Stressed Gini 0.60

Note: scorecard full-F6E robustness exists as metrics only (no row-level PD
parquet was generated in Phase 2B optional run); excluded from this round.
Per source: ECE, Brier, mean predicted vs observed, O:E by decile,
reliability data — saved to `artifacts/calibration_verification/`.

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation import compute_gini, compute_ks, compute_brier, compute_calibration_metrics

ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
PDV_PATH = REPO_ROOT / "artifacts/pd_quality_stress/pd_variants.parquet"
OUT_DIR  = REPO_ROOT / "artifacts/calibration_verification"
OUT_DIR.mkdir(parents=True, exist_ok=True)

eco = pd.read_parquet(ECO_PATH)
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
print(f"OOT rows: {len(oot):,}, base default rate: {(oot['default_flag_12m']==1).mean():.5%}")

pdv = pd.read_parquet(PDV_PATH)
pdv = pdv.set_index("aid").loc[oot["aid"]].reset_index()  # align order
print(f"PD variants aligned: {pdv.shape}")

T0 = time.time()

OOT rows: 64,027, base default rate: 1.91638%
PD variants aligned: (64027, 8)


In [2]:
PD_SOURCES = {
    "LightGBM Platt (PRIMARY)":     oot["pd_lgb_platt"].to_numpy(),
    "LR full-F6E Platt":            oot["pd_lr_full_platt"].to_numpy(),
    "LR no-F6E Platt":              oot["pd_lr_nof6e_platt"].to_numpy(),
    "Scorecard no-F6E Platt":       oot["pd_sc_platt"].to_numpy(),
    "Stressed Gini 0.60":           pdv["pd_gini_60"].to_numpy(),
    "Stressed Gini 0.45":           pdv["pd_gini_45"].to_numpy(),
    "Stressed Gini 0.30":           pdv["pd_gini_30"].to_numpy(),
}
y = oot["default_flag_12m"].astype(int).to_numpy()
base_rate = float(y.mean())
print(f"Base rate: {base_rate:.5%}")

Base rate: 1.91638%


In [3]:
# Compute per-source metrics
def reliability_table(y_true, y_pred, n_bins=10):
    s = pd.Series(y_pred)
    cuts = pd.qcut(s.rank(method="first"), n_bins, labels=False, duplicates="drop")
    rows = []
    for b in range(int(cuts.max()) + 1):
        m = cuts == b
        if not m.any(): continue
        rows.append({
            "bin": b + 1,
            "n": int(m.sum()),
            "mean_pred": float(y_pred[m].mean()),
            "obs_rate": float(y_true[m].mean()),
            "obs_to_exp": (float(y_true[m].mean()) / float(y_pred[m].mean())) if y_pred[m].mean() > 0 else float("nan"),
        })
    return pd.DataFrame(rows)

summary_rows = []
reliability_data = {}
oe_data = {}
for name, p in PD_SOURCES.items():
    p = np.clip(p, 1e-9, 1 - 1e-9)
    cal = compute_calibration_metrics(y, p, n_bins=10)
    summary_rows.append({
        "pd_source": name,
        "n": len(y),
        "base_rate": base_rate,
        "mean_predicted": float(p.mean()),
        "AUC": (compute_gini(y, p) + 1) / 2,
        "Gini": compute_gini(y, p),
        "KS": compute_ks(y, p),
        "Brier": compute_brier(y, p),
        "ECE": cal.get("ece", float("nan")),
        "mean_pred_to_obs_ratio": float(p.mean()) / base_rate if base_rate > 0 else float("nan"),
    })
    rel = reliability_table(y, p)
    reliability_data[name] = rel
    oe_data[name] = {f"o_to_e_bin{i}": float(rel.iloc[i-1]["obs_to_exp"]) if i <= len(rel) else float("nan")
                     for i in range(1, 11)}

summary = pd.DataFrame(summary_rows)
print("=== CALIBRATION SUMMARY ===")
print(summary.round(5).to_string(index=False))

# Save
summary.to_csv(OUT_DIR / "calibration_summary.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'calibration_summary.csv'}")

oe_df = pd.DataFrame(oe_data).T
oe_df.to_csv(OUT_DIR / "o_to_e_by_decile.csv")
print(f"Saved: {OUT_DIR / 'o_to_e_by_decile.csv'}")

# Save reliability data per source
all_rel = []
for name, rel in reliability_data.items():
    rel = rel.copy()
    rel.insert(0, "pd_source", name)
    all_rel.append(rel)
all_rel_df = pd.concat(all_rel, ignore_index=True)
all_rel_df.to_csv(OUT_DIR / "reliability_data.csv", index=False)
print(f"Saved: {OUT_DIR / 'reliability_data.csv'}")

print(f"\nWall: {time.time()-T0:.1f}s")

=== CALIBRATION SUMMARY ===
               pd_source     n  base_rate  mean_predicted     AUC    Gini      KS   Brier     ECE  mean_pred_to_obs_ratio
LightGBM Platt (PRIMARY) 64027    0.01916         0.00939 0.90222 0.80443 0.64611 0.01721 0.00980                 0.48977
       LR full-F6E Platt 64027    0.01916         0.02070 0.86404 0.72808 0.55945 0.01671 0.00261                 1.08024
         LR no-F6E Platt 64027    0.01916         0.02076 0.83998 0.67997 0.54579 0.01703 0.00294                 1.08304
  Scorecard no-F6E Platt 64027    0.01916         0.00918 0.80304 0.60607 0.48131 0.01794 0.00998                 0.47900
      Stressed Gini 0.60 64027    0.01916         0.01916 0.80148 0.60296 0.46904 0.02039 0.01429                 1.00000
      Stressed Gini 0.45 64027    0.01916         0.01916 0.72279 0.44559 0.32829 0.02418 0.02381                 1.00000
      Stressed Gini 0.30 64027    0.01916         0.02177 0.65134 0.30268 0.22881 0.02868 0.03124                 1.13

# Section 10 - 07_visualization.ipynb

**Section metadata**

- **Purpose**: Phase 4.4 - generates the nine publication figures (`fig1_profit_curves` through `fig9_sensitivity_hierarchy`) in PDF and PNG.
- **Main artifacts read**: All upstream artifacts (PD predictions, economics, stress grids, bootstrap).
- **Main artifacts written**: `artifacts/figures/fig1_*` through `fig9_*` (PDF and PNG).
- **Expensive rerun expected**: Low.

---


# Phase 4.4 — Publication Visualization

9 figures, each saved as PDF (300 DPI) + PNG (150 DPI), color-blind-safe
palette (matplotlib viridis / colorbrewer Paired).

Figures:
1. Profit curves per anchor scenario
2. Cutoff gap vs PD discrimination quality (Gini)
3. op_cost vs k* per anchor
4. ASB vs Lifetime profit by tenor
5. Bootstrap CI distributions (4-panel density plot)
6. Calibration reliability diagrams (4-panel, base PD models)
7. Feature importance comparison (top 20)
8. Combined stress matrix heatmap
9. Sensitivity hierarchy (parameter spread on k*)

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ECO_DIR = REPO_ROOT / "artifacts/economic_framework"
PDV_DIR = REPO_ROOT / "artifacts/pd_quality_stress"
OPC_DIR = REPO_ROOT / "artifacts/op_cost_robustness"
CAL_DIR = REPO_ROOT / "artifacts/calibration_verification"
PD_MODEL = REPO_ROOT / "artifacts/pd_model"
OUT_DIR = REPO_ROOT / "artifacts/figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Color-blind-safe palette (ColorBrewer Paired-like)
PALETTE = {
    "optimistic_base":            "#1f77b4",  # blue
    "realistic_central_boundary": "#2ca02c",  # green
    "moderate_interior":          "#ff7f0e",  # orange
    "adverse_stress":             "#d62728",  # red
}
ANCHOR_LABELS = {
    "optimistic_base":            "Optimistic base",
    "realistic_central_boundary": "Realistic central (boundary)",
    "moderate_interior":          "Moderate interior",
    "adverse_stress":             "Adverse stress",
}

# Common figure setup
plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.fontsize": 9, "figure.dpi": 100,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
})

def save_fig(fig, name):
    pdf_path = OUT_DIR / f"{name}.pdf"
    png_path = OUT_DIR / f"{name}.png"
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=150)
    print(f"  Saved: {pdf_path.name} + {png_path.name}")
    plt.close(fig)

T0 = time.time()

## Figure 1 — Profit curves per anchor scenario

In [2]:
# Recreate profit curves: cumulative profit as we approve from lowest PD upward
eco = pd.read_parquet(ECO_DIR / "economics_per_account.parquet")
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
y = oot["default_flag_12m"].astype(int).to_numpy()

# Need to recompute per-anchor profit on OOT for this plot
import sys; sys.path.insert(0, str(REPO_ROOT))
from src.economics import (
    batch_lifetime_economics, apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)

ANCHORS = {
    "optimistic_base": {"pd_multiplier":1.0,"cost_of_funds_annual":0.0,"acquisition_cost":0,"lgd":0.55,"apr_strategy":"tiered_uncapped"},
    "realistic_central_boundary": {"pd_multiplier":2.0,"cost_of_funds_annual":0.03,"acquisition_cost":250,"lgd":0.65,"apr_strategy":"tiered_cap_24"},
    "moderate_interior": {"pd_multiplier":3.0,"cost_of_funds_annual":0.03,"acquisition_cost":250,"lgd":0.65,"apr_strategy":"flat_18"},
    "adverse_stress": {"pd_multiplier":5.0,"cost_of_funds_annual":0.06,"acquisition_cost":500,"lgd":0.85,"apr_strategy":"flat_18"},
}
def apr_array(pd_arr, strategy):
    if strategy == "tiered_uncapped": return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":   return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy.startswith("flat_"):  return np.full_like(pd_arr, int(strategy.split("_")[1])/100.0)

from sklearn.metrics import roc_curve
pd_raw = oot["pd_lgb_platt"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 6))
youden_per_anchor = {}
for name, a in ANCHORS.items():
    pd_str = np.clip(pd_raw * a["pd_multiplier"], 0.0, 0.999)
    apr_arr = apr_array(pd_str, a["apr_strategy"])
    out = batch_lifetime_economics(
        pd_12m=pd_str, loan_amount=oot["loan_amount"].to_numpy(),
        n_installments=oot["n_installments"].to_numpy(),
        apr=apr_arr, lgd=a["lgd"],
        op_cost_annual=0.0,
        cost_of_funds_annual=a["cost_of_funds_annual"],
        acquisition_cost=float(a["acquisition_cost"]),
    )
    profit = out["Expected_Profit"]
    order = np.argsort(pd_str)
    cum = np.cumsum(profit[order]) / 1e6  # $M
    pct = np.arange(1, len(cum) + 1) / len(cum) * 100
    color = PALETTE[name]
    ax.plot(pct, cum, label=ANCHOR_LABELS[name], color=color, linewidth=1.8)
    # Mark profit-optimal k*
    k_star = int(np.argmax(cum)) + 1 if cum.max() > 0 else 1
    ax.scatter([100*k_star/len(cum)], [cum[k_star-1]], color=color, s=60, zorder=10, marker="o", edgecolor="black", linewidth=0.7)
    # Youden
    fpr, tpr, thr = roc_curve(y, pd_str)
    j_idx = int(np.argmax(tpr - fpr))
    thr_y = float(thr[j_idx])
    youden_per_anchor[name] = thr_y
    accepted_y = pd_str <= thr_y
    if accepted_y.any():
        # Find pct corresponding to youden
        pct_y = float(accepted_y.mean() * 100)
        # Profit at Youden = sum of profit for accepted rows
        prof_y = float(profit[accepted_y].sum()) / 1e6
        ax.scatter([pct_y], [prof_y], color=color, s=80, zorder=10, marker="x", linewidth=2.0)

ax.set_xlabel("Cumulative accepted (% of OOT, sorted by PD ascending)")
ax.set_ylabel("Cumulative Expected Profit ($M)")
ax.set_title("Figure 1 — Profit curves by anchor scenario (OOT, n=64,027)")
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.legend(loc="upper left", framealpha=0.9)
# Annotation legend for markers
custom = [
    Line2D([0],[0], marker="o", color="black", markerfacecolor="white", markersize=8, linestyle="None", label="profit-optimal k*"),
    Line2D([0],[0], marker="x", color="black", markersize=8, linestyle="None", label="Youden's J"),
]
leg2 = ax.legend(handles=custom, loc="lower right", framealpha=0.9)
ax.add_artist(ax.legend(loc="upper left", framealpha=0.9))
ax.text(0.99, 0.02, "Source: artifacts/economic_framework/economics_per_account.parquet (OOT)",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig1_profit_curves")

  Saved: fig1_profit_curves.pdf + fig1_profit_curves.png


## Figure 2 — Cutoff gap vs PD discrimination quality

In [3]:
pa = pd.read_csv(PDV_DIR / "cutoffs_by_gini.csv")
fig, ax = plt.subplots(figsize=(9, 6))
for anchor in ANCHORS:
    sub = pa[pa["anchor"] == anchor].sort_values("pd_variant_gini")
    ax.plot(sub["pd_variant_gini"], sub["cutoff_gap"], marker="o",
            linewidth=2, markersize=7, color=PALETTE[anchor],
            label=ANCHOR_LABELS[anchor])
ax.set_xlabel("PD discrimination Gini (achieved)")
ax.set_ylabel("Cutoff gap: profit k* − Youden k (pp)")
ax.set_title("Figure 2 — Profit framework value WIDENS with weaker PD models")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax.invert_xaxis()  # weaker → right
ax.legend(loc="upper left", framealpha=0.9)
ax.text(0.99, 0.02, "Source: artifacts/pd_quality_stress/cutoffs_by_gini.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig2_cutoff_gap_vs_gini")

  Saved: fig2_cutoff_gap_vs_gini.pdf + fig2_cutoff_gap_vs_gini.png


## Figure 3 — op_cost vs k* per anchor

In [4]:
pb = pd.read_csv(OPC_DIR / "cutoffs_by_op_cost.csv")
fig, ax = plt.subplots(figsize=(9, 6))
for anchor in ["realistic_central_boundary", "moderate_interior", "adverse_stress"]:
    sub = pb[pb["anchor"] == anchor].sort_values("op_cost_annual")
    ax.plot(sub["op_cost_annual"] * 100, sub["k_star_approve_pct"],
            marker="s", linewidth=2, markersize=7, color=PALETTE[anchor],
            label=ANCHOR_LABELS[anchor])
ax.axhline(99, color="gray", linestyle=":", linewidth=0.8, label="approve-all threshold (k*=99%)")
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="reject-most threshold (k*=50%)")
ax.set_xlabel("Operating cost (% of loan amount per year)")
ax.set_ylabel("Profit-optimal approval rate k* (%)")
ax.set_title("Figure 3 — op_cost drives profit-optimal cutoff downward, reject-most at adverse + 4%")
ax.set_ylim(-5, 105)
ax.legend(loc="lower left", framealpha=0.9, fontsize=8)
# Annotate the reject-most cell
ax.annotate("REJECT-MOST", xy=(4, 0.31), xytext=(2.7, 18),
            fontsize=9, color="darkred", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="darkred"))
ax.text(0.99, 0.02, "Source: artifacts/op_cost_robustness/cutoffs_by_op_cost.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig3_op_cost_vs_kstar")

  Saved: fig3_op_cost_vs_kstar.pdf + fig3_op_cost_vs_kstar.png


## Figure 4 — ASB vs Lifetime profit by tenor

In [5]:
asb = pd.read_csv(ECO_DIR / "asb_comparison.csv")
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(asb))
w = 0.35
ax.bar(x - w/2, asb["LT_total_profit"] / 1e6, width=w, label="Lifetime (locked)",
       color="#2ca02c", edgecolor="black", linewidth=0.4)
ax.bar(x + w/2, asb["ASB_total_profit"] / 1e6, width=w, label="ASB single-period (benchmark)",
       color="#9ecae1", edgecolor="black", linewidth=0.4)
for i, row in asb.iterrows():
    pct = (row["ASB_LT_total_ratio"] - 1) * 100
    ax.text(i + w/2, row["ASB_total_profit"]/1e6 + 5, f"{pct:+.1f}%",
            ha="center", fontsize=9, color="darkblue")
ax.set_xticks(x)
ax.set_xticklabels(asb["tenor"])
ax.set_xlabel("Tenor")
ax.set_ylabel("Total Expected Profit ($M)")
ax.set_title("Figure 4 — ASB single-period UNDERESTIMATES profit, especially for 36m loans")
ax.legend(loc="upper left", framealpha=0.9)
ax.text(0.99, 0.02, "Source: artifacts/economic_framework/asb_comparison.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig4_asb_vs_lifetime")

  Saved: fig4_asb_vs_lifetime.pdf + fig4_asb_vs_lifetime.png


## Figure 5 — Bootstrap CI distributions (profit_uplift)

In [6]:
bs = pd.read_parquet(ECO_DIR / "bootstrap_anchor_results.parquet")
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=False)
anchor_order = ["optimistic_base","realistic_central_boundary","moderate_interior","adverse_stress"]
for ax, anchor in zip(axes.flat, anchor_order):
    sub = bs[bs["anchor"] == anchor]
    vals = sub["profit_uplift"] / 1e6
    color = PALETTE[anchor]
    ax.hist(vals, bins=30, color=color, alpha=0.7, edgecolor="black", linewidth=0.4)
    p2_5, p50, p97_5 = np.percentile(vals, [2.5, 50, 97.5])
    ax.axvline(p2_5, color="black", linestyle="--", linewidth=0.8, label=f"2.5%: ${p2_5:.2f}M")
    ax.axvline(p50,  color="black", linestyle="-",  linewidth=1.0, label=f"median: ${p50:.2f}M")
    ax.axvline(p97_5,color="black", linestyle="--", linewidth=0.8, label=f"97.5%: ${p97_5:.2f}M")
    ax.axvline(0, color="red", linestyle=":", linewidth=0.8)
    ax.set_title(ANCHOR_LABELS[anchor], color=color, fontweight="bold")
    ax.set_xlabel("Profit uplift over Youden's J ($M)")
    ax.set_ylabel("Bootstrap resample frequency")
    ax.legend(fontsize=7, framealpha=0.9, loc="upper left")
fig.suptitle("Figure 5 — Bootstrap CI distributions of profit uplift (1000 OOT resamples)", fontsize=12, y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/economic_framework/bootstrap_anchor_results.parquet (OOT, n=64,027 stratified)",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig5_bootstrap_ci_density")

  Saved: fig5_bootstrap_ci_density.pdf + fig5_bootstrap_ci_density.png


## Figure 6 — Calibration reliability diagrams (4 panels)

In [7]:
rel = pd.read_csv(CAL_DIR / "reliability_data.csv")
panels = [
    ("LightGBM Platt (PRIMARY)", "LightGBM tuned + Platt"),
    ("LR full-F6E Platt",        "LR full-F6E + Platt"),
    ("LR no-F6E Platt",          "LR no-F6E + Platt"),
    ("Scorecard no-F6E Platt",   "Scorecard no-F6E + Platt"),
]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (key, title) in zip(axes.flat, panels):
    sub = rel[rel["pd_source"] == key]
    if len(sub) == 0:
        ax.set_title(f"{title} (DATA MISSING)")
        continue
    # Reliability diagram: x = mean_pred, y = obs_rate
    ax.plot([0, sub["mean_pred"].max()*1.1], [0, sub["mean_pred"].max()*1.1], color="gray", linestyle="--", linewidth=0.7, label="perfect")
    ax.plot(sub["mean_pred"], sub["obs_rate"], marker="o", linewidth=1.6, color="#1f77b4", label="observed")
    ax.fill_between(sub["mean_pred"], sub["obs_rate"], sub["mean_pred"], color="#1f77b4", alpha=0.15)
    ax.set_xlabel("Mean predicted PD (bin)")
    ax.set_ylabel("Observed default rate (bin)")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Figure 6 — Calibration reliability diagrams (10 deciles, OOT)", y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/calibration_verification/reliability_data.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig6_reliability_diagrams")

  Saved: fig6_reliability_diagrams.pdf + fig6_reliability_diagrams.png


## Figure 7 — Feature importance comparison (top 20)

In [8]:
# LightGBM gain
lgb_imp = pd.read_csv(PD_MODEL / "lightgbm_feature_importance.csv").sort_values("importance_gain", ascending=False).head(20)
# LR full-F6E coefficients (absolute value, normalized for visual)
lr_coef = pd.read_csv(REPO_ROOT / "artifacts/phase2_rerun_v2/final_coefficients.csv")
lr_coef = lr_coef[lr_coef["feature"] != "const"].copy()
lr_coef["abs_coef"] = lr_coef["coef"].abs()
lr_coef = lr_coef.sort_values("abs_coef", ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
# LightGBM
ax = axes[0]
ax.barh(lgb_imp["feature"][::-1], lgb_imp["importance_gain"][::-1] / 1e3,
        color="#1f77b4", edgecolor="black", linewidth=0.3)
ax.set_xlabel("Total split gain (×10³)")
ax.set_title("LightGBM tuned — top 20 by gain")
# LR full-F6E
ax = axes[1]
colors = ["#2ca02c" if c > 0 else "#d62728" for c in lr_coef["coef"][::-1]]
ax.barh(lr_coef["feature"][::-1], lr_coef["coef"][::-1], color=colors, edgecolor="black", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Logit coefficient (signed)")
ax.set_title("LR full-F6E — final 22 features (signed coef)")
fig.suptitle("Figure 7 — Feature importance: LightGBM (gain) vs LR (signed coefficient)", y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/pd_model/lightgbm_feature_importance.csv + artifacts/phase2_rerun_v2/final_coefficients.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig7_feature_importance")

  Saved: fig7_feature_importance.pdf + fig7_feature_importance.png


## Figure 8 — Combined stress matrix heatmap

In [9]:
pc = pd.read_csv(ECO_DIR / "phase4_2_combined_grid.csv")
# Build heatmap rows: scenario × Gini, color = profit_uplift in $M
# Use op_cost as facet (3 panels)
op_levels = sorted(pc["op_cost_annual"].unique())
gini_order = ["raw","gini_60","gini_45","gini_30"]
scn_order  = ["realistic_central_boundary","moderate_interior","adverse_stress"]
scn_labels = ["realistic","moderate","adverse"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
import matplotlib.colors as mcolors
vmin = pc["profit_uplift"].min() / 1e6
vmax = pc["profit_uplift"].max() / 1e6
norm = mcolors.TwoSlopeNorm(vmin=min(vmin, -1), vcenter=0, vmax=max(vmax, 1))

for ax, op in zip(axes, op_levels):
    sub = pc[pc["op_cost_annual"] == op]
    mat = np.zeros((len(scn_order), len(gini_order)))
    for i, sc in enumerate(scn_order):
        for j, g in enumerate(gini_order):
            r = sub[(sub["anchor"] == sc) & (sub["pd_variant"] == g)]
            mat[i, j] = float(r["profit_uplift"].iloc[0]) / 1e6 if len(r) else 0.0
    im = ax.imshow(mat, cmap="RdYlGn", norm=norm, aspect="auto")
    ax.set_xticks(range(len(gini_order)))
    ax.set_xticklabels([f"raw\n(0.80)" if g=="raw" else g.replace("gini_","Gini\n0.").replace("0.6","0.60").replace("0.4","0.45").replace("0.3","0.30") for g in gini_order], fontsize=8)
    ax.set_yticks(range(len(scn_order)))
    ax.set_yticklabels(scn_labels, fontsize=9)
    ax.set_title(f"op_cost = {op*100:.0f}%")
    for i in range(len(scn_order)):
        for j in range(len(gini_order)):
            ax.text(j, i, f"${mat[i,j]:.1f}M", ha="center", va="center",
                    fontsize=8, color="black", fontweight="bold")
fig.colorbar(im, ax=axes, fraction=0.025, label="Profit uplift over Youden ($M)")
fig.suptitle("Figure 8 — Combined stress matrix: profit uplift by scenario × Gini × op_cost", y=1.02)
fig.text(0.99, -0.02, "Source: artifacts/economic_framework/phase4_2_combined_grid.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig8_stress_heatmap")

  Saved: fig8_stress_heatmap.pdf + fig8_stress_heatmap.png


## Figure 9 — Sensitivity hierarchy

In [10]:
# From Phase 3.2 stress grid - parameter spread on k*
sens_data = [
    ("PD multiplier",  3.54),
    ("APR strategy",   2.32),
    ("LGD",            1.28),
    ("Acquisition",    0.43),
    ("Cost of funds",  0.43),
]
labels, spreads = zip(*sens_data)
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(labels))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(labels)))
ax.barh(y_pos, spreads, color=colors, edgecolor="black", linewidth=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Spread on k* (percentage points across grid range)")
ax.set_title("Figure 9 — Sensitivity hierarchy: PD multiplier and APR strategy dominate cutoff variation")
for i, v in enumerate(spreads):
    ax.text(v + 0.05, i, f"{v:.2f} pp", va="center", fontsize=9)
ax.set_xlim(0, 4.2)
ax.text(0.99, 0.02, "Source: Phase 3.2 stress grid driver analysis",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig9_sensitivity_hierarchy")
print(f"\nTotal viz wall: {time.time() - T0:.1f}s")

  Saved: fig9_sensitivity_hierarchy.pdf + fig9_sensitivity_hierarchy.png

Total viz wall: 9.9s
